In [1]:
# Spotlight - Fairness Model
# Notebook 04

import pandas as pd
import numpy as np

print("Fairness model notebook ready!")

Fairness model notebook ready!


In [2]:
# Load the data required for the fairness model

small_matrix = pd.read_csv("../data/small_matrix.csv")
item_features = pd.read_csv("../data/item_daily_features.csv")

# Recreate the watch score used throughout the project
small_matrix["watch_score"] = (
    small_matrix["watch_ratio"].clip(upper=3.0)
)

# Remove interactions without a valid date
small_matrix = small_matrix.dropna(subset=["date"]).copy()

# Correctly parse KuaiRec's YYYYMMDD date format
small_matrix["date"] = pd.to_datetime(
    small_matrix["date"].astype(int).astype(str),
    format="%Y%m%d"
)

# Temporal split
train_end = pd.Timestamp("2020-08-29")

train_data = small_matrix[
    small_matrix["date"] <= train_end
].copy()

test_data = small_matrix[
    small_matrix["date"] > train_end
].copy()

print("Training interactions:", len(train_data))
print("Testing interactions:", len(test_data))
print("Training users:", train_data["user_id"].nunique())
print("Training videos:", train_data["video_id"].nunique())

Training interactions: 4387382
Testing interactions: 107196
Training users: 1411
Training videos: 3288


In [3]:
# Create the video-to-creator mapping

video_creator = (
    item_features[
        ["video_id", "author_id"]
    ]
    .dropna(subset=["video_id", "author_id"])
    .drop_duplicates("video_id")
    .copy()
)

print("Videos with creator mapping:", len(video_creator))
print("Unique creators:", video_creator["author_id"].nunique())

display(video_creator.head(10))

Videos with creator mapping: 10728
Unique creators: 8369


,video_id,author_id
0,0,3309
63,1,4978
126,2,939
189,3,5889
252,4,4284
304,5,3483
367,6,725
430,7,2742
492,8,3427
554,9,4523


In [4]:
# Restrict the creator population to videos used in our experiment

experiment_video_ids = small_matrix["video_id"].unique()

experiment_video_creator = (
    video_creator[
        video_creator["video_id"].isin(experiment_video_ids)
    ]
    .copy()
)

experiment_creators = (
    experiment_video_creator["author_id"]
    .unique()
)

print("Experiment videos:", len(experiment_video_ids))
print("Experiment creators:", len(experiment_creators))

print(
    "Creator mappings:",
    len(experiment_video_creator)
)

Experiment videos: 3327
Experiment creators: 2031
Creator mappings: 3327


In [5]:
# Calculate creator popularity from total historical show counts

creator_stats = (
    item_features
    .groupby("author_id")["show_cnt"]
    .sum()
    .reset_index(name="total_shows")
)

# Keep only creators represented in our experiment
creator_stats_exp = creator_stats[
    creator_stats["author_id"].isin(experiment_creators)
].copy()

print(
    "Creators in experiment:",
    len(creator_stats_exp)
)

print(
    "Average total shows:",
    creator_stats_exp["total_shows"].mean()
)

print(
    "Median total shows:",
    creator_stats_exp["total_shows"].median()
)

print(
    "Minimum total shows:",
    creator_stats_exp["total_shows"].min()
)

print(
    "Maximum total shows:",
    creator_stats_exp["total_shows"].max()
)

Creators in experiment: 2031
Average total shows: 14936152.600689316
Median total shows: 5574655.0
Minimum total shows: 150267
Maximum total shows: 375062603


In [6]:
# Define creator groups using the 25th percentile of total shows

creator_threshold = (
    creator_stats_exp["total_shows"]
    .quantile(0.25)
)

creator_stats_exp["creator_group"] = np.where(
    creator_stats_exp["total_shows"] <= creator_threshold,
    "Emerging",
    "Established"
)

print(
    "Emerging threshold:",
    creator_threshold
)

print("\nCreator counts:")
display(
    creator_stats_exp["creator_group"]
    .value_counts()
)

Emerging threshold: 1139461.5

Creator counts:


creator_group
Established    1523
Emerging        508
Name: count, dtype: int64

In [7]:
# Attach creator popularity and fairness group to each video

video_fairness = experiment_video_creator.merge(
    creator_stats_exp[
        ["author_id", "total_shows", "creator_group"]
    ],
    on="author_id",
    how="left"
)

print("Videos with fairness information:", len(video_fairness))
print("Unique videos:", video_fairness["video_id"].nunique())
print("Missing creator groups:", video_fairness["creator_group"].isna().sum())

display(
    video_fairness.head(10)
)

Videos with fairness information: 3327
Unique videos: 3327
Missing creator groups: 0


,video_id,author_id,total_shows,creator_group
0,103,1862,6886568,Established
1,109,1853,1338871,Established
2,120,4450,2649109,Established
3,122,888,39301005,Established
4,128,2269,27471863,Established
5,130,6512,24265619,Established
6,131,7420,26988517,Established
7,133,2624,1500282,Established
8,136,3542,2044055,Established
9,137,6045,10432120,Established


In [8]:
# Recreate the interest-aware user-video candidate scores

video_tags = (
    item_features[["video_id", "video_tag_name"]]
    .dropna(subset=["video_tag_name"])
    .drop_duplicates("video_id")
)

interaction_videos = small_matrix["video_id"].unique()

video_tags = video_tags[
    video_tags["video_id"].isin(interaction_videos)
].copy()

# Calculate smoothed user interest by original tag
user_history_tags = train_data.merge(
    video_tags,
    on="video_id",
    how="left"
).dropna(subset=["video_tag_name"])

user_interest = (
    user_history_tags
    .groupby(["user_id", "video_tag_name"])
    .agg(
        interactions=("video_id", "count"),
        avg_watch_score=("watch_score", "mean")
    )
    .reset_index()
)

# User-level average engagement
user_means = (
    train_data
    .groupby("user_id")["watch_score"]
    .mean()
    .rename("user_mean_watch_score")
    .reset_index()
)

user_interest = user_interest.merge(
    user_means,
    on="user_id",
    how="left"
)

# Same smoothing used in Notebook 03
smoothing = 5

user_interest["smoothed_interest"] = (
    (
        user_interest["interactions"]
        * user_interest["avg_watch_score"]
        + smoothing * user_interest["user_mean_watch_score"]
    )
    /
    (
        user_interest["interactions"] + smoothing
    )
)

print("Interest profile rows:", len(user_interest))
print("Users:", user_interest["user_id"].nunique())
print("Original tags:", user_interest["video_tag_name"].nunique())

Interest profile rows: 476973
Users: 1411
Original tags: 345


In [9]:
# Create all user-video candidate pairs for tagged videos

video_tag_lookup = (
    video_tags[
        ["video_id", "video_tag_name"]
    ]
    .drop_duplicates("video_id")
    .copy()
)

users = train_data["user_id"].unique()
videos = video_tag_lookup["video_id"].unique()

user_video_pairs = pd.MultiIndex.from_product(
    [users, videos],
    names=["user_id", "video_id"]
).to_frame(index=False)

print("Candidate user-video pairs:", len(user_video_pairs))
print("Users:", user_video_pairs["user_id"].nunique())
print("Videos:", user_video_pairs["video_id"].nunique())

Candidate user-video pairs: 4690164
Users: 1411
Videos: 3324


In [10]:
# Attach creator and fairness information to every candidate video

user_video_candidates = user_video_pairs.merge(
    video_fairness[
        ["video_id", "author_id", "total_shows", "creator_group"]
    ],
    on="video_id",
    how="left"
)

print("Candidate rows:", len(user_video_candidates))
print("Missing creator IDs:", user_video_candidates["author_id"].isna().sum())
print("Missing creator groups:", user_video_candidates["creator_group"].isna().sum())

print("\nCreator group distribution:")
display(
    user_video_candidates["creator_group"].value_counts()
)

Candidate rows: 4690164
Missing creator IDs: 0
Missing creator groups: 0

Creator group distribution:


creator_group
Established    3953622
Emerging        736542
Name: count, dtype: int64

In [12]:
# Attach the video's original tag to each candidate

user_video_candidates = user_video_candidates.merge(
    video_tag_lookup[
        ["video_id", "video_tag_name"]
    ],
    on="video_id",
    how="left"
)

print("Candidate rows:", len(user_video_candidates))
print(
    "Missing video tags:",
    user_video_candidates["video_tag_name"].isna().sum()
)

display(
    user_video_candidates.head(10)
)

Candidate rows: 4690164
Missing video tags: 0


,user_id,video_id,author_id,total_shows,creator_group,video_tag_name
0,14,103,1862,6886568,Established,面食
1,14,109,1853,1338871,Established,摩托车
2,14,120,4450,2649109,Established,搞笑
3,14,122,888,39301005,Established,猴
4,14,128,2269,27471863,Established,美食
5,14,130,6512,24265619,Established,侧面
6,14,131,7420,26988517,Established,电影黑条
7,14,133,2624,1500282,Established,猫
8,14,136,3542,2044055,Established,人群聚集
9,14,137,6045,10432120,Established,影视介绍


In [13]:
# Attach each user's learned interest in the video's tag

user_video_candidates = user_video_candidates.merge(
    user_interest[
        [
            "user_id",
            "video_tag_name",
            "smoothed_interest"
        ]
    ],
    on=["user_id", "video_tag_name"],
    how="left"
)

# No learned interest evidence is treated as zero relevance
user_video_candidates["interest_relevance"] = (
    user_video_candidates["smoothed_interest"].fillna(0)
)

print("Candidate rows:", len(user_video_candidates))
print(
    "Non-zero interest scores:",
    (user_video_candidates["interest_relevance"] > 0).sum()
)
print(
    "Zero interest scores:",
    (user_video_candidates["interest_relevance"] == 0).sum()
)

Candidate rows: 4690164
Non-zero interest scores: 4678474
Zero interest scores: 11690


In [14]:
# Normalize interest relevance separately for each user

user_video_candidates["interest_relevance_norm"] = (
    user_video_candidates
    .groupby("user_id")["interest_relevance"]
    .transform(
        lambda x: (
            (x - x.min()) / (x.max() - x.min())
            if x.max() > x.min()
            else 0
        )
    )
)

print(
    "Minimum normalized relevance:",
    user_video_candidates["interest_relevance_norm"].min()
)

print(
    "Maximum normalized relevance:",
    user_video_candidates["interest_relevance_norm"].max()
)

print("\nUser 14 top interest scores:")
display(
    user_video_candidates[
        user_video_candidates["user_id"] == 14
    ]
    .sort_values(
        "interest_relevance_norm",
        ascending=False
    )
    [["video_id", "video_tag_name",
      "creator_group",
      "interest_relevance",
      "interest_relevance_norm"]]
    .head(10)
)

Minimum normalized relevance: 0.0
Maximum normalized relevance: 1.0

User 14 top interest scores:


,video_id,video_tag_name,creator_group,interest_relevance,interest_relevance_norm
710,2117,孕妇知识,Established,1.378070,1.000000
2343,7157,孕妇知识,Established,1.378070,1.000000
1769,5374,孕妇知识,Established,1.378070,1.000000
2622,8200,穴位按摩,Established,1.306115,0.947786
3009,9594,穴位按摩,Established,1.306115,0.947786
3308,10410,穴位按摩,Established,1.306115,0.947786
1783,5421,聊天截屏,Established,1.290261,0.936281
534,1305,轮滑,Emerging,1.290261,0.936281
3197,10019,工作场所,Established,1.290261,0.936281
1489,4180,饮料,Established,1.285701,0.932972


In [15]:
# Assign a controlled fairness boost to Emerging creators

fairness_boost = 0.10

user_video_candidates["fairness_boost"] = np.where(
    user_video_candidates["creator_group"] == "Emerging",
    fairness_boost,
    0
)

print("Fairness boost:", fairness_boost)

print("\nBoost distribution:")
display(
    user_video_candidates["fairness_boost"].value_counts()
)

Fairness boost: 0.1

Boost distribution:


fairness_boost
0.0    3953622
0.1     736542
Name: count, dtype: int64

In [16]:
# Combine user relevance with the controlled fairness boost

user_video_candidates["fairness_score"] = (
    user_video_candidates["interest_relevance_norm"]
    + user_video_candidates["fairness_boost"]
)

print("Fairness score range:")
print(
    "Minimum:",
    user_video_candidates["fairness_score"].min()
)
print(
    "Maximum:",
    user_video_candidates["fairness_score"].max()
)

print("\nUser 14 top fairness-aware candidates:")

display(
    user_video_candidates[
        user_video_candidates["user_id"] == 14
    ]
    .sort_values(
        "fairness_score",
        ascending=False
    )
    [
        [
            "video_id",
            "video_tag_name",
            "creator_group",
            "interest_relevance_norm",
            "fairness_boost",
            "fairness_score"
        ]
    ]
    .head(10)
)


Fairness score range:
Minimum: 0.0
Maximum: 1.1

User 14 top fairness-aware candidates:


,video_id,video_tag_name,creator_group,interest_relevance_norm,fairness_boost,fairness_score
534,1305,轮滑,Emerging,0.936281,0.1,1.036281
710,2117,孕妇知识,Established,1.000000,0.0,1.000000
2343,7157,孕妇知识,Established,1.000000,0.0,1.000000
1769,5374,孕妇知识,Established,1.000000,0.0,1.000000
204,600,卖包,Emerging,0.882956,0.1,0.982956
929,2519,字幕遮脸,Emerging,0.868023,0.1,0.968023
2856,8736,篮球_篮球场,Emerging,0.866485,0.1,0.966485
381,832,篮球_篮球场,Emerging,0.866485,0.1,0.966485
2365,7190,篮球_篮球场,Emerging,0.866485,0.1,0.966485
2120,6222,篮球_篮球场,Emerging,0.866485,0.1,0.966485


In [17]:
# Generate Top-10 fairness-aware recommendations for every user

fairness_recommendations = (
    user_video_candidates
    .sort_values(
        ["user_id", "fairness_score"],
        ascending=[True, False]
    )
    .groupby("user_id")
    .head(10)
    .reset_index(drop=True)
)

print("Users:", fairness_recommendations["user_id"].nunique())
print("Recommendation slots:", len(fairness_recommendations))
print("Recommendations per user:")
print(
    fairness_recommendations
    .groupby("user_id")
    .size()
    .value_counts()
    .sort_index()
)

Users: 1411
Recommendation slots: 14110
Recommendations per user:
10    1411
Name: count, dtype: int64


In [18]:
# Measure recommendation exposure by creator group

fairness_exposure = (
    fairness_recommendations["creator_group"]
    .value_counts()
    .rename_axis("creator_group")
    .reset_index(name="recommendation_slots")
)

fairness_exposure["exposure_percentage"] = (
    fairness_exposure["recommendation_slots"]
    / len(fairness_recommendations)
    * 100
)

display(fairness_exposure)

,creator_group,recommendation_slots,exposure_percentage
0,Emerging,11218,79.503898
1,Established,2892,20.496102


In [19]:
# Compare recommendation exposure with creator population representation

population_share = {
    "Emerging": 25.0,
    "Established": 75.0
}

fairness_exposure["population_percentage"] = (
    fairness_exposure["creator_group"]
    .map(population_share)
)

fairness_exposure["exposure_gap"] = (
    fairness_exposure["exposure_percentage"]
    - fairness_exposure["population_percentage"]
)

display(fairness_exposure)

,creator_group,recommendation_slots,exposure_percentage,population_percentage,exposure_gap
0,Emerging,11218,79.503898,25.0,54.503898
1,Established,2892,20.496102,75.0,-54.503898


In [20]:
# Test different fairness boost strengths

fairness_strengths = [0.00, 0.02, 0.04, 0.06, 0.08, 0.10]

fairness_sensitivity_results = []

for strength in fairness_strengths:

    # Calculate score for this fairness strength
    test_score = (
        user_video_candidates["interest_relevance_norm"]
        + np.where(
            user_video_candidates["creator_group"] == "Emerging",
            strength,
            0
        )
    )

    # Get Top-10 per user
    temp_recommendations = (
        user_video_candidates.assign(
            test_score=test_score
        )
        .sort_values(
            ["user_id", "test_score"],
            ascending=[True, False]
        )
        .groupby("user_id")
        .head(10)
    )

    # Emerging exposure
    emerging_slots = (
        temp_recommendations["creator_group"]
        .eq("Emerging")
        .sum()
    )

    emerging_exposure = (
        emerging_slots
        / len(temp_recommendations)
        * 100
    )

    exposure_gap = (
        emerging_exposure - 25.0
    )

    fairness_sensitivity_results.append({
        "fairness_strength": strength,
        "emerging_slots": emerging_slots,
        "emerging_exposure": emerging_exposure,
        "exposure_gap": exposure_gap
    })

fairness_sensitivity_results = pd.DataFrame(
    fairness_sensitivity_results
)

display(fairness_sensitivity_results)

,fairness_strength,emerging_slots,emerging_exposure,exposure_gap
0,0.00,3947,27.973069,2.973069
1,0.02,6226,44.124734,19.124734
2,0.04,7624,54.032601,29.032601
3,0.06,8951,63.437279,38.437279
4,0.08,10152,71.948972,46.948972
5,0.10,11218,79.503898,54.503898


In [21]:
# Evaluate recommendation relevance at different fairness strengths

fairness_relevance_results = []

for strength in fairness_strengths:

    # Calculate fairness-adjusted score
    test_score = (
        user_video_candidates["interest_relevance_norm"]
        + np.where(
            user_video_candidates["creator_group"] == "Emerging",
            strength,
            0
        )
    )

    # Generate Top-10 recommendations
    temp_recommendations = (
        user_video_candidates.assign(
            test_score=test_score
        )
        .sort_values(
            ["user_id", "test_score"],
            ascending=[True, False]
        )
        .groupby("user_id")
        .head(10)
    )

    # Evaluate against temporal test data
    test_by_user = (
        test_data
        .groupby("user_id")["video_id"]
        .apply(set)
        .to_dict()
    )

    user_hit_rates = []

    for user_id, recs in temp_recommendations.groupby("user_id"):

        recommended_videos = set(
            recs["video_id"]
        )

        actual_videos = test_by_user.get(
            user_id,
            set()
        )

        hits = len(
            recommended_videos.intersection(
                actual_videos
            )
        )

        hit_rate = hits / 10
        user_hit_rates.append(hit_rate)

    average_hit_rate = np.mean(user_hit_rates)

    fairness_relevance_results.append({
        "fairness_strength": strength,
        "average_hit_rate_at_10": average_hit_rate
    })

fairness_relevance_results = pd.DataFrame(
    fairness_relevance_results
)

display(fairness_relevance_results)

,fairness_strength,average_hit_rate_at_10
0,0.00,0.014174
1,0.02,0.011198
2,0.04,0.009284
3,0.06,0.007583
4,0.08,0.006237
5,0.10,0.005315


In [22]:
# Identify videos each user has already watched during training

user_watched_videos = (
    train_data
    .groupby("user_id")["video_id"]
    .apply(set)
    .to_dict()
)

print(
    "Users with training history:",
    len(user_watched_videos)
)

print(
    "User 14 training-watched videos:",
    len(user_watched_videos[14])
)

Users with training history: 1411
User 14 training-watched videos: 3158


In [23]:
# Remove videos already watched by each user during training

user_video_candidates["already_watched"] = [
    video_id in user_watched_videos[user_id]
    for user_id, video_id
    in zip(
        user_video_candidates["user_id"],
        user_video_candidates["video_id"]
    )
]

recommendation_candidates = (
    user_video_candidates[
        ~user_video_candidates["already_watched"]
    ]
    .copy()
)

print(
    "Original candidate pairs:",
    len(user_video_candidates)
)

print(
    "Remaining recommendation candidates:",
    len(recommendation_candidates)
)

print(
    "Removed previously watched pairs:",
    len(user_video_candidates)
    - len(recommendation_candidates)
)

print(
    "Users remaining:",
    recommendation_candidates["user_id"].nunique()
)

Original candidate pairs: 4690164
Remaining recommendation candidates: 306863
Removed previously watched pairs: 4383301
Users remaining: 1411


In [24]:
# Re-evaluate fairness strengths using only unseen videos

fairness_corrected_results = []

# Test-set videos for each user
test_by_user = (
    test_data
    .groupby("user_id")["video_id"]
    .apply(set)
    .to_dict()
)

for strength in fairness_strengths:

    # Interest relevance + Emerging creator boost
    test_score = (
        recommendation_candidates["interest_relevance_norm"]
        + np.where(
            recommendation_candidates["creator_group"] == "Emerging",
            strength,
            0
        )
    )

    # Generate Top-10 recommendations
    temp_recommendations = (
        recommendation_candidates
        .assign(test_score=test_score)
        .sort_values(
            ["user_id", "test_score"],
            ascending=[True, False]
        )
        .groupby("user_id")
        .head(10)
    )

    # Calculate HitRate@10
    user_hit_rates = []

    for user_id, recs in temp_recommendations.groupby("user_id"):

        recommended_videos = set(
            recs["video_id"]
        )

        actual_videos = test_by_user.get(
            user_id,
            set()
        )

        hits = len(
            recommended_videos.intersection(
                actual_videos
            )
        )

        user_hit_rates.append(hits / 10)

    average_hit_rate = np.mean(user_hit_rates)

    # Calculate Emerging exposure
    emerging_slots = (
        temp_recommendations["creator_group"]
        .eq("Emerging")
        .sum()
    )

    emerging_exposure = (
        emerging_slots
        / len(temp_recommendations)
        * 100
    )

    exposure_gap = (
        emerging_exposure - 25.0
    )

    fairness_corrected_results.append({
        "fairness_strength": strength,
        "emerging_exposure": emerging_exposure,
        "exposure_gap": exposure_gap,
        "average_hit_rate_at_10": average_hit_rate
    })

fairness_corrected_results = pd.DataFrame(
    fairness_corrected_results
)

display(fairness_corrected_results)

,fairness_strength,emerging_exposure,exposure_gap,average_hit_rate_at_10
0,0.00,7.335223,-17.664777,0.438767
1,0.02,11.956060,-13.043940,0.420269
2,0.04,15.889440,-9.110560,0.406662
3,0.06,19.971651,-5.028349,0.390645
4,0.08,25.279943,0.279943,0.371297
5,0.10,31.098512,6.098512,0.352020


In [25]:
# Measure creator coverage under each fairness strength

fairness_coverage_results = []

total_emerging_creators = (
    creator_stats_exp["creator_group"]
    .eq("Emerging")
    .sum()
)

total_established_creators = (
    creator_stats_exp["creator_group"]
    .eq("Established")
    .sum()
)

for strength in fairness_strengths:

    test_score = (
        recommendation_candidates["interest_relevance_norm"]
        + np.where(
            recommendation_candidates["creator_group"] == "Emerging",
            strength,
            0
        )
    )

    temp_recommendations = (
        recommendation_candidates
        .assign(test_score=test_score)
        .sort_values(
            ["user_id", "test_score"],
            ascending=[True, False]
        )
        .groupby("user_id")
        .head(10)
    )

    exposed_creators = (
        temp_recommendations[
            ["author_id", "creator_group"]
        ]
        .drop_duplicates()
    )

    emerging_exposed = (
        exposed_creators["creator_group"]
        .eq("Emerging")
        .sum()
    )

    established_exposed = (
        exposed_creators["creator_group"]
        .eq("Established")
        .sum()
    )

    fairness_coverage_results.append({
        "fairness_strength": strength,
        "emerging_creators_exposed": emerging_exposed,
        "emerging_creator_coverage": (
            emerging_exposed
            / total_emerging_creators
            * 100
        ),
        "established_creators_exposed": established_exposed,
        "established_creator_coverage": (
            established_exposed
            / total_established_creators
            * 100
        )
    })

fairness_coverage_results = pd.DataFrame(
    fairness_coverage_results
)

display(fairness_coverage_results)

,fairness_strength,emerging_creators_exposed,emerging_creator_coverage,established_creators_exposed,established_creator_coverage
0,0.00,198,38.976378,755,49.573211
1,0.02,237,46.653543,742,48.719632
2,0.04,265,52.165354,735,48.260013
3,0.06,284,55.905512,713,46.815496
4,0.08,313,61.614173,694,45.567958
5,0.10,346,68.110236,681,44.714380


In [26]:
# Measure creator diversity in the Top-10 recommendations

creator_diversity_results = []

for strength in fairness_strengths:

    test_score = (
        recommendation_candidates["interest_relevance_norm"]
        + np.where(
            recommendation_candidates["creator_group"] == "Emerging",
            strength,
            0
        )
    )

    temp_recommendations = (
        recommendation_candidates
        .assign(test_score=test_score)
        .sort_values(
            ["user_id", "test_score"],
            ascending=[True, False]
        )
        .groupby("user_id")
        .head(10)
    )

    unique_creators_per_user = (
        temp_recommendations
        .groupby("user_id")["author_id"]
        .nunique()
    )

    creator_diversity_results.append({
        "fairness_strength": strength,
        "mean_unique_creators_at_10": (
            unique_creators_per_user.mean()
        ),
        "median_unique_creators_at_10": (
            unique_creators_per_user.median()
        ),
        "minimum_unique_creators_at_10": (
            unique_creators_per_user.min()
        ),
        "maximum_unique_creators_at_10": (
            unique_creators_per_user.max()
        )
    })

creator_diversity_results = pd.DataFrame(
    creator_diversity_results
)

display(creator_diversity_results)

,fairness_strength,mean_unique_creators_at_10,median_unique_creators_at_10,minimum_unique_creators_at_10,maximum_unique_creators_at_10
0,0.00,9.675408,10.0,5,10
1,0.02,9.697378,10.0,5,10
2,0.04,9.715804,10.0,5,10
3,0.06,9.733522,10.0,5,10
4,0.08,9.751240,10.0,5,10
5,0.10,9.776754,10.0,6,10


In [27]:
# Combine fairness, relevance, coverage, and diversity results

fairness_summary = (
    fairness_corrected_results
    .merge(
        fairness_coverage_results,
        on="fairness_strength"
    )
    .merge(
        creator_diversity_results,
        on="fairness_strength"
    )
)

display(fairness_summary)

,fairness_strength,emerging_exposure,exposure_gap,average_hit_rate_at_10,emerging_creators_exposed,emerging_creator_coverage,established_creators_exposed,established_creator_coverage,mean_unique_creators_at_10,median_unique_creators_at_10,minimum_unique_creators_at_10,maximum_unique_creators_at_10
0,0.00,7.335223,-17.664777,0.438767,198,38.976378,755,49.573211,9.675408,10.0,5,10
1,0.02,11.956060,-13.043940,0.420269,237,46.653543,742,48.719632,9.697378,10.0,5,10
2,0.04,15.889440,-9.110560,0.406662,265,52.165354,735,48.260013,9.715804,10.0,5,10
3,0.06,19.971651,-5.028349,0.390645,284,55.905512,713,46.815496,9.733522,10.0,5,10
4,0.08,25.279943,0.279943,0.371297,313,61.614173,694,45.567958,9.751240,10.0,5,10
5,0.10,31.098512,6.098512,0.352020,346,68.110236,681,44.714380,9.776754,10.0,6,10


In [28]:
# Combine fairness, relevance, coverage, and creator-diversity results

fairness_summary = (
    fairness_corrected_results
    .merge(
        fairness_coverage_results,
        on="fairness_strength"
    )
    .merge(
        creator_diversity_results,
        on="fairness_strength"
    )
)

display(fairness_summary)

,fairness_strength,emerging_exposure,exposure_gap,average_hit_rate_at_10,emerging_creators_exposed,emerging_creator_coverage,established_creators_exposed,established_creator_coverage,mean_unique_creators_at_10,median_unique_creators_at_10,minimum_unique_creators_at_10,maximum_unique_creators_at_10
0,0.00,7.335223,-17.664777,0.438767,198,38.976378,755,49.573211,9.675408,10.0,5,10
1,0.02,11.956060,-13.043940,0.420269,237,46.653543,742,48.719632,9.697378,10.0,5,10
2,0.04,15.889440,-9.110560,0.406662,265,52.165354,735,48.260013,9.715804,10.0,5,10
3,0.06,19.971651,-5.028349,0.390645,284,55.905512,713,46.815496,9.733522,10.0,5,10
4,0.08,25.279943,0.279943,0.371297,313,61.614173,694,45.567958,9.751240,10.0,5,10
5,0.10,31.098512,6.098512,0.352020,346,68.110236,681,44.714380,9.776754,10.0,6,10


In [29]:
# Attach creator and fairness information to every candidate video

user_video_candidates = user_video_pairs.merge(
    video_fairness[
        ["video_id", "author_id", "total_shows", "creator_group"]
    ],
    on="video_id",
    how="left"
)

print("Candidate rows:", len(user_video_candidates))
print("Missing creator IDs:", user_video_candidates["author_id"].isna().sum())
print("Missing creator groups:", user_video_candidates["creator_group"].isna().sum())

print("\nCreator group distribution:")
display(
    user_video_candidates["creator_group"].value_counts()
)

Candidate rows: 4690164
Missing creator IDs: 0
Missing creator groups: 0

Creator group distribution:


creator_group
Established    3953622
Emerging        736542
Name: count, dtype: int64

## Notebook 04 — Fairness Model Findings

The fairness-aware recommender was evaluated using six fairness-strength values
from 0.00 to 0.10. The fairness strength adds an exposure boost to videos
belonging to Emerging creators while retaining the learned user-interest
relevance score.

As fairness strength increased, Emerging-creator exposure increased from
7.34% at strength 0.00 to 31.10% at strength 0.10. Emerging-creator coverage
also increased from 38.98% to 68.11%.

At fairness strength 0.08, Emerging creators received 25.28% of recommendation
slots, compared with their 25% representation in the creator population.
The resulting exposure gap was +0.28 percentage points.

However, increasing the fairness strength was associated with a reduction in
HitRate@10, from 43.88% at strength 0.00 to 35.20% at strength 0.10. This
demonstrates a relevance–fairness trade-off in the current re-ranking
mechanism.

Creator diversity changed only slightly, with mean unique creators per Top-10
increasing from 9.68 at strength 0.00 to 9.78 at strength 0.10.

These results establish that creator exposure can be systematically modified
through fairness-aware re-ranking, while also showing that stronger exposure
intervention can affect recommendation relevance. The subsequent exploration
strategy will investigate whether controlled exploration can introduce
additional creator/content diversity while retaining user-interest signals.

In [1]:
# Cell 1: Fairness model setup

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

# Paths (notebook is inside the notebooks folder)
PROCESSED_DIR = "../processed_data"
RESULTS_DIR = "../results"

# Load baseline recommendations
baseline_recommendations = pd.read_csv(
    os.path.join(RESULTS_DIR, "baseline_recommendations.csv")
)

# Load creator information
video_creator_groups = pd.read_csv(
    os.path.join(PROCESSED_DIR, "video_creator_groups.csv")
)

creator_popularity = pd.read_csv(
    os.path.join(PROCESSED_DIR, "creator_popularity.csv")
)

# Load final user interest profiles
interest_profiles = pd.read_csv(
    os.path.join(PROCESSED_DIR, "interest_profiles.csv")
)

print("Fairness Model data loaded successfully!")

print("\nBaseline recommendations:", baseline_recommendations.shape)
print("Video-creator groups:", video_creator_groups.shape)
print("Creator popularity:", creator_popularity.shape)
print("Interest profiles:", interest_profiles.shape)

print("\nBaseline recommendations:")
display(baseline_recommendations.head())

print("\nVideo-creator groups:")
display(video_creator_groups.head())

print("\nInterest profiles:")
display(interest_profiles.head())

Fairness Model data loaded successfully!

Baseline recommendations: (14110, 3)
Video-creator groups: (3327, 4)
Creator popularity: (2020, 5)
Interest profiles: (40918, 4)

Baseline recommendations:


,user_id,video_id,predicted_preference
0,14,3992,0.997153
1,14,7263,0.995710
2,14,835,0.990655
3,14,6177,0.990421
4,14,6086,0.966525



Video-creator groups:


,video_id,author_id,average_video_daily_shows,creator_group
0,103,1862,120356.982143,Established
1,109,1853,23561.714286,Emerging
2,120,4450,47238.071429,Established
3,122,888,171880.415179,Established
4,128,2269,488870.392857,Established



Interest profiles:


,user_id,category_ids,mean_category_preference,interest_weight
0,14,0,0.701592,0.035495
1,14,1,0.763993,0.038652
2,14,2,0.620291,0.031382
3,14,3,0.553769,0.028016
4,14,4,0.735436,0.037207


In [2]:
# Cell 2: Baseline creator exposure diagnostics

# Merge baseline recommendations with creator classifications
baseline_with_creators = baseline_recommendations.merge(
    video_creator_groups[["video_id", "author_id", "creator_group"]],
    on="video_id",
    how="left"
)

# Check creator mapping
print("Missing creator classifications:")
print(baseline_with_creators["creator_group"].isna().sum())

print("\nCreator group distribution in baseline recommendations:")
display(
    baseline_with_creators["creator_group"]
    .value_counts()
)

print("\nExposure distribution:")
baseline_group_exposure = (
    baseline_with_creators
    .groupby("creator_group")
    .agg(
        recommendation_slots=("video_id", "count"),
        unique_videos=("video_id", "nunique"),
        unique_creators=("author_id", "nunique"),
        unique_users=("user_id", "nunique")
    )
    .reset_index()
)

baseline_group_exposure["exposure_share"] = (
    baseline_group_exposure["recommendation_slots"]
    / baseline_group_exposure["recommendation_slots"].sum()
)

display(baseline_group_exposure)

# Creator repetition diagnostics
creator_repetition = (
    baseline_with_creators
    .groupby(["user_id", "author_id"])
    .size()
    .reset_index(name="videos_per_creator")
)

print("\nCreator repetition per user:")
display(creator_repetition["videos_per_creator"].describe())

print("\nUsers receiving repeated videos from the same creator:")
print(
    (creator_repetition["videos_per_creator"] > 1).sum()
)

Missing creator classifications:
0

Creator group distribution in baseline recommendations:


creator_group
Established    10432
Emerging        3678
Name: count, dtype: int64


Exposure distribution:


,creator_group,recommendation_slots,unique_videos,unique_creators,unique_users,exposure_share
0,Emerging,3678,278,262,1284,0.260666
1,Established,10432,956,624,1411,0.739334



Creator repetition per user:


count    13767.000000
mean         1.024915
std          0.193317
min          1.000000
25%          1.000000
50%          1.000000
75%          1.000000
max          6.000000
Name: videos_per_creator, dtype: float64


Users receiving repeated videos from the same creator:
281


In [3]:
# Cell 3: Inspect creator-level baseline exposure

baseline_creator_detail = pd.read_csv(
    os.path.join(RESULTS_DIR, "baseline_creator_exposure_detail.csv")
)

print("Creator exposure dataset shape:", baseline_creator_detail.shape)
print("\nColumns:")
print(baseline_creator_detail.columns.tolist())

print("\nSample:")
display(baseline_creator_detail.head(10))

print("\nCreators with zero exposure:")
display(baseline_creator_detail.head(10))

Creator exposure dataset shape: (2019, 3)

Columns:
['author_id', 'creator_group', 'recommendation_count']

Sample:


,author_id,creator_group,recommendation_count
0,1862,Established,1
1,1853,Emerging,4
2,4450,Established,26
3,888,Established,3
4,2269,Established,0
5,6512,Established,0
6,7420,Established,0
7,2624,Emerging,21
8,3542,Emerging,5
9,6045,Established,1



Creators with zero exposure:


,author_id,creator_group,recommendation_count
0,1862,Established,1
1,1853,Emerging,4
2,4450,Established,26
3,888,Established,3
4,2269,Established,0
5,6512,Established,0
6,7420,Established,0
7,2624,Emerging,21
8,3542,Emerging,5
9,6045,Established,1


In [4]:
# Cell 3: Creator-level exposure concentration analysis

# Separate Emerging and Established creators
emerging_creators = baseline_creator_detail[
    baseline_creator_detail["creator_group"] == "Emerging"
].copy()

established_creators = baseline_creator_detail[
    baseline_creator_detail["creator_group"] == "Established"
].copy()

# Identify creators receiving zero recommendations
zero_exposure_creators = baseline_creator_detail[
    baseline_creator_detail["recommendation_count"] == 0
].copy()

zero_exposure_emerging = emerging_creators[
    emerging_creators["recommendation_count"] == 0
]

zero_exposure_established = established_creators[
    established_creators["recommendation_count"] == 0
]

# Print counts
print("Total creators:", len(baseline_creator_detail))
print("Emerging creators:", len(emerging_creators))
print("Established creators:", len(established_creators))

print("\nZero-exposure creators:")
print("Total:", len(zero_exposure_creators))
print("Emerging:", len(zero_exposure_emerging))
print("Established:", len(zero_exposure_established))

print("\nZero-exposure percentage:")
print(
    "Emerging:",
    round(len(zero_exposure_emerging) / len(emerging_creators) * 100, 2),
    "%"
)

print(
    "Established:",
    round(len(zero_exposure_established) / len(established_creators) * 100, 2),
    "%"
)

print("\nEmerging creators with zero exposure:")
display(zero_exposure_emerging.head(10))

print("\nCreators with the highest recommendation exposure:")
display(
    baseline_creator_detail
    .sort_values("recommendation_count", ascending=False)
    .head(10)
)

Total creators: 2019
Emerging creators: 504
Established creators: 1515

Zero-exposure creators:
Total: 1133
Emerging: 242
Established: 891

Zero-exposure percentage:
Emerging: 48.02 %
Established: 58.81 %

Emerging creators with zero exposure:


,author_id,creator_group,recommendation_count
10,3159,Emerging,0
23,2376,Emerging,0
63,7050,Emerging,0
73,3282,Emerging,0
105,1146,Emerging,0
115,5123,Emerging,0
124,1189,Emerging,0
125,3618,Emerging,0
135,2687,Emerging,0
159,1980,Emerging,0



Creators with the highest recommendation exposure:


,author_id,creator_group,recommendation_count
27,5690,Established,342
224,5718,Established,209
15,6332,Established,197
297,7318,Established,186
265,6783,Established,169
88,6476,Established,168
391,4394,Established,168
249,6911,Established,156
149,2230,Established,150
370,239,Emerging,126


In [5]:
# Cell 4: Quantify creator-level exposure concentration

def creator_exposure_summary(group_data, group_name):

    exposed = group_data[
        group_data["recommendation_count"] > 0
    ].copy()

    total_exposure = group_data["recommendation_count"].sum()

    exposed["exposure_share"] = (
        exposed["recommendation_count"] / total_exposure
    )

    exposed = exposed.sort_values(
        "recommendation_count",
        ascending=False
    )

    print(f"\n{group_name} CREATOR EXPOSURE")
    print("-" * 40)

    print("Total creators:", len(group_data))
    print("Exposed creators:", len(exposed))
    print("Total recommendation slots:", total_exposure)

    print("\nExposure statistics:")
    display(group_data["recommendation_count"].describe())

    print("\nTop 10 creators' share of total group exposure:")

    for n in [1, 5, 10, 20]:
        share = (
            exposed.head(n)["recommendation_count"].sum()
            / total_exposure
        ) * 100

        print(f"Top {n} creators: {share:.2f}%")

    print("\nTop 10 exposed creators:")
    display(exposed.head(10))


creator_exposure_summary(
    emerging_creators,
    "EMERGING"
)

creator_exposure_summary(
    established_creators,
    "ESTABLISHED"
)


EMERGING CREATOR EXPOSURE
----------------------------------------
Total creators: 504
Exposed creators: 262
Total recommendation slots: 3678

Exposure statistics:


count    504.000000
mean       7.297619
std       14.728649
min        0.000000
25%        0.000000
50%        1.000000
75%        8.000000
max      126.000000
Name: recommendation_count, dtype: float64


Top 10 creators' share of total group exposure:
Top 1 creators: 3.43%
Top 5 creators: 12.89%
Top 10 creators: 21.34%
Top 20 creators: 34.37%

Top 10 exposed creators:


,author_id,creator_group,recommendation_count,exposure_share
370,239,Emerging,126,0.034258
1302,5509,Emerging,99,0.026917
687,5167,Emerging,91,0.024742
228,2291,Emerging,87,0.023654
1768,784,Emerging,71,0.019304
1247,7121,Emerging,65,0.017673
1897,3630,Emerging,64,0.017401
1295,4537,Emerging,64,0.017401
665,4016,Emerging,60,0.016313
1405,1711,Emerging,58,0.015769



ESTABLISHED CREATOR EXPOSURE
----------------------------------------
Total creators: 1515
Exposed creators: 624
Total recommendation slots: 10432

Exposure statistics:


count    1515.000000
mean        6.885809
std        21.028905
min         0.000000
25%         0.000000
50%         0.000000
75%         4.000000
max       342.000000
Name: recommendation_count, dtype: float64


Top 10 creators' share of total group exposure:
Top 1 creators: 3.28%
Top 5 creators: 10.57%
Top 10 creators: 17.91%
Top 20 creators: 27.40%

Top 10 exposed creators:


,author_id,creator_group,recommendation_count,exposure_share
27,5690,Established,342,0.032784
224,5718,Established,209,0.020035
15,6332,Established,197,0.018884
297,7318,Established,186,0.017830
265,6783,Established,169,0.016200
88,6476,Established,168,0.016104
391,4394,Established,168,0.016104
249,6911,Established,156,0.014954
149,2230,Established,150,0.014379
1073,6263,Established,123,0.011791


In [6]:
# Cell 4B: Emerging creator exposure concentration

exposed_emerging = emerging_creators[
    emerging_creators["recommendation_count"] > 0
].sort_values(
    "recommendation_count",
    ascending=False
)

total_emerging_exposure = emerging_creators[
    "recommendation_count"
].sum()

print("Emerging creator exposure concentration:")

for n in [1, 5, 10, 20]:
    share = (
        exposed_emerging.head(n)["recommendation_count"].sum()
        / total_emerging_exposure
    ) * 100

    print(f"Top {n} creators: {share:.2f}%")

print("\nTop 10 Emerging creators:")
display(exposed_emerging.head(10))

Emerging creator exposure concentration:
Top 1 creators: 3.43%
Top 5 creators: 12.89%
Top 10 creators: 21.34%
Top 20 creators: 34.37%

Top 10 Emerging creators:


,author_id,creator_group,recommendation_count
370,239,Emerging,126
1302,5509,Emerging,99
687,5167,Emerging,91
228,2291,Emerging,87
1768,784,Emerging,71
1247,7121,Emerging,65
1897,3630,Emerging,64
1295,4537,Emerging,64
665,4016,Emerging,60
1405,1711,Emerging,58


In [7]:
# Cell 5: Inspect baseline recommendation pipeline requirements

print("Baseline recommendation columns:")
print(baseline_recommendations.columns.tolist())

print("\nBaseline recommendation shape:")
print(baseline_recommendations.shape)

print("\nUnique users:")
print(baseline_recommendations["user_id"].nunique())

print("\nUnique recommended videos:")
print(baseline_recommendations["video_id"].nunique())

print("\nRecommendations per user:")
display(
    baseline_recommendations
    .groupby("user_id")
    .size()
    .describe()
)

print("\nBaseline recommendation sample:")
display(
    baseline_recommendations
    .sort_values(
        ["user_id", "predicted_preference"],
        ascending=[True, False]
    )
    .head(20)
)

Baseline recommendation columns:
['user_id', 'video_id', 'predicted_preference']

Baseline recommendation shape:
(14110, 3)

Unique users:
1411

Unique recommended videos:
1234

Recommendations per user:


count    1411.0
mean       10.0
std         0.0
min        10.0
25%        10.0
50%        10.0
75%        10.0
max        10.0
dtype: float64


Baseline recommendation sample:


,user_id,video_id,predicted_preference
0,14,3992,0.997153
1,14,7263,0.995710
2,14,835,0.990655
3,14,6177,0.990421
4,14,6086,0.966525
5,14,3141,0.962310
6,14,3107,0.961977
7,14,152,0.950126
8,14,2065,0.949943
9,14,4268,0.945336


In [9]:
from sklearn.metrics.pairwise import cosine_similarity
# Cell 6: Reconstruct the User-CF scoring components

# Load training interactions
training_interactions = pd.read_csv(
    os.path.join(PROCESSED_DIR, "training_interactions.csv")
)

# Reconstruct the preference score used by the baseline
training_interactions["preference_score"] = (
    training_interactions["watch_ratio"].clip(0, 1)
)

# Construct user-item interaction matrix
user_item_matrix = training_interactions.pivot(
    index="user_id",
    columns="video_id",
    values="preference_score"
).fillna(0)

print("User-item matrix shape:", user_item_matrix.shape)
print("Number of users:", user_item_matrix.shape[0])
print("Number of candidate videos:", user_item_matrix.shape[1])

# Recalculate user-user cosine similarity
user_similarity_matrix = cosine_similarity(user_item_matrix)

user_similarity_df = pd.DataFrame(
    user_similarity_matrix,
    index=user_item_matrix.index,
    columns=user_item_matrix.index
)

print("\nUser similarity matrix shape:", user_similarity_df.shape)

# Verify similarity properties
print("\nSimilarity matrix diagonal:")
print(user_similarity_df.values.diagonal()[:5])

print("\nSimilarity statistics:")
display(user_similarity_df.stack().describe())

User-item matrix shape: (1411, 3288)
Number of users: 1411
Number of candidate videos: 3288

User similarity matrix shape: (1411, 1411)

Similarity matrix diagonal:
[1. 1. 1. 1. 1.]

Similarity statistics:


count    1.990921e+06
mean     8.745551e-01
std      5.448672e-02
min      4.907042e-01
25%      8.461624e-01
50%      8.883674e-01
75%      9.146522e-01
max      1.000000e+00
dtype: float64

In [10]:
# Cell 7: Generate Top-50 baseline candidate pool

TOP_CANDIDATES = 50
N_NEIGHBORS = 20

# Convert matrices to NumPy arrays
user_ids = user_item_matrix.index.to_numpy()
video_ids = user_item_matrix.columns.to_numpy()

interaction_array = user_item_matrix.to_numpy()
similarity_array = user_similarity_df.to_numpy()

# Build weighted similarity matrix using the 20 nearest neighbors
neighbor_similarity = np.zeros_like(similarity_array)

for user_idx in range(len(user_ids)):

    # Exclude the user themselves
    similarities = similarity_array[user_idx].copy()
    similarities[user_idx] = -np.inf

    # Select 20 nearest neighbors
    neighbor_indices = np.argsort(similarities)[-N_NEIGHBORS:]

    # Retain similarities of selected neighbors
    neighbor_similarity[user_idx, neighbor_indices] = (
        similarity_array[user_idx, neighbor_indices]
    )

# Predict preferences using similarity-weighted neighbor preferences
weighted_scores = neighbor_similarity @ interaction_array

similarity_sum = neighbor_similarity.sum(axis=1, keepdims=True)

predicted_scores = weighted_scores / np.maximum(similarity_sum, 1e-10)

# Exclude videos already interacted with by each user
predicted_scores[interaction_array > 0] = -np.inf

print("Predicted score matrix shape:", predicted_scores.shape)

# Generate Top-50 candidates for each user
candidate_records = []

for user_idx, user_id in enumerate(user_ids):

    scores = predicted_scores[user_idx]

    # Select the highest-scoring unseen videos
    candidate_indices = np.argsort(scores)[::-1][:TOP_CANDIDATES]

    for rank, video_idx in enumerate(candidate_indices, start=1):

        candidate_records.append({
            "user_id": user_id,
            "video_id": video_ids[video_idx],
            "predicted_preference": scores[video_idx],
            "candidate_rank": rank
        })

baseline_candidates = pd.DataFrame(candidate_records)

print("\nCandidate pool shape:", baseline_candidates.shape)
print("Unique users:", baseline_candidates["user_id"].nunique())
print("Unique videos:", baseline_candidates["video_id"].nunique())

print("\nCandidates per user:")
display(baseline_candidates.groupby("user_id").size().describe())

print("\nCandidate sample:")
display(baseline_candidates.head(15))

Predicted score matrix shape: (1411, 3288)

Candidate pool shape: (70550, 4)
Unique users: 1411
Unique videos: 2909

Candidates per user:


count    1411.0
mean       50.0
std         0.0
min        50.0
25%        50.0
50%        50.0
75%        50.0
max        50.0
dtype: float64


Candidate sample:


,user_id,video_id,predicted_preference,candidate_rank
0,14,3992,0.997153,1
1,14,7263,0.995710,2
2,14,835,0.990655,3
3,14,6177,0.990421,4
4,14,4026,0.974648,5
5,14,6086,0.966525,6
6,14,3141,0.962310,7
7,14,3107,0.961977,8
8,14,152,0.950126,9
9,14,2065,0.949943,10


In [11]:
# Cell 8: Validate reconstructed Top-10 against original baseline

# Extract reconstructed Top-10 from the Top-50 candidate pool
reconstructed_top10 = (
    baseline_candidates
    .sort_values(["user_id", "candidate_rank"])
    .groupby("user_id")
    .head(10)
    .copy()
)

# Compare original and reconstructed recommendations
original_top10 = baseline_recommendations.copy()

comparison = original_top10.merge(
    reconstructed_top10[["user_id", "video_id", "candidate_rank"]],
    on=["user_id", "video_id"],
    how="left"
)

# Top-10 overlap per user
overlap_per_user = (
    comparison.groupby("user_id")["candidate_rank"]
    .apply(lambda x: x.notna().sum())
)

print("Top-10 overlap per user:")
display(overlap_per_user.describe())

print("\nUsers with all 10 original videos present in reconstructed Top-10:")
print((overlap_per_user == 10).sum())

print("\nUsers with all 10 original videos present in Top-50:")
print(
    comparison.groupby("user_id")["candidate_rank"]
    .apply(lambda x: x.notna().sum())
    .eq(10)
    .sum()
)

# Exact ranking agreement
original_rankings = (
    original_top10
    .sort_values(["user_id", "predicted_preference"], ascending=[True, False])
    .groupby("user_id")["video_id"]
    .apply(list)
)

reconstructed_rankings = (
    reconstructed_top10
    .sort_values(["user_id", "candidate_rank"])
    .groupby("user_id")["video_id"]
    .apply(list)
)

ranking_agreement = (
    original_rankings == reconstructed_rankings
)

print("\nExact Top-10 ranking agreement:")
print(ranking_agreement.sum(), "/", len(ranking_agreement))

print("\nExact ranking agreement percentage:")
print(round(ranking_agreement.mean() * 100, 2), "%")

# Compare score differences for videos appearing in both lists
score_comparison = original_top10.merge(
    reconstructed_top10[["user_id", "video_id", "predicted_preference"]],
    on=["user_id", "video_id"],
    suffixes=("_original", "_reconstructed")
)

score_comparison["score_difference"] = (
    score_comparison["predicted_preference_original"]
    - score_comparison["predicted_preference_reconstructed"]
)

print("\nMatched recommendation score differences:")
display(score_comparison["score_difference"].abs().describe())

print("\nValidation complete.")

Top-10 overlap per user:


count    1411.000000
mean        8.145287
std         2.194322
min         0.000000
25%         7.000000
50%         9.000000
75%        10.000000
max        10.000000
Name: candidate_rank, dtype: float64


Users with all 10 original videos present in reconstructed Top-10:
536

Users with all 10 original videos present in Top-50:
536

Exact Top-10 ranking agreement:
264 / 1411

Exact ranking agreement percentage:
18.71 %

Matched recommendation score differences:


count    1.149300e+04
mean     1.566658e-16
std      1.362359e-16
min      0.000000e+00
25%      0.000000e+00
50%      1.110223e-16
75%      2.220446e-16
max      7.771561e-16
Name: score_difference, dtype: float64


Validation complete.


In [12]:
# Cell 9: Validate original Top-10 against actual Top-50 candidates

# Check original recommendations against Top-50 candidate pool
top50_comparison = original_top10.merge(
    baseline_candidates[["user_id", "video_id", "candidate_rank"]],
    on=["user_id", "video_id"],
    how="left"
)

# Count original videos found in Top-50 for each user
top50_overlap_per_user = (
    top50_comparison.groupby("user_id")["candidate_rank"]
    .apply(lambda x: x.notna().sum())
)

print("Original Top-10 videos found in reconstructed Top-50:")
display(top50_overlap_per_user.describe())

print(
    "\nUsers with all 10 original recommendations inside reconstructed Top-50:",
    (top50_overlap_per_user == 10).sum()
)

print(
    "Users with at least 8 original recommendations inside reconstructed Top-50:",
    (top50_overlap_per_user >= 8).sum()
)

# Check rank positions of the original recommendations within Top-50
print("\nRank distribution of original recommendations inside Top-50:")
display(top50_comparison["candidate_rank"].describe())

# Check whether original recommendation scores match reconstructed scores
top50_score_comparison = original_top10.merge(
    baseline_candidates[["user_id", "video_id", "predicted_preference"]],
    on=["user_id", "video_id"],
    how="inner",
    suffixes=("_original", "_candidate")
)

top50_score_comparison["absolute_score_difference"] = (
    top50_score_comparison["predicted_preference_original"]
    - top50_score_comparison["predicted_preference_candidate"]
).abs()

print("\nScore consistency for videos found in Top-50:")
display(top50_score_comparison["absolute_score_difference"].describe())

print("\nValidation completed.")

Original Top-10 videos found in reconstructed Top-50:


count    1411.000000
mean        9.952516
std         0.597761
min         0.000000
25%        10.000000
50%        10.000000
75%        10.000000
max        10.000000
Name: candidate_rank, dtype: float64


Users with all 10 original recommendations inside reconstructed Top-50: 1398
Users with at least 8 original recommendations inside reconstructed Top-50: 1403

Rank distribution of original recommendations inside Top-50:


count    14043.000000
mean         7.307413
std          5.229622
min          1.000000
25%          4.000000
50%          7.000000
75%         10.000000
max         49.000000
Name: candidate_rank, dtype: float64


Score consistency for videos found in Top-50:


count    1.404300e+04
mean     1.559672e-16
std      1.350235e-16
min      0.000000e+00
25%      0.000000e+00
50%      1.110223e-16
75%      2.220446e-16
max      7.771561e-16
Name: absolute_score_difference, dtype: float64


Validation completed.


In [13]:
# Cell 10: Investigate unmatched original baseline recommendations

# Identify original recommendations missing from Top-50 candidates
missing_original_recommendations = original_top10.merge(
    baseline_candidates[["user_id", "video_id"]],
    on=["user_id", "video_id"],
    how="left",
    indicator=True
)

missing_original_recommendations = (
    missing_original_recommendations[
        missing_original_recommendations["_merge"] == "left_only"
    ]
    .drop(columns="_merge")
)

print("Missing original recommendation rows:")
print(len(missing_original_recommendations))

print("\nUsers affected:")
print(
    missing_original_recommendations["user_id"].nunique()
)

print("\nMissing recommendations by user:")
display(
    missing_original_recommendations
    .groupby("user_id")
    .size()
    .sort_values(ascending=False)
)

print("\nMissing recommendation details:")
display(missing_original_recommendations.head(20))

Missing original recommendation rows:
67

Users affected:
13

Missing recommendations by user:


user_id
2568    10
3706    10
1731     9
4653     9
4882     9
6991     5
368      4
6658     3
2367     2
2461     2
4673     2
6415     1
4924     1
dtype: int64


Missing recommendation details:


,user_id,video_id,predicted_preference
646,368,765,0.967640
647,368,3141,0.949570
648,368,3183,0.949143
649,368,2065,0.938490
3281,1731,6764,0.966080
3282,1731,6309,0.960058
3283,1731,9592,0.931251
3284,1731,3998,0.922603
3285,1731,6747,0.908122
3286,1731,2970,0.907709


In [14]:
# Cell 11: Diagnose missing original recommendations in the reconstructed ranking

missing_rank_records = []

for _, row in missing_original_recommendations.iterrows():

    user_id = row["user_id"]
    video_id = row["video_id"]
    original_score = row["predicted_preference"]

    user_idx = user_item_matrix.index.get_loc(user_id)
    video_idx = user_item_matrix.columns.get_loc(video_id)

    reconstructed_score = predicted_scores[user_idx, video_idx]

    # Rank in the complete reconstructed candidate list
    user_scores = predicted_scores[user_idx]

    reconstructed_rank = (
        (user_scores > reconstructed_score).sum() + 1
    )

    missing_rank_records.append({
        "user_id": user_id,
        "video_id": video_id,
        "original_score": original_score,
        "reconstructed_score": reconstructed_score,
        "score_difference": abs(original_score - reconstructed_score),
        "reconstructed_rank": reconstructed_rank
    })

missing_rank_diagnostics = pd.DataFrame(missing_rank_records)

display(
    missing_rank_diagnostics.sort_values(
        "reconstructed_rank"
    ).head(20)
)

print("\nReconstructed rank distribution:")
display(
    missing_rank_diagnostics["reconstructed_rank"].describe()
)

print("\nMissing videos ranked within Top-50:")
print(
    (missing_rank_diagnostics["reconstructed_rank"] <= 50).sum()
)

print("\nMissing videos ranked within Top-100:")
print(
    (missing_rank_diagnostics["reconstructed_rank"] <= 100).sum()
)

print("\nScore difference summary:")
display(
    missing_rank_diagnostics["score_difference"].describe()
)

,user_id,video_id,original_score,reconstructed_score,score_difference,reconstructed_rank
57,4924.0,4291.0,0.978539,0.978539,2.220446e-16,51
13,2367.0,6063.0,0.980868,0.980868,1.110223e-16,52
46,4673.0,10087.0,0.980085,0.980085,2.220446e-16,52
48,4882.0,5834.0,0.892754,0.892754,0.000000e+00,52
58,6415.0,1415.0,0.718385,0.718385,1.110223e-16,52
14,2367.0,7778.0,0.977662,0.977662,2.220446e-16,55
62,6991.0,5535.0,0.919452,0.919452,2.220446e-16,55
47,4673.0,9096.0,0.976977,0.976977,1.110223e-16,58
63,6991.0,8922.0,0.907465,0.907465,1.110223e-16,59
64,6991.0,833.0,0.905115,0.905115,3.330669e-16,60



Reconstructed rank distribution:


count     67.000000
mean     178.716418
std      150.497740
min       51.000000
25%       67.000000
50%       95.000000
75%      236.500000
max      543.000000
Name: reconstructed_rank, dtype: float64


Missing videos ranked within Top-50:
0

Missing videos ranked within Top-100:
36

Score difference summary:


count    6.700000e+01
mean     1.126794e-16
std      1.013353e-16
min      0.000000e+00
25%      0.000000e+00
50%      1.110223e-16
75%      2.220446e-16
max      3.330669e-16
Name: score_difference, dtype: float64

In [16]:
# Cell 12: Diagnose baseline reconstruction for one affected user
K_NEIGHBORS = 20

diagnostic_user = 2568

# Original-style neighbour selection
original_similarities = user_similarity_df.loc[diagnostic_user].drop(diagnostic_user)
original_nearest_users = original_similarities.nlargest(K_NEIGHBORS)

# Original-style weighted preference calculation
original_neighbour_ratings = interaction_matrix.loc[original_nearest_users.index]

original_weighted_scores = np.dot(
    original_nearest_users.values,
    original_neighbour_ratings.values
)

original_similarity_sum = np.abs(original_nearest_users.values).sum()

original_predicted_scores = pd.Series(
    original_weighted_scores / original_similarity_sum,
    index=interaction_matrix.columns
)

# Remove videos already watched by the target user
watched_videos = set(
    training_interactions.loc[
        training_interactions["user_id"] == diagnostic_user,
        "video_id"
    ]
)

original_candidate_scores = original_predicted_scores[
    ~original_predicted_scores.index.isin(watched_videos)
]

# Reproduce original deterministic ranking
original_top50 = (
    original_candidate_scores
    .rename("predicted_preference")
    .reset_index()
    .sort_values(
        ["predicted_preference", "video_id"],
        ascending=[False, True],
        kind="mergesort"
    )
    .head(50)
    .reset_index(drop=True)
)

original_top50["original_rank"] = np.arange(1, 51)

print("User:", diagnostic_user)
print("Original-style candidate pool:", len(original_top50))

display(original_top50.head(20))

NameError: name 'interaction_matrix' is not defined

In [18]:
# Identify available matrices safely

global_vars = list(globals().items())

print("Possible interaction matrices:")
for name, obj in global_vars:
    if isinstance(obj, pd.DataFrame) and obj.shape[0] > 100 and obj.shape[1] > 100:
        print(f"{name}: {obj.shape}")

print("\nPossible similarity matrices:")
for name, obj in global_vars:
    if isinstance(obj, pd.DataFrame) and obj.shape[0] == obj.shape[1] and obj.shape[0] > 100:
        print(f"{name}: {obj.shape}")

Possible interaction matrices:
user_item_matrix: (1411, 3288)
user_similarity_df: (1411, 1411)

Possible similarity matrices:
user_similarity_df: (1411, 1411)


In [19]:
# Cell 12: Diagnose baseline reconstruction for one affected user

diagnostic_user = 2568

# 1. Select nearest neighbours using original pandas logic
original_similarities = (
    user_similarity_df.loc[diagnostic_user]
    .drop(diagnostic_user)
)

original_nearest_users = original_similarities.nlargest(20)

# 2. Retrieve neighbour preference vectors
original_neighbour_ratings = user_item_matrix.loc[
    original_nearest_users.index
]

# 3. Calculate weighted preference scores
original_weighted_scores = np.dot(
    original_nearest_users.values,
    original_neighbour_ratings.values
)

original_similarity_sum = original_nearest_users.values.sum()

original_predicted_scores = pd.Series(
    original_weighted_scores / original_similarity_sum,
    index=user_item_matrix.columns
)

# 4. Exclude videos already watched by this user
watched_videos = set(
    user_item_matrix.loc[diagnostic_user][
        user_item_matrix.loc[diagnostic_user] > 0
    ].index
)

original_candidate_scores = original_predicted_scores[
    ~original_predicted_scores.index.isin(watched_videos)
]

# 5. Reproduce original deterministic ranking
original_top50 = (
    original_candidate_scores
    .rename("predicted_preference")
    .reset_index()
    .sort_values(
        ["predicted_preference", "video_id"],
        ascending=[False, True],
        kind="mergesort"
    )
    .head(50)
    .reset_index(drop=True)
)

original_top50["original_rank"] = np.arange(1, 51)

print("Diagnostic user:", diagnostic_user)
print("Nearest neighbours:", len(original_nearest_users))
print("Watched videos excluded:", len(watched_videos))
print("Candidate videos available:", len(original_candidate_scores))
print("Top-50 generated:", len(original_top50))

display(original_top50.head(20))

Diagnostic user: 2568
Nearest neighbours: 20
Watched videos excluded: 1606
Candidate videos available: 1682
Top-50 generated: 50


,video_id,predicted_preference,original_rank
0,133,1.0,1
1,217,1.0,2
2,250,1.0,3
3,289,1.0,4
4,336,1.0,5
5,517,1.0,6
6,600,1.0,7
7,619,1.0,8
8,634,1.0,9
9,639,1.0,10


In [20]:
# Cell 12: Diagnose baseline reconstruction for one affected user

diagnostic_user = 2568

# 1. Select nearest neighbours using original pandas logic
original_similarities = (
    user_similarity_df.loc[diagnostic_user]
    .drop(diagnostic_user)
)

original_nearest_users = original_similarities.nlargest(20)

# 2. Retrieve neighbour preference vectors
original_neighbour_ratings = user_item_matrix.loc[
    original_nearest_users.index
]

# 3. Calculate weighted preference scores
original_weighted_scores = np.dot(
    original_nearest_users.values,
    original_neighbour_ratings.values
)

original_similarity_sum = original_nearest_users.values.sum()

original_predicted_scores = pd.Series(
    original_weighted_scores / original_similarity_sum,
    index=user_item_matrix.columns
)

# 4. Exclude videos already watched by this user
watched_videos = set(
    user_item_matrix.loc[diagnostic_user][
        user_item_matrix.loc[diagnostic_user] > 0
    ].index
)

original_candidate_scores = original_predicted_scores[
    ~original_predicted_scores.index.isin(watched_videos)
]

# 5. Reproduce original deterministic ranking
original_top50 = (
    original_candidate_scores
    .rename("predicted_preference")
    .reset_index()
    .sort_values(
        ["predicted_preference", "video_id"],
        ascending=[False, True],
        kind="mergesort"
    )
    .head(50)
    .reset_index(drop=True)
)

original_top50["original_rank"] = np.arange(1, 51)

print("Diagnostic user:", diagnostic_user)
print("Nearest neighbours:", len(original_nearest_users))
print("Watched videos excluded:", len(watched_videos))
print("Candidate videos available:", len(original_candidate_scores))
print("Top-50 generated:", len(original_top50))

display(original_top50.head(20))

Diagnostic user: 2568
Nearest neighbours: 20
Watched videos excluded: 1606
Candidate videos available: 1682
Top-50 generated: 50


,video_id,predicted_preference,original_rank
0,133,1.0,1
1,217,1.0,2
2,250,1.0,3
3,289,1.0,4
4,336,1.0,5
5,517,1.0,6
6,600,1.0,7
7,619,1.0,8
8,634,1.0,9
9,639,1.0,10


In [21]:
# Compare original-style and reconstructed Top-50 for user 2568

reconstructed_top50_user = baseline_candidates[
    baseline_candidates["user_id"] == diagnostic_user
].sort_values("candidate_rank")

original_videos = set(original_top50["video_id"])
reconstructed_videos = set(reconstructed_top50_user["video_id"])

print("Original Top-50:", len(original_videos))
print("Reconstructed Top-50:", len(reconstructed_videos))
print("Matching videos:", len(original_videos & reconstructed_videos))
print("Missing from reconstruction:", len(original_videos - reconstructed_videos))
print("Extra in reconstruction:", len(reconstructed_videos - original_videos))

print("\nOriginal-only videos:")
print(sorted(original_videos - reconstructed_videos))

print("\nReconstructed-only videos:")
print(sorted(reconstructed_videos - original_videos))

Original Top-50: 50
Reconstructed Top-50: 50
Matching videos: 34
Missing from reconstruction: 16
Extra in reconstruction: 16

Original-only videos:
[133, 517, 1412, 1445, 1945, 2312, 2343, 2383, 2458, 2472, 3720, 3722, 3742, 4740, 5453, 5464]

Reconstructed-only videos:
[6778, 7070, 7119, 7135, 7158, 7169, 7299, 7300, 9615, 9758, 9794, 9804, 9830, 10095, 10217, 10382]


In [23]:
# Cell 14: Compare score distributions and candidate differences

# Recreate the comparison sets
original_videos = set(original_top50["video_id"])
reconstructed_videos = set(reconstructed_top50_user["video_id"])

original_only = original_videos - reconstructed_videos
reconstructed_only = reconstructed_videos - original_videos

print("Matching videos:", len(original_videos & reconstructed_videos))
print("Original-only videos:", len(original_only))
print("Reconstructed-only videos:", len(reconstructed_only))

print("\nORIGINAL TOP-50 SCORE DISTRIBUTION")
display(original_top50["predicted_preference"].value_counts())

print("\nRECONSTRUCTED TOP-50 SCORE DISTRIBUTION")
display(reconstructed_top50_user["predicted_preference"].value_counts())

print("\nORIGINAL TOP-50 BOUNDARY (last 10)")
display(original_top50.tail(10))

print("\nRECONSTRUCTED TOP-50 BOUNDARY (last 10)")
display(reconstructed_top50_user.tail(10))

print("\nOriginal-only videos")
display(
    original_top50[
        original_top50["video_id"].isin(original_only)
    ].sort_values("original_rank")
)

print("\nReconstructed-only videos")
display(
    reconstructed_top50_user[
        reconstructed_top50_user["video_id"].isin(reconstructed_only)
    ].sort_values("candidate_rank")
)

Matching videos: 34
Original-only videos: 16
Reconstructed-only videos: 16

ORIGINAL TOP-50 SCORE DISTRIBUTION


predicted_preference
1.0    50
Name: count, dtype: int64


RECONSTRUCTED TOP-50 SCORE DISTRIBUTION


predicted_preference
1.0    50
Name: count, dtype: int64


ORIGINAL TOP-50 BOUNDARY (last 10)


,video_id,predicted_preference,original_rank
40,4234,1.0,41
41,4282,1.0,42
42,4289,1.0,43
43,4312,1.0,44
44,4680,1.0,45
45,4689,1.0,46
46,4715,1.0,47
47,4740,1.0,48
48,5453,1.0,49
49,5464,1.0,50


... (2 outputs left out)


Original-only videos


,video_id,predicted_preference,original_rank
0,133,1.0,1
5,517,1.0,6
13,1412,1.0,14
14,1445,1.0,15
15,1945,1.0,16
16,2312,1.0,17
17,2343,1.0,18
18,2383,1.0,19
19,2458,1.0,20
20,2472,1.0,21



Reconstructed-only videos


,user_id,video_id,predicted_preference,candidate_rank
24900,2568,10382,1.0,1
24906,2568,9758,1.0,7
24907,2568,9794,1.0,8
24909,2568,9615,1.0,10
24917,2568,7299,1.0,18
24918,2568,7300,1.0,19
24919,2568,9804,1.0,20
24923,2568,7119,1.0,24
24924,2568,7135,1.0,25
24925,2568,7158,1.0,26


In [24]:
# Cell 15: Exact-style Top-50 candidate reconstruction

TOP_CANDIDATES = 50
N_NEIGHBORS = 20

candidate_records = []

for user_id in user_item_matrix.index:

    # Select neighbors using the original baseline method
    similarities = user_similarity_df.loc[user_id].drop(user_id)
    nearest_users = similarities.nlargest(N_NEIGHBORS)

    # Get neighbours' preference vectors
    neighbor_ratings = user_item_matrix.loc[nearest_users.index]

    # Calculate weighted preference scores
    weighted_scores = np.dot(
        nearest_users.values,
        neighbor_ratings.values
    )

    similarity_sum = nearest_users.values.sum()

    predicted_scores = pd.Series(
        weighted_scores / similarity_sum,
        index=user_item_matrix.columns
    )

    # Exclude videos already watched by the user
    watched_videos = user_item_matrix.loc[user_id]
    watched_videos = watched_videos[watched_videos > 0].index

    predicted_scores = predicted_scores[
        ~predicted_scores.index.isin(watched_videos)
    ]

    # IMPORTANT: Match the original baseline tie-breaking
    ranked_candidates = (
        predicted_scores
        .rename("predicted_preference")
        .reset_index()
        .sort_values(
            ["predicted_preference", "video_id"],
            ascending=[False, True],
            kind="mergesort"
        )
        .head(TOP_CANDIDATES)
        .reset_index(drop=True)
    )

    ranked_candidates["user_id"] = user_id
    ranked_candidates["candidate_rank"] = np.arange(
        1, len(ranked_candidates) + 1
    )

    candidate_records.append(ranked_candidates)

baseline_candidates = pd.concat(
    candidate_records,
    ignore_index=True
)

baseline_candidates = baseline_candidates[
    ["user_id", "video_id", "predicted_preference", "candidate_rank"]
]

print("Candidate pool shape:", baseline_candidates.shape)
print("Users:", baseline_candidates["user_id"].nunique())
print("Candidates per user:")
display(baseline_candidates.groupby("user_id").size().describe())

print("Unique candidate videos:", baseline_candidates["video_id"].nunique())
print("Duplicate user-video pairs:",
      baseline_candidates.duplicated(["user_id", "video_id"]).sum())

display(
    baseline_candidates[
        baseline_candidates["user_id"] == 2568
    ].sort_values("candidate_rank").head(10)
)

Candidate pool shape: (70550, 4)
Users: 1411
Candidates per user:


count    1411.0
mean       50.0
std         0.0
min        50.0
25%        50.0
50%        50.0
75%        50.0
max        50.0
dtype: float64

Unique candidate videos: 2909
Duplicate user-video pairs: 0


,user_id,video_id,predicted_preference,candidate_rank
24900,2568,133,1.0,1
24901,2568,217,1.0,2
24902,2568,250,1.0,3
24903,2568,289,1.0,4
24904,2568,336,1.0,5
24905,2568,517,1.0,6
24906,2568,600,1.0,7
24907,2568,619,1.0,8
24908,2568,634,1.0,9
24909,2568,639,1.0,10


In [25]:
# Cell 16: Validate original baseline Top-10 against Top-50 candidate pool

baseline_top10 = baseline_recommendations[
    ["user_id", "video_id"]
].drop_duplicates()

candidate_top50 = baseline_candidates[
    ["user_id", "video_id"]
].drop_duplicates()

validation = baseline_top10.merge(
    candidate_top50,
    on=["user_id", "video_id"],
    how="left",
    indicator=True
)

validation["found_in_top50"] = validation["_merge"] == "both"

# Per-user validation
user_validation = validation.groupby("user_id")["found_in_top50"].agg(
    ["sum", "count"]
)

user_validation["all_10_found"] = user_validation["sum"] == 10

print("Total baseline recommendations:", len(baseline_top10))
print("Found in Top-50 candidates:", validation["found_in_top50"].sum())
print("Missing recommendations:", (~validation["found_in_top50"]).sum())
print("Users with all 10 baseline videos present:",
      user_validation["all_10_found"].sum())
print("Total users:", user_validation.shape[0])

print("\nUsers missing baseline recommendations:")
display(
    user_validation[user_validation["all_10_found"] == False]
)

print("\nMissing recommendation records:")
display(
    validation[validation["found_in_top50"] == False]
)

Total baseline recommendations: 14110
Found in Top-50 candidates: 14043
Missing recommendations: 67
Users with all 10 baseline videos present: 1398
Total users: 1411

Users missing baseline recommendations:


,sum,count,all_10_found
user_id,,,
368,6,10,False
1731,1,10,False
2367,8,10,False
2461,8,10,False
2568,0,10,False
3706,0,10,False
4653,1,10,False
4673,8,10,False
4882,1,10,False



Missing recommendation records:


,user_id,video_id,_merge,found_in_top50
646,368,765,left_only,False
647,368,3141,left_only,False
648,368,3183,left_only,False
649,368,2065,left_only,False
3281,1731,6764,left_only,False
...,...,...,...,...
13745,6991,5535,left_only,False
13746,6991,8922,left_only,False
13747,6991,833,left_only,False
13748,6991,694,left_only,False


In [26]:
# Cell 17: Diagnose missing baseline recommendations

missing_baseline = validation[
    validation["found_in_top50"] == False
][["user_id", "video_id"]].copy()

diagnostic_records = []

for user_id in missing_baseline["user_id"].unique():

    similarities = user_similarity_df.loc[user_id].drop(user_id)
    nearest_users = similarities.nlargest(N_NEIGHBORS)

    neighbor_ratings = user_item_matrix.loc[nearest_users.index]

    weighted_scores = np.dot(
        nearest_users.values,
        neighbor_ratings.values
    )

    similarity_sum = nearest_users.values.sum()

    predicted_scores = pd.Series(
        weighted_scores / similarity_sum,
        index=user_item_matrix.columns
    )

    watched_videos = user_item_matrix.loc[user_id]
    watched_videos = watched_videos[watched_videos > 0].index

    predicted_scores = predicted_scores[
        ~predicted_scores.index.isin(watched_videos)
    ]

    # Rank the complete candidate list
    full_ranked = (
        predicted_scores
        .rename("predicted_preference")
        .reset_index()
        .sort_values(
            ["predicted_preference", "video_id"],
            ascending=[False, True],
            kind="mergesort"
        )
        .reset_index(drop=True)
    )

    full_ranked["reconstructed_rank"] = np.arange(
        1, len(full_ranked) + 1
    )

    missing_for_user = missing_baseline[
        missing_baseline["user_id"] == user_id
    ]

    for video_id in missing_for_user["video_id"]:

        video_result = full_ranked[
            full_ranked["video_id"] == video_id
        ]

        if len(video_result) > 0:

            diagnostic_records.append({
                "user_id": user_id,
                "video_id": video_id,
                "reconstructed_score": video_result.iloc[0]["predicted_preference"],
                "reconstructed_rank": video_result.iloc[0]["reconstructed_rank"]
            })

missing_diagnostics = pd.DataFrame(diagnostic_records)

display(
    missing_diagnostics.sort_values(
        ["user_id", "reconstructed_rank"]
    )
)

print("\nRank distribution of missing baseline videos:")
display(missing_diagnostics["reconstructed_rank"].describe())

,user_id,video_id,reconstructed_score,reconstructed_rank
0,368,765,0.967640,62.0
1,368,3141,0.949570,80.0
2,368,3183,0.949143,82.0
3,368,2065,0.938490,95.0
4,1731,6764,0.966080,67.0
...,...,...,...,...
62,6991,5535,0.919452,55.0
63,6991,8922,0.907465,59.0
64,6991,833,0.905115,60.0
65,6991,694,0.903197,62.0



Rank distribution of missing baseline videos:


count     67.000000
mean     178.731343
std      150.513725
min       51.000000
25%       67.000000
50%       95.000000
75%      236.500000
max      543.000000
Name: reconstructed_rank, dtype: float64

In [27]:
# Cell 18: Compare neighbour-selection methods

diagnostic_user = 2568

similarities = user_similarity_df.loc[diagnostic_user].copy()

# Method 1: pandas nlargest (current reconstruction)
nlargest_neighbors = (
    similarities
    .drop(diagnostic_user)
    .nlargest(N_NEIGHBORS)
)

# Method 2: NumPy argsort-based selection
similarity_values = similarities.values
user_ids = similarities.index.values

user_position = np.where(user_ids == diagnostic_user)[0][0]

sorted_positions = np.argsort(similarity_values)

argsort_neighbor_positions = sorted_positions[
    -(N_NEIGHBORS + 1):-1
]

argsort_neighbors = pd.Series(
    similarity_values[argsort_neighbor_positions],
    index=user_ids[argsort_neighbor_positions]
)

print("Pandas nlargest neighbors:")
display(nlargest_neighbors)

print("\nNumPy argsort neighbors:")
display(argsort_neighbors)

nlargest_set = set(nlargest_neighbors.index)
argsort_set = set(argsort_neighbors.index)

print("\nNeighbor overlap:", len(nlargest_set & argsort_set), "/", N_NEIGHBORS)
print("Neighbors unique to nlargest:", nlargest_set - argsort_set)
print("Neighbors unique to argsort:", argsort_set - nlargest_set)


Pandas nlargest neighbors:


user_id
6927    0.680378
3223    0.676732
6099    0.676414
3764    0.676182
2118    0.675949
4239    0.675768
1704    0.675717
477     0.675068
3404    0.675008
1584    0.675003
1641    0.674825
5568    0.674744
5888    0.674624
683     0.674461
885     0.674198
2205    0.674028
2245    0.674008
7147    0.673938
3948    0.673818
4162    0.673811
Name: 2568, dtype: float64


NumPy argsort neighbors:


4162    0.673811
3948    0.673818
7147    0.673938
2245    0.674008
2205    0.674028
885     0.674198
683     0.674461
5888    0.674624
5568    0.674744
1641    0.674825
1584    0.675003
3404    0.675008
477     0.675068
1704    0.675717
4239    0.675768
2118    0.675949
3764    0.676182
6099    0.676414
3223    0.676732
6927    0.680378
dtype: float64


Neighbor overlap: 20 / 20
Neighbors unique to nlargest: set()
Neighbors unique to argsort: set()


In [28]:
# Cell 19: Inspect the original baseline recommendation implementation

import nbformat
import os

baseline_notebook_path = "../notebooks/02_baseline_recommender.ipynb"

baseline_nb = nbformat.read(
    baseline_notebook_path,
    as_version=4
)

for cell_number, cell in enumerate(baseline_nb.cells):

    if cell.cell_type == "code":

        source = cell.source

        if any(keyword in source for keyword in [
            "predict_user_preferences",
            "recommend_for_user",
            "baseline_recommendations"
        ]):

            print("=" * 80)
            print(f"CELL {cell_number}")
            print("=" * 80)
            print(source)
            print()

CELL 13
# Cell 14: Predict user preferences using nearest neighbours

def predict_user_preferences(user_id, k=K_NEIGHBORS):

    # Get similarities for the target user
    similarities = user_similarity_df.loc[user_id].drop(user_id)

    # Select top K similar users
    nearest_users = similarities.nlargest(k)

    # Retrieve their interaction data
    neighbour_ratings = interaction_matrix.loc[nearest_users.index]

    # Calculate similarity-weighted preference scores
    weighted_scores = np.dot(
        nearest_users.values,
        neighbour_ratings.values
    )

    # Calculate total similarity weight
    similarity_sum = np.abs(nearest_users.values).sum()

    predicted_scores = weighted_scores / similarity_sum

    return pd.Series(
        predicted_scores,
        index=interaction_matrix.columns
    )


print("Prediction function defined successfully!")

CELL 14
# Cell 15: Test predicted preferences for one user

test_user = interaction_matrix.index[0]

predicted_preferences 

In [29]:
# Cell 20: Inspect the original baseline matrix construction

for cell_number, cell in enumerate(baseline_nb.cells):

    if cell.cell_type == "code":

        source = cell.source

        if any(keyword in source for keyword in [
            "interaction_matrix =",
            "pivot_table",
            "pivot(",
            "preference_score",
            "watch_ratio"
        ]):

            print("=" * 80)
            print(f"CELL {cell_number}")
            print("=" * 80)
            print(source)
            print()

CELL 6
# Cell 7: Analyse watch ratio distribution

print("Watch ratio statistics:")
display(training_data["watch_ratio"].describe())

print("\nWatch ratios below zero:",
      (training_data["watch_ratio"] < 0).sum())

print("Watch ratios equal to zero:",
      (training_data["watch_ratio"] == 0).sum())

print("Watch ratios between zero and one:",
      ((training_data["watch_ratio"] > 0) &
       (training_data["watch_ratio"] <= 1)).sum())

print("Watch ratios above one:",
      (training_data["watch_ratio"] > 1).sum())

CELL 7
# Cell 8: Create capped watch ratio

training_data["preference_score"] = (
    training_data["watch_ratio"].clip(lower=0, upper=1)
)

print("Preference signal created successfully!")

print("\nPreference score statistics:")
display(training_data["preference_score"].describe())

print("\nValues above 1 after capping:",
      (training_data["preference_score"] > 1).sum())

print("Values below 0 after capping:",
      (training_data["preference_score"] < 0).sum())

In [31]:
# Cell 21: Reconstruct the original interaction matrix

import pandas as pd
import numpy as np

# Load training data
training_data = pd.read_csv(
    "../processed_data/training_interactions.csv"
)

# Recreate preference score exactly as in Notebook 02
training_data["preference_score"] = (
    training_data["watch_ratio"].clip(lower=0, upper=1)
)

# Recreate interaction matrix
interaction_matrix = training_data.pivot_table(
    index="user_id",
    columns="video_id",
    values="preference_score",
    aggfunc="mean",
    fill_value=0
)

print("Training data loaded:", training_data.shape)
print("Interaction matrix:", interaction_matrix.shape)

Training data loaded: (4387382, 10)
Interaction matrix: (1411, 3288)


In [33]:
# Cell 22: Compare interaction matrices

print("INTERACTION MATRIX VALIDATION")
print("=" * 60)

print("Original-style matrix dimensions:", interaction_matrix.shape)
print("Notebook 04 matrix dimensions:", user_item_matrix.shape)

print("\nUser index identical:",
      interaction_matrix.index.equals(user_item_matrix.index))

print("Video columns identical:",
      interaction_matrix.columns.equals(user_item_matrix.columns))

if interaction_matrix.shape == user_item_matrix.shape:

    matrix_difference = (
        interaction_matrix.values - user_item_matrix.values
    )

    print("\nExact matrix equality:",
          np.array_equal(
              interaction_matrix.values,
              user_item_matrix.values
          ))

    print("Maximum absolute difference:",
          np.abs(matrix_difference).max())

    print("Number of differing values:",
          np.count_nonzero(matrix_difference))

else:
    print("\nMatrix dimensions differ.")



INTERACTION MATRIX VALIDATION
Original-style matrix dimensions: (1411, 3288)
Notebook 04 matrix dimensions: (1411, 3288)

User index identical: True
Video columns identical: True

Exact matrix equality: True
Maximum absolute difference: 0.0
Number of differing values: 0


In [34]:
# Cell 23: Inspect original user similarity construction

for cell_number, cell in enumerate(baseline_nb.cells):

    if cell.cell_type == "code":

        source = cell.source

        if any(keyword in source for keyword in [
            "cosine_similarity",
            "user_similarity_matrix",
            "user_similarity_df"
        ]):

            print("=" * 80)
            print(f"CELL {cell_number}")
            print("=" * 80)
            print(source)
            print()

CELL 0
# Cell 1: Import required libraries

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import os
from sklearn.metrics.pairwise import cosine_similarity

# Display settings
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

print("Libraries imported successfully!")

CELL 9
# Cell 10: Calculate cosine similarity between users

user_similarity = cosine_similarity(interaction_matrix)

user_similarity_df = pd.DataFrame(
    user_similarity,
    index=interaction_matrix.index,
    columns=interaction_matrix.index
)

print("User similarity calculated successfully!")

print("\nSimilarity matrix dimensions:", user_similarity_df.shape)

print("\nSample similarity values:")
display(user_similarity_df.iloc[:5, :5])

CELL 11
# Cell 12: Identify most similar users

target_user = interaction_matrix.index[0]

similar_users = (
    user_similarity_df.loc[target_user]
    .drop(target_user)
    .sort_values(ascending=False)


In [35]:
# Cell 24: Validate cosine similarity matrices

original_similarity = cosine_similarity(interaction_matrix)

original_similarity_df = pd.DataFrame(
    original_similarity,
    index=interaction_matrix.index,
    columns=interaction_matrix.index
)

print("SIMILARITY MATRIX VALIDATION")
print("=" * 60)

print("Dimensions identical:",
      original_similarity_df.shape == user_similarity_df.shape)

print("User indices identical:",
      original_similarity_df.index.equals(user_similarity_df.index))

print("Columns identical:",
      original_similarity_df.columns.equals(user_similarity_df.columns))

similarity_difference = (
    original_similarity_df.values - user_similarity_df.values
)

print("\nExact matrix equality:",
      np.array_equal(
          original_similarity_df.values,
          user_similarity_df.values
      ))

print("Maximum absolute difference:",
      np.abs(similarity_difference).max())

print("Number of differing values:",
      np.count_nonzero(similarity_difference))

SIMILARITY MATRIX VALIDATION
Dimensions identical: True
User indices identical: True
Columns identical: True

Exact matrix equality: False
Maximum absolute difference: 5.662137425588298e-15
Number of differing values: 1791040


In [36]:
# Cell 25: Compare nearest neighbours for all affected users

affected_users = [
    368, 1731, 2367, 2461, 2568, 3706, 4653,
    4673, 4882, 4924, 6415, 6658, 6991
]

K_NEIGHBORS = 20

neighbor_comparison = []

for user_id in affected_users:

    original_neighbors = (
        original_similarity_df.loc[user_id]
        .drop(user_id)
        .nlargest(K_NEIGHBORS)
    )

    reconstructed_neighbors = (
        user_similarity_df.loc[user_id]
        .drop(user_id)
        .nlargest(K_NEIGHBORS)
    )

    original_set = set(original_neighbors.index)
    reconstructed_set = set(reconstructed_neighbors.index)

    overlap = len(original_set.intersection(reconstructed_set))

    neighbor_comparison.append({
        "user_id": user_id,
        "neighbor_overlap": overlap,
        "original_only": len(original_set - reconstructed_set),
        "reconstructed_only": len(reconstructed_set - original_set),
        "identical_neighbors": original_set == reconstructed_set
    })

neighbor_comparison_df = pd.DataFrame(neighbor_comparison)

display(neighbor_comparison_df)

print("\nUsers with identical neighbor sets:",
      neighbor_comparison_df["identical_neighbors"].sum(),
      "out of", len(affected_users))

,user_id,neighbor_overlap,original_only,reconstructed_only,identical_neighbors
0,368,20,0,0,True
1,1731,20,0,0,True
2,2367,20,0,0,True
3,2461,20,0,0,True
4,2568,20,0,0,True
5,3706,20,0,0,True
6,4653,20,0,0,True
7,4673,20,0,0,True
8,4882,20,0,0,True
9,4924,20,0,0,True



Users with identical neighbor sets: 13 out of 13


In [38]:
# Cell 26: Final candidate pool validation

baseline_check = baseline_recommendations.merge(
    baseline_candidates[["user_id", "video_id"]],
    on=["user_id", "video_id"],
    how="left",
    indicator=True
)

missing_count = (baseline_check["_merge"] == "left_only").sum()

print("CANDIDATE POOL VALIDATION")
print("=" * 60)

print("Original baseline recommendations:", len(baseline_recommendations))
print("Candidate pool size:", len(baseline_candidates))
print("Users:", baseline_candidates["user_id"].nunique())
print("Candidates per user:")
display(baseline_candidates.groupby("user_id").size().describe())

print("\nUnique candidate videos:", baseline_candidates["video_id"].nunique())

print("\nDuplicate user-video pairs:",
      baseline_candidates.duplicated(["user_id", "video_id"]).sum())

print("\nOriginal baseline recommendations missing:",
      missing_count)

print("Original baseline recommendations retained:",
      len(baseline_recommendations) - missing_count)

print("\nAll original recommendations retained:",
      missing_count == 0)

assert baseline_candidates.groupby("user_id").size().eq(50).all()
assert baseline_candidates.duplicated(["user_id", "video_id"]).sum() == 0
assert missing_count == 0

print("\nAll candidate pool validation checks passed!")

CANDIDATE POOL VALIDATION
Original baseline recommendations: 14110
Candidate pool size: 70550
Users: 1411
Candidates per user:


count    1411.0
mean       50.0
std         0.0
min        50.0
25%        50.0
50%        50.0
75%        50.0
max        50.0
dtype: float64


Unique candidate videos: 2909

Duplicate user-video pairs: 0

Original baseline recommendations missing: 67
Original baseline recommendations retained: 14043

All original recommendations retained: False


AssertionError: 

In [39]:
# Cell 27: Build a baseline-preserving candidate pool

candidate_pool_records = []

for user_id in baseline_recommendations["user_id"].unique():

    # Original frozen Top-10 recommendations
    original_top10 = (
        baseline_recommendations[
            baseline_recommendations["user_id"] == user_id
        ]
        .copy()
        .sort_values(
            ["predicted_preference", "video_id"],
            ascending=[False, True],
            kind="mergesort"
        )
    )

    # Reconstructed candidates for this user
    reconstructed_candidates = (
        baseline_candidates[
            baseline_candidates["user_id"] == user_id
        ]
        .copy()
        .sort_values(
            ["predicted_preference", "video_id"],
            ascending=[False, True],
            kind="mergesort"
        )
    )

    # Protect original baseline recommendations
    protected_video_ids = set(original_top10["video_id"])

    # Supplemental candidates must not duplicate protected videos
    supplemental_candidates = reconstructed_candidates[
        ~reconstructed_candidates["video_id"].isin(protected_video_ids)
    ]

    # Select 40 additional candidates
    supplemental_top40 = supplemental_candidates.head(40)

    # Combine protected baseline Top-10 and supplemental Top-40
    combined_candidates = pd.concat(
        [
            original_top10[
                ["user_id", "video_id", "predicted_preference"]
            ],
            supplemental_top40[
                ["user_id", "video_id", "predicted_preference"]
            ]
        ],
        ignore_index=True
    )

    # Rank the combined candidate pool
    combined_candidates = (
        combined_candidates
        .sort_values(
            ["predicted_preference", "video_id"],
            ascending=[False, True],
            kind="mergesort"
        )
        .reset_index(drop=True)
    )

    combined_candidates["candidate_rank"] = (
        np.arange(1, len(combined_candidates) + 1)
    )

    candidate_pool_records.append(combined_candidates)


# Combine all users
spotlight_candidate_pool = pd.concat(
    candidate_pool_records,
    ignore_index=True
)

print("Baseline-preserving candidate pool created!")
print("Shape:", spotlight_candidate_pool.shape)
print("Users:", spotlight_candidate_pool["user_id"].nunique())
print("Unique videos:", spotlight_candidate_pool["video_id"].nunique())

Baseline-preserving candidate pool created!
Shape: (70550, 4)
Users: 1411
Unique videos: 2909


In [40]:
# Cell 28: Validate baseline-preserving candidate pool

print("FINAL CANDIDATE POOL VALIDATION")
print("=" * 60)

# Basic dimensions
print("Shape:", spotlight_candidate_pool.shape)
print("Unique users:", spotlight_candidate_pool["user_id"].nunique())
print("Unique videos:", spotlight_candidate_pool["video_id"].nunique())

# Candidates per user
candidate_counts = spotlight_candidate_pool.groupby("user_id").size()

print("\nCandidates per user:")
display(candidate_counts.describe())

# Duplicate user-video combinations
duplicate_pairs = spotlight_candidate_pool.duplicated(
    ["user_id", "video_id"]
).sum()

print("Duplicate user-video pairs:", duplicate_pairs)

# Verify original baseline Top-10 retention
baseline_validation = baseline_recommendations.merge(
    spotlight_candidate_pool[["user_id", "video_id"]],
    on=["user_id", "video_id"],
    how="left",
    indicator=True
)

missing_baseline = (
    baseline_validation["_merge"] == "left_only"
).sum()

print("\nOriginal baseline recommendations:", len(baseline_recommendations))
print("Missing baseline recommendations:", missing_baseline)
print("Baseline recommendations retained:",
      len(baseline_recommendations) - missing_baseline)

# Assertions
assert candidate_counts.eq(50).all(), "Not all users have exactly 50 candidates!"
assert duplicate_pairs == 0, "Duplicate user-video pairs detected!"
assert missing_baseline == 0, "Some original baseline recommendations are missing!"

print("\n✅ ALL VALIDATION CHECKS PASSED!")

FINAL CANDIDATE POOL VALIDATION
Shape: (70550, 4)
Unique users: 1411
Unique videos: 2909

Candidates per user:


count    1411.0
mean       50.0
std         0.0
min        50.0
25%        50.0
50%        50.0
75%        50.0
max        50.0
dtype: float64

Duplicate user-video pairs: 0

Original baseline recommendations: 14110
Missing baseline recommendations: 0
Baseline recommendations retained: 14110

✅ ALL VALIDATION CHECKS PASSED!


In [41]:
# Cell 29: Save the validated Spotlight candidate pool

candidate_pool_path = os.path.join(
    RESULTS_DIR,
    "spotlight_candidate_pool.csv"
)

spotlight_candidate_pool.to_csv(
    candidate_pool_path,
    index=False
)

print("Candidate pool saved successfully!")
print("Path:", candidate_pool_path)
print("Shape:", spotlight_candidate_pool.shape)

Candidate pool saved successfully!
Path: ../results\spotlight_candidate_pool.csv
Shape: (70550, 4)


In [42]:
# Cell 30: Independent read-back verification

saved_candidate_pool = pd.read_csv(
    os.path.join(RESULTS_DIR, "spotlight_candidate_pool.csv")
)

print("SAVED CANDIDATE POOL VERIFICATION")
print("=" * 60)

print("File successfully loaded:", True)
print("Shape:", saved_candidate_pool.shape)
print("Users:", saved_candidate_pool["user_id"].nunique())
print("Unique videos:", saved_candidate_pool["video_id"].nunique())

print("\nCandidates per user:")
display(saved_candidate_pool.groupby("user_id").size().describe())

print("\nDuplicate user-video pairs:",
      saved_candidate_pool.duplicated(["user_id", "video_id"]).sum())

# Verify original baseline Top-10 preservation
saved_validation = baseline_recommendations.merge(
    saved_candidate_pool[["user_id", "video_id"]],
    on=["user_id", "video_id"],
    how="left",
    indicator=True
)

saved_missing = (
    saved_validation["_merge"] == "left_only"
).sum()

print("\nOriginal baseline recommendations missing:", saved_missing)

assert saved_candidate_pool.shape == (70550, 4)
assert saved_candidate_pool["user_id"].nunique() == 1411
assert saved_candidate_pool.groupby("user_id").size().eq(50).all()
assert saved_candidate_pool.duplicated(["user_id", "video_id"]).sum() == 0
assert saved_missing == 0

print("\n✅ SAVED CANDIDATE POOL VERIFIED SUCCESSFULLY!")

SAVED CANDIDATE POOL VERIFICATION
File successfully loaded: True
Shape: (70550, 4)
Users: 1411
Unique videos: 2909

Candidates per user:


count    1411.0
mean       50.0
std         0.0
min        50.0
25%        50.0
50%        50.0
75%        50.0
max        50.0
dtype: float64


Duplicate user-video pairs: 0

Original baseline recommendations missing: 0

✅ SAVED CANDIDATE POOL VERIFIED SUCCESSFULLY!


In [43]:
# Cell 31: Prepare candidate-level metadata

# Merge creator information
spotlight_features = saved_candidate_pool.merge(
    video_creator_groups[
        ["video_id", "author_id", "creator_group"]
    ],
    on="video_id",
    how="left"
)

# Merge category information
spotlight_features = spotlight_features.merge(
    video_metadata[["video_id", "category_ids"]],
    on="video_id",
    how="left"
)

print("SPOTLIGHT FEATURE PREPARATION")
print("=" * 60)

print("Shape:", spotlight_features.shape)
print("Unique users:", spotlightlight_features["user_id"].nunique()
                 if "spotlight_features" in globals() else spotlight_features["user_id"].nunique())

print("\nMissing creator IDs:",
      spotlight_features["author_id"].isna().sum())

print("Missing creator groups:",
      spotlight_features["creator_group"].isna().sum())

print("Missing category metadata:",
      spotlight_features["category_ids"].isna().sum())

print("\nCreator group distribution:")
display(spotlight_features["creator_group"].value_counts(dropna=False))

print("\nCandidate videos by creator group:")
display(
    spotlight_features.groupby("creator_group", dropna=False)
    ["video_id"].nunique()
)

print("\nSample candidate features:")
display(spotlight_features.head(10))

NameError: name 'video_metadata' is not defined

In [44]:
# Load video metadata from the processed dataset

video_metadata = pd.read_csv(
    os.path.join(PROCESSED_DIR, "video_metadata.csv")
)

print("Video metadata loaded successfully!")
print("Shape:", video_metadata.shape)
print("Columns:", video_metadata.columns.tolist())

Video metadata loaded successfully!
Shape: (3327, 12)
Columns: ['video_id', 'category_ids', 'category_count', 'manual_cover_text', 'caption', 'topic_tag', 'first_level_category_id', 'first_level_category_name', 'second_level_category_id', 'second_level_category_name', 'third_level_category_id', 'third_level_category_name']


In [45]:
# Cell 31: Prepare candidate-level metadata

# Merge creator information
spotlight_features = saved_candidate_pool.merge(
    video_creator_groups[
        ["video_id", "author_id", "creator_group"]
    ],
    on="video_id",
    how="left"
)

# Merge video category information
spotlight_features = spotlight_features.merge(
    video_metadata[["video_id", "category_ids"]],
    on="video_id",
    how="left"
)

# Basic validation
print("SPOTLIGHT FEATURE PREPARATION")
print("=" * 60)

print("Shape:", spotlight_features.shape)
print("Unique users:", spotlight_features["user_id"].nunique())
print("Unique videos:", spotlightlight_features["video_id"].nunique()
      if False else spotlight_features["video_id"].nunique())

print("\nMissing creator IDs:",
      spotlight_features["author_id"].isna().sum())

print("Missing creator groups:",
      spotlight_features["creator_group"].isna().sum())

print("Missing category metadata:",
      spotlight_features["category_ids"].isna().sum())

print("\nCreator group distribution:")
display(spotlight_features["creator_group"].value_counts(dropna=False))

print("\nCandidate videos by creator group:")
display(
    spotlight_features.groupby("creator_group", dropna=False)["video_id"]
    .nunique()
)

print("\nSample candidate features:")
display(spotlight_features.head(10))

SPOTLIGHT FEATURE PREPARATION
Shape: (70550, 7)
Unique users: 1411
Unique videos: 2909

Missing creator IDs: 0
Missing creator groups: 0
Missing category metadata: 0

Creator group distribution:


creator_group
Established    58126
Emerging       12424
Name: count, dtype: int64


Candidate videos by creator group:


creator_group
Emerging        455
Established    2454
Name: video_id, dtype: int64


Sample candidate features:


,user_id,video_id,predicted_preference,candidate_rank,author_id,creator_group,category_ids
0,14,3992,0.997153,1,7828,Established,[8]
1,14,7263,0.995710,2,2579,Established,"[1, 26]"
2,14,835,0.990655,3,1481,Established,"[28, 25, 9]"
3,14,6177,0.990421,4,1704,Established,[11]
4,14,4026,0.974648,5,7935,Established,[26]
5,14,6086,0.966525,6,5096,Established,[28]
6,14,3141,0.962310,7,3076,Established,[16]
7,14,3107,0.961977,8,7318,Established,[28]
8,14,152,0.950126,9,2461,Emerging,[7]
9,14,2065,0.949943,10,2595,Emerging,"[9, 17]"


In [47]:
# Diagnose the category representation in interest_profiles

print("INTEREST PROFILE STRUCTURE")
print("=" * 60)

print("Shape:", interest_profiles.shape)
print("\nDtypes:")
print(interest_profiles.dtypes)

print("\nFirst 10 rows:")
display(interest_profiles.head(10))

print("\nCategory value examples:")
display(interest_profiles["category_ids"].head(10).tolist())

print("\nCategory value types:")
display(
    interest_profiles["category_ids"]
    .apply(type)
    .value_counts()
)

INTEREST PROFILE STRUCTURE
Shape: (40918, 4)

Dtypes:
user_id                       int64
category_ids                 object
mean_category_preference    float64
interest_weight             float64
dtype: object

First 10 rows:


,user_id,category_ids,mean_category_preference,interest_weight
0,14,[],0.701592,0.035495
1,14,[],0.763993,0.038652
2,14,[],0.620291,0.031382
3,14,[],0.553769,0.028016
4,14,[],0.735436,0.037207
5,14,[],0.767601,0.038835
6,14,[],0.656392,0.033208
7,14,[],0.814710,0.041218
8,14,[],0.757394,0.038318
9,14,[],0.725393,0.036699



Category value examples:


[[], [], [], [], [], [], [], [], [], []]


Category value types:


category_ids
<class 'list'>    40918
Name: count, dtype: int64

In [48]:
# Restore interest profiles from the original saved file

interest_profiles = pd.read_csv(
    os.path.join(PROCESSED_DIR, "interest_profiles.csv")
)

print("INTEREST PROFILES RESTORED")
print("=" * 60)

print("Shape:", interest_profiles.shape)
print("\nData types:")
print(interest_profiles.dtypes)

print("\nFirst 10 rows:")
display(interest_profiles.head(10))

print("\nCategory ID values:")
display(
    interest_profiles["category_ids"]
    .head(10)
    .tolist()
)

print("\nUnique categories:", interest_profiles["category_ids"].nunique())
print("Unique users:", interest_profiles["user_id"].nunique())

assert interest_profiles["category_ids"].notna().all()

print("\n✅ ORIGINAL INTEREST PROFILES RESTORED!")

INTEREST PROFILES RESTORED
Shape: (40918, 4)

Data types:
user_id                       int64
category_ids                  int64
mean_category_preference    float64
interest_weight             float64
dtype: object

First 10 rows:


,user_id,category_ids,mean_category_preference,interest_weight
0,14,0,0.701592,0.035495
1,14,1,0.763993,0.038652
2,14,2,0.620291,0.031382
3,14,3,0.553769,0.028016
4,14,4,0.735436,0.037207
5,14,5,0.767601,0.038835
6,14,6,0.656392,0.033208
7,14,7,0.814710,0.041218
8,14,8,0.757394,0.038318
9,14,9,0.725393,0.036699



Category ID values:


[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]


Unique categories: 29
Unique users: 1411

✅ ORIGINAL INTEREST PROFILES RESTORED!


In [49]:
# Cell 32: Build user-interest matching layer (corrected)

import ast
import numpy as np
import pandas as pd

# --------------------------------------------------
# 1. Ensure video categories are properly parsed
# --------------------------------------------------

def parse_category_list(value):
    if isinstance(value, list):
        return value

    if pd.isna(value):
        return []

    try:
        parsed = ast.literal_eval(value)
        return parsed if isinstance(parsed, list) else []
    except (ValueError, SyntaxError):
        return []


spotlight_features["category_ids"] = (
    spotlight_features["category_ids"].apply(parse_category_list)
)


# --------------------------------------------------
# 2. Prepare user-category interest weights
#    (category_ids are already individual integers)
# --------------------------------------------------

user_interest_weights = interest_profiles[
    ["user_id", "category_ids", "interest_weight"]
].copy()

user_interest_weights = user_interest_weights.rename(
    columns={"category_ids": "category_id"}
)

user_interest_weights["category_id"] = (
    pd.to_numeric(user_interest_weights["category_id"])
    .astype(int)
)


# --------------------------------------------------
# 3. Expand candidate videos into category-level rows
# --------------------------------------------------

candidate_category_data = spotlight_features[
    ["user_id", "video_id", "category_ids"]
].explode("category_ids")

candidate_category_data = candidate_category_data.rename(
    columns={"category_ids": "category_id"}
)

candidate_category_data = candidate_category_data.dropna(
    subset=["category_id"]
)

candidate_category_data["category_id"] = (
    pd.to_numeric(candidate_category_data["category_id"])
    .astype(int)
)


# --------------------------------------------------
# 4. Match videos with the user's interest profile
# --------------------------------------------------

candidate_interest_matches = candidate_category_data.merge(
    user_interest_weights,
    on=["user_id", "category_id"],
    how="left",
    validate="many_to_one"
)


# --------------------------------------------------
# 5. Calculate video-level interest match scores
# --------------------------------------------------

video_interest_scores = (
    candidate_interest_matches
    .groupby(["user_id", "video_id"])["interest_weight"]
    .mean()
    .reset_index()
    .rename(
        columns={
            "interest_weight": "interest_match_score"
        }
    )
)


# --------------------------------------------------
# 6. Merge scores back into the candidate dataset
# --------------------------------------------------

# Remove score column if this cell is rerun
spotlight_features = spotlight_features.drop(
    columns=["interest_match_score"],
    errors="ignore"
)

spotlight_features = spotlight_features.merge(
    video_interest_scores,
    on=["user_id", "video_id"],
    how="left",
    validate="one_to_one"
)


# --------------------------------------------------
# 7. Validate results
# --------------------------------------------------

print("USER INTEREST MATCHING")
print("=" * 60)

print("Dataset shape:", spotlight_features.shape)

print("Missing interest scores:",
      spotlight_features["interest_match_score"].isna().sum())

print("\nInterest-match score statistics:")
display(
    spotlight_features["interest_match_score"].describe()
)

print("\nScore distribution:")
display(
    spotlight_features["interest_match_score"]
    .value_counts(bins=10, sort=False)
)

print("\nSample candidate scores:")
display(
    spotlight_features[
        [
            "user_id",
            "video_id",
            "creator_group",
            "category_ids",
            "predicted_preference",
            "interest_match_score"
        ]
    ].head(15)
)


# --------------------------------------------------
# 8. Assertions
# --------------------------------------------------

assert spotlight_features["interest_match_score"].notna().all()

assert spotlight_features["interest_match_score"].between(
    0, 1
).all()

assert spotlight_features.shape[0] == 70550

print("\n✅ INTEREST MATCHING COMPLETED SUCCESSFULLY!")

USER INTEREST MATCHING
Dataset shape: (70550, 8)
Missing interest scores: 0

Interest-match score statistics:


count    70550.000000
mean         0.038538
std          0.003628
min          0.000000
25%          0.037230
50%          0.039128
75%          0.040975
max          0.050603
Name: interest_match_score, dtype: float64


Score distribution:


(-0.0010506, 0.00506]        1
(0.00506, 0.0101]            3
(0.0101, 0.0152]             2
(0.0152, 0.0202]            22
(0.0202, 0.0253]           734
(0.0253, 0.0304]          1871
(0.0304, 0.0354]          6739
(0.0354, 0.0405]         39180
(0.0405, 0.0455]         21963
(0.0455, 0.0506]            35
Name: count, dtype: int64


Sample candidate scores:


,user_id,video_id,creator_group,category_ids,predicted_preference,interest_match_score
0,14,3992,Established,[8],0.997153,0.038318
1,14,7263,Established,"[1, 26]",0.995710,0.037221
2,14,835,Established,"[28, 25, 9]",0.990655,0.038353
3,14,6177,Established,[11],0.990421,0.037879
4,14,4026,Established,[26],0.974648,0.035791
5,14,6086,Established,[28],0.966525,0.040774
6,14,3141,Established,[16],0.962310,0.026535
7,14,3107,Established,[28],0.961977,0.040774
8,14,152,Emerging,[7],0.950126,0.041218
9,14,2065,Emerging,"[9, 17]",0.949943,0.038018



✅ INTEREST MATCHING COMPLETED SUCCESSFULLY!


In [50]:
# Cell 33: Diagnose the interest signal

print("INTEREST SIGNAL DIAGNOSTICS")
print("=" * 60)

# --------------------------------------------------
# 1. Interest-score variation within each user
# --------------------------------------------------

user_interest_variation = (
    spotlight_features
    .groupby("user_id")["interest_match_score"]
    .agg(
        interest_mean="mean",
        interest_std="std",
        interest_min="min",
        interest_max="max",
        unique_scores="nunique"
    )
)

user_interest_variation["interest_range"] = (
    user_interest_variation["interest_max"]
    - user_interest_variation["interest_min"]
)

print("\nWithin-user interest variation:")
display(user_interest_variation.describe())


# --------------------------------------------------
# 2. Users with the least variation in interest scores
# --------------------------------------------------

print("\nUsers with the lowest interest-score variation:")
display(
    user_interest_variation.sort_values("interest_std")
    .head(10)
)


# --------------------------------------------------
# 3. Relationship between CF relevance and interest
# --------------------------------------------------

overall_correlation = spotlight_features[
    ["predicted_preference", "interest_match_score"]
].corr().iloc[0, 1]

print("\nCorrelation between collaborative preference and interest:")
print(round(overall_correlation, 4))


# --------------------------------------------------
# 4. Compare interest scores by creator group
# --------------------------------------------------

print("\nInterest scores by creator group:")

display(
    spotlight_features.groupby("creator_group")[
        "interest_match_score"
    ].agg(
        ["count", "mean", "std", "min", "median", "max"]
    )
)


# --------------------------------------------------
# 5. Compare relevance scores by creator group
# --------------------------------------------------

print("\nCollaborative preference scores by creator group:")

display(
    spotlight_features.groupby("creator_group")[
        "predicted_preference"
    ].agg(
        ["count", "mean", "std", "min", "median", "max"]
    )
)


# --------------------------------------------------
# 6. Check users with nearly constant interest scores
# --------------------------------------------------

low_variation_users = (
    user_interest_variation["interest_std"] < 0.001
).sum()

print("\nUsers with interest-score std below 0.001:",
      low_variation_users)

print("Total users:", user_interest_variation.shape[0])

print("\n✅ INTEREST SIGNAL DIAGNOSTICS COMPLETED!")

INTEREST SIGNAL DIAGNOSTICS

Within-user interest variation:


,interest_mean,interest_std,interest_min,interest_max,unique_scores,interest_range
count,1411.000000,1411.000000,1411.000000,1411.000000,1411.000000,1411.000000
mean,0.038538,0.003397,0.027320,0.042189,24.124734,0.014869
std,0.001036,0.000894,0.003676,0.001304,3.516329,0.003985
min,0.034569,0.001269,0.000000,0.038002,13.000000,0.004561
25%,0.037867,0.002741,0.025103,0.041364,22.000000,0.011976
50%,0.038576,0.003291,0.026428,0.042146,24.000000,0.015423
75%,0.039226,0.003972,0.030090,0.042935,27.000000,0.017308
max,0.042083,0.007605,0.036361,0.050603,35.000000,0.046441



Users with the lowest interest-score variation:


,interest_mean,interest_std,interest_min,interest_max,unique_scores,interest_range
user_id,,,,,,
412,0.038655,0.001269,0.035451,0.040012,21,0.004561
6603,0.038208,0.001351,0.035283,0.041181,23,0.005898
4914,0.037275,0.001369,0.033404,0.039630,23,0.006226
407,0.037240,0.001376,0.032994,0.040342,29,0.007349
2372,0.037585,0.001378,0.032521,0.040724,26,0.008203
5552,0.039336,0.001543,0.033675,0.040163,14,0.006488
1853,0.040235,0.001550,0.035209,0.042412,28,0.007203
2653,0.039367,0.001630,0.033198,0.040674,14,0.007476
3987,0.039315,0.001635,0.035865,0.041523,22,0.005658



Correlation between collaborative preference and interest:
0.1585

Interest scores by creator group:


,count,mean,std,min,median,max
creator_group,,,,,,
Emerging,12424,0.038164,0.003790,0.00000,0.038877,0.050603
Established,58126,0.038618,0.003588,0.00559,0.039204,0.049373



Collaborative preference scores by creator group:


,count,mean,std,min,median,max
creator_group,,,,,,
Emerging,12424,0.932128,0.086532,0.149491,0.959743,1.0
Established,58126,0.872739,0.122510,0.000000,0.904138,1.0



Users with interest-score std below 0.001: 0
Total users: 1411

✅ INTEREST SIGNAL DIAGNOSTICS COMPLETED!


In [51]:
# Cell 34: Within-user rank normalization

# Higher percentile = better candidate for that user

spotlight_features["cf_percentile"] = (
    spotlight_features
    .groupby("user_id")["predicted_preference"]
    .rank(method="average", pct=True)
)

spotlight_features["interest_percentile"] = (
    spotlight_features
    .groupby("user_id")["interest_match_score"]
    .rank(method="average", pct=True)
)


# --------------------------------------------------
# Diagnostics
# --------------------------------------------------

print("WITHIN-USER RANK NORMALIZATION")
print("=" * 60)

print("\nNormalized score statistics:")
display(
    spotlight_features[
        ["cf_percentile", "interest_percentile"]
    ].describe()
)

print("\nCorrelation between normalized signals:")
print(
    spotlight_features[
        ["cf_percentile", "interest_percentile"]
    ].corr()
)

print("\nAverage normalized scores by creator group:")
display(
    spotlight_features.groupby("creator_group")[
        ["cf_percentile", "interest_percentile"]
    ].mean()
)

print("\nSample normalized candidates:")
display(
    spotlight_features[
        [
            "user_id",
            "video_id",
            "creator_group",
            "predicted_preference",
            "interest_match_score",
            "cf_percentile",
            "interest_percentile"
        ]
    ].head(15)
)

assert spotlight_features["cf_percentile"].between(0, 1).all()
assert spotlight_features["interest_percentile"].between(0, 1).all()

print("\n✅ WITHIN-USER NORMALIZATION COMPLETED!")

WITHIN-USER RANK NORMALIZATION

Normalized score statistics:


,cf_percentile,interest_percentile
count,70550.000000,70550.000000
mean,0.510000,0.510000
std,0.288355,0.285141
min,0.020000,0.020000
25%,0.260000,0.260000
50%,0.520000,0.510000
75%,0.760000,0.780000
max,1.000000,1.000000



Correlation between normalized signals:
                     cf_percentile  interest_percentile
cf_percentile             1.000000             0.132377
interest_percentile       0.132377             1.000000

Average normalized scores by creator group:


,cf_percentile,interest_percentile
creator_group,,
Emerging,0.605385,0.470562
Established,0.489612,0.518430



Sample normalized candidates:


,user_id,video_id,creator_group,predicted_preference,interest_match_score,cf_percentile,interest_percentile
0,14,3992,Established,0.997153,0.038318,1.00,0.63
1,14,7263,Established,0.995710,0.037221,0.98,0.44
2,14,835,Established,0.990655,0.038353,0.96,0.66
3,14,6177,Established,0.990421,0.037879,0.94,0.56
4,14,4026,Established,0.974648,0.035791,0.92,0.18
5,14,6086,Established,0.966525,0.040774,0.90,0.84
6,14,3141,Established,0.962310,0.026535,0.88,0.08
7,14,3107,Established,0.961977,0.040774,0.86,0.84
8,14,152,Emerging,0.950126,0.041218,0.84,0.96
9,14,2065,Emerging,0.949943,0.038018,0.82,0.58



✅ WITHIN-USER NORMALIZATION COMPLETED!


In [52]:
# Cell 35: Interest-constraint threshold sensitivity analysis

interest_thresholds = [0.10, 0.20, 0.30, 0.40]

threshold_results = []
user_threshold_results = []

for threshold in interest_thresholds:

    eligible = spotlight_features[
        spotlight_features["interest_percentile"] >= threshold
    ].copy()

    # Overall candidate eligibility
    threshold_results.append({
        "interest_threshold": threshold,
        "eligible_candidates": len(eligible),
        "eligible_users": eligible["user_id"].nunique(),
        "emerging_candidates": (
            eligible["creator_group"] == "Emerging"
        ).sum(),
        "established_candidates": (
            eligible["creator_group"] == "Established"
        ).sum(),
        "emerging_share": (
            (eligible["creator_group"] == "Emerging").mean()
            if len(eligible) > 0 else 0
        )
    })

    # Candidate availability per user
    eligible_per_user = (
        eligible.groupby("user_id")
        .size()
        .reindex(
            spotlight_features["user_id"].unique(),
            fill_value=0
        )
    )

    user_threshold_results.append({
        "interest_threshold": threshold,
        "mean_eligible_candidates": eligible_per_user.mean(),
        "median_eligible_candidates": eligible_per_user.median(),
        "min_eligible_candidates": eligible_per_user.min(),
        "users_with_at_least_10": (
            eligible_per_user >= 10
        ).sum(),
        "users_with_at_least_20": (
            eligible_per_user >= 20
        ).sum(),
        "users_with_at_least_30": (
            eligible_per_user >= 30
        ).sum()
    })


threshold_summary = pd.DataFrame(threshold_results)

user_threshold_summary = pd.DataFrame(user_threshold_results)


print("INTEREST THRESHOLD SENSITIVITY")
print("=" * 60)

print("\nCandidate availability:")
display(threshold_summary)

print("\nEligible candidates per user:")
display(user_threshold_summary)


print("\n✅ THRESHOLD SENSITIVITY ANALYSIS COMPLETED!")

INTEREST THRESHOLD SENSITIVITY

Candidate availability:


,interest_threshold,eligible_candidates,eligible_users,emerging_candidates,established_candidates,emerging_share
0,0.1,64762,1411,11312,53450,0.174670
1,0.2,57580,1411,10210,47370,0.177319
2,0.3,50496,1411,8842,41654,0.175103
3,0.4,43554,1411,7293,36261,0.167447



Eligible candidates per user:


,interest_threshold,mean_eligible_candidates,median_eligible_candidates,min_eligible_candidates,users_with_at_least_10,users_with_at_least_20,users_with_at_least_30
0,0.1,45.897945,46.0,42,1411,1411,1411
1,0.2,40.807938,41.0,36,1411,1411,1411
2,0.3,35.787385,36.0,28,1411,1411,1410
3,0.4,30.867470,31.0,25,1411,1411,1326



✅ THRESHOLD SENSITIVITY ANALYSIS COMPLETED!


In [53]:
# Cell 36: Apply provisional interest eligibility constraint

# Provisional threshold selected from sensitivity analysis
INTEREST_THRESHOLD = 0.30

# Mark candidates as eligible/ineligible for interest-constrained exploration
spotlight_features["interest_eligible"] = (
    spotlight_features["interest_percentile"]
    >= INTEREST_THRESHOLD
)

# --------------------------------------------------
# 1. Overall eligibility
# --------------------------------------------------

eligible_candidates = spotlight_features[
    spotlight_features["interest_eligible"]
].copy()

print("SPOTLIGHT INTEREST CONSTRAINT")
print("=" * 60)

print("Interest threshold:", INTEREST_THRESHOLD)

print("Total candidates:", len(spotlight_features))

print("Eligible candidates:", len(eligible_candidates))

print("Eligibility rate:",
      round(
          len(eligible_candidates) / len(spotlight_features) * 100,
          2
      ), "%")


# --------------------------------------------------
# 2. Candidates available per user
# --------------------------------------------------

eligible_per_user = (
    eligible_candidates
    .groupby("user_id")
    .size()
    .reindex(
        spotlight_features["user_id"].unique(),
        fill_value=0
    )
)

print("\nEligible candidates per user:")
display(eligible_per_user.describe())


# --------------------------------------------------
# 3. Eligibility by creator group
# --------------------------------------------------

print("\nEligibility by creator group:")

eligibility_by_group = (
    spotlight_features.groupby("creator_group")
    .agg(
        total_candidates=("video_id", "count"),
        eligible_candidates=("interest_eligible", "sum")
    )
)

eligibility_by_group["eligibility_rate"] = (
    eligibility_by_group["eligible_candidates"]
    / eligibility_by_group["total_candidates"]
)

display(eligibility_by_group)


# --------------------------------------------------
# 4. Emerging representation after interest constraint
# --------------------------------------------------

emerging_share_before = (
    spotlight_features["creator_group"]
    .eq("Emerging")
    .mean()
)

emerging_share_after = (
    eligible_candidates["creator_group"]
    .eq("Emerging")
    .mean()
)

print("\nEmerging candidate share:")
print("Before constraint:",
      round(emerging_share_before * 100, 2), "%")

print("After constraint:",
      round(emerging_share_after * 100, 2), "%")


# --------------------------------------------------
# 5. Validation
# --------------------------------------------------

assert eligible_per_user.min() >= 10

assert spotlight_features["interest_eligible"].dtype == bool

print("\n✅ INTEREST CONSTRAINT APPLIED SUCCESSFULLY!")

SPOTLIGHT INTEREST CONSTRAINT
Interest threshold: 0.3
Total candidates: 70550
Eligible candidates: 50496
Eligibility rate: 71.57 %

Eligible candidates per user:


count    1411.000000
mean       35.787385
std         1.039031
min        28.000000
25%        35.000000
50%        36.000000
75%        36.000000
max        41.000000
dtype: float64


Eligibility by creator group:


,total_candidates,eligible_candidates,eligibility_rate
creator_group,,,
Emerging,12424,8842,0.711687
Established,58126,41654,0.716616



Emerging candidate share:
Before constraint: 17.61 %
After constraint: 17.51 %

✅ INTEREST CONSTRAINT APPLIED SUCCESSFULLY!


In [54]:
# Cell 37: Creator opportunity vs. baseline exposure

# --------------------------------------------------
# 1. Creator classification lookup
# --------------------------------------------------

creator_lookup = (
    video_creator_groups[
        ["author_id", "creator_group"]
    ]
    .drop_duplicates()
)


# --------------------------------------------------
# 2. Eligible candidate opportunity
#    Number of distinct users for whom a creator
#    has at least one interest-eligible candidate
# --------------------------------------------------

eligible_creator_opportunity = (
    spotlight_features[
        spotlight_features["interest_eligible"]
    ][["user_id", "author_id", "creator_group"]]
    .drop_duplicates(["user_id", "author_id"])
    .groupby(["author_id", "creator_group"])
    .agg(
        eligible_user_opportunities=("user_id", "nunique")
    )
    .reset_index()
)


# --------------------------------------------------
# 3. Baseline creator exposure
#    Number of Top-10 recommendation slots received
# --------------------------------------------------

baseline_with_creator = baseline_recommendations.merge(
    video_creator_groups[
        ["video_id", "author_id", "creator_group"]
    ],
    on="video_id",
    how="left"
)

baseline_creator_exposure = (
    baseline_with_creator
    .groupby(["author_id", "creator_group"])
    .agg(
        baseline_exposure_slots=("video_id", "count"),
        baseline_exposed_users=("user_id", "nunique"),
        baseline_unique_videos=("video_id", "nunique")
    )
    .reset_index()
)


# --------------------------------------------------
# 4. Combine opportunity and exposure
# --------------------------------------------------

creator_opportunity_exposure = creator_lookup.merge(
    eligible_creator_opportunity,
    on=["author_id", "creator_group"],
    how="left"
).merge(
    baseline_creator_exposure,
    on=["author_id", "creator_group"],
    how="left"
)


# --------------------------------------------------
# 5. Fill creators with zero baseline exposure
# --------------------------------------------------

exposure_columns = [
    "baseline_exposure_slots",
    "baseline_exposed_users",
    "baseline_unique_videos"
]

creator_opportunity_exposure[exposure_columns] = (
    creator_opportunity_exposure[exposure_columns]
    .fillna(0)
)


creator_opportunity_exposure[
    "eligible_user_opportunities"
] = (
    creator_opportunity_exposure[
        "eligible_user_opportunities"
    ].fillna(0)
)


# --------------------------------------------------
# 6. Exposure relative to opportunity
# --------------------------------------------------

creator_opportunity_exposure[
    "baseline_exposure_per_opportunity"
] = np.where(
    creator_opportunity_exposure[
        "eligible_user_opportunities"
    ] > 0,
    creator_opportunity_exposure[
        "baseline_exposure_slots"
    ] / creator_opportunity_exposure[
        "eligible_user_opportunities"
    ],
    np.nan
)


# --------------------------------------------------
# 7. Diagnostics
# --------------------------------------------------

print("CREATOR OPPORTUNITY VS. EXPOSURE")
print("=" * 60)

print("Creators:", len(creator_opportunity_exposure))

print("\nOverall creator statistics:")
display(
    creator_opportunity_exposure[
        [
            "eligible_user_opportunities",
            "baseline_exposure_slots",
            "baseline_exposed_users",
            "baseline_exposure_per_opportunity"
        ]
    ].describe()
)

print("\nStatistics by creator group:")
display(
    creator_opportunity_exposure.groupby("creator_group")[
        [
            "eligible_user_opportunities",
            "baseline_exposure_slots",
            "baseline_exposed_users",
            "baseline_exposure_per_opportunity"
        ]
    ].mean()
)

print("\nCreators with eligible opportunities but zero baseline exposure:")

display(
    creator_opportunity_exposure[
        (creator_opportunity_exposure["eligible_user_opportunities"] > 0)
        & (creator_opportunity_exposure["baseline_exposure_slots"] == 0)
    ].groupby("creator_group").size()
)

print("\nTop 15 creators by eligible user opportunities:")
display(
    creator_opportunity_exposure.sort_values(
        "eligible_user_opportunities",
        ascending=False
    ).head(15)
)

print("\nTop 15 creators by baseline exposure:")
display(
    creator_opportunity_exposure.sort_values(
        "baseline_exposure_slots",
        ascending=False
    ).head(15)
)

print("\n✅ CREATOR OPPORTUNITY–EXPOSURE ANALYSIS COMPLETED!")

CREATOR OPPORTUNITY VS. EXPOSURE
Creators: 2031

Overall creator statistics:


,eligible_user_opportunities,baseline_exposure_slots,baseline_exposed_users,baseline_exposure_per_opportunity
count,2031.000000,2031.000000,2031.000000,1509.000000
mean,23.548991,6.947317,6.778434,0.325761
std,45.047566,19.592868,18.364217,1.499377
min,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.000000,0.000000,0.000000
50%,7.000000,0.000000,0.000000,0.043011
75%,27.000000,5.000000,5.000000,0.250000
max,641.000000,342.000000,284.000000,27.000000



Statistics by creator group:


,eligible_user_opportunities,baseline_exposure_slots,baseline_exposed_users,baseline_exposure_per_opportunity
creator_group,,,,
Emerging,17.374257,7.283168,7.269307,0.398945
Established,25.778218,6.885809,6.664026,0.301817



Creators with eligible opportunities but zero baseline exposure:


creator_group
Emerging       125
Established    562
dtype: int64


Top 15 creators by eligible user opportunities:


,author_id,creator_group,eligible_user_opportunities,baseline_exposure_slots,baseline_exposed_users,baseline_unique_videos,baseline_exposure_per_opportunity
27,5690,Established,641.0,342.0,284.0,24.0,0.533541
265,6783,Established,456.0,169.0,149.0,9.0,0.370614
224,5718,Established,450.0,209.0,191.0,8.0,0.464444
297,7318,Established,357.0,186.0,179.0,9.0,0.521008
88,6476,Established,349.0,168.0,147.0,12.0,0.481375
266,1481,Established,332.0,116.0,113.0,6.0,0.349398
451,6426,Established,316.0,96.0,95.0,3.0,0.303797
174,4615,Established,306.0,101.0,97.0,6.0,0.330065
35,6229,Established,300.0,85.0,75.0,10.0,0.283333
100,1580,Established,291.0,80.0,75.0,5.0,0.274914



Top 15 creators by baseline exposure:


,author_id,creator_group,eligible_user_opportunities,baseline_exposure_slots,baseline_exposed_users,baseline_unique_videos,baseline_exposure_per_opportunity
27,5690,Established,641.0,342.0,284.0,24.0,0.533541
224,5718,Established,450.0,209.0,191.0,8.0,0.464444
15,6332,Established,288.0,197.0,171.0,8.0,0.684028
297,7318,Established,357.0,186.0,179.0,9.0,0.521008
265,6783,Established,456.0,169.0,149.0,9.0,0.370614
391,4394,Established,221.0,168.0,152.0,5.0,0.760181
88,6476,Established,349.0,168.0,147.0,12.0,0.481375
249,6911,Established,288.0,156.0,143.0,9.0,0.541667
149,2230,Established,264.0,150.0,138.0,4.0,0.568182
370,239,Emerging,87.0,126.0,126.0,1.0,1.448276



✅ CREATOR OPPORTUNITY–EXPOSURE ANALYSIS COMPLETED!


In [55]:
# Cell 38: Opportunity-adjusted creator coverage

# --------------------------------------------------
# 1. Eligible user-creator opportunities
# --------------------------------------------------

eligible_user_creator_pairs = (
    spotlight_features[
        spotlight_features["interest_eligible"]
    ][["user_id", "author_id", "creator_group"]]
    .drop_duplicates()
)


# --------------------------------------------------
# 2. Baseline user-creator exposure pairs
# --------------------------------------------------

baseline_user_creator_pairs = (
    baseline_with_creator[
        ["user_id", "author_id", "creator_group"]
    ]
    .drop_duplicates()
)


# --------------------------------------------------
# 3. Identify eligible opportunities actually exposed
# --------------------------------------------------

eligible_exposed_pairs = eligible_user_creator_pairs.merge(
    baseline_user_creator_pairs[
        ["user_id", "author_id"]
    ],
    on=["user_id", "author_id"],
    how="inner"
)


# --------------------------------------------------
# 4. Creator-level opportunity coverage
# --------------------------------------------------

creator_opportunity_coverage = (
    eligible_user_creator_pairs
    .groupby(["author_id", "creator_group"])
    .agg(
        eligible_user_opportunities=("user_id", "nunique")
    )
    .reset_index()
    .merge(
        eligible_exposed_pairs.groupby(
            ["author_id", "creator_group"]
        ).agg(
            eligible_users_exposed=("user_id", "nunique")
        ).reset_index(),
        on=["author_id", "creator_group"],
        how="left"
    )
)

creator_opportunity_coverage[
    "eligible_users_exposed"
] = (
    creator_opportunity_coverage[
        "eligible_users_exposed"
    ].fillna(0)
)

creator_opportunity_coverage[
    "opportunity_coverage"
] = (
    creator_opportunity_coverage[
        "eligible_users_exposed"
    ]
    / creator_opportunity_coverage[
        "eligible_user_opportunities"
    ]
)


# --------------------------------------------------
# 5. Group-level summary
# --------------------------------------------------

coverage_summary = (
    creator_opportunity_coverage
    .groupby("creator_group")
    .agg(
        creators=("author_id", "nunique"),
        mean_opportunities=("eligible_user_opportunities", "mean"),
        total_opportunities=("eligible_user_opportunities", "sum"),
        mean_opportunity_coverage=("opportunity_coverage", "mean"),
        median_opportunity_coverage=("opportunity_coverage", "median"),
        zero_coverage_creators=(
            "opportunity_coverage",
            lambda x: (x == 0).sum()
        )
    )
)

print("OPPORTUNITY-ADJUSTED CREATOR COVERAGE")
print("=" * 60)

display(coverage_summary)


# --------------------------------------------------
# 6. Creators with opportunity but no eligible exposure
# --------------------------------------------------

zero_coverage_creators = creator_opportunity_coverage[
    creator_opportunity_coverage["opportunity_coverage"] == 0
].copy()

print("\nCreators with zero opportunity coverage:")
print("Total:", len(zero_coverage_creators))

display(
    zero_coverage_creators
    .sort_values("eligible_user_opportunities", ascending=False)
    .head(20)
)


# --------------------------------------------------
# 7. Validation
# --------------------------------------------------

assert creator_opportunity_coverage[
    "opportunity_coverage"
].between(0, 1).all()

assert (
    eligible_exposed_pairs["user_id"].nunique()
    <= eligible_user_creator_pairs["user_id"].nunique()
)

print("\n✅ OPPORTUNITY-ADJUSTED COVERAGE COMPLETED!")

OPPORTUNITY-ADJUSTED CREATOR COVERAGE


,creators,mean_opportunities,total_opportunities,mean_opportunity_coverage,median_opportunity_coverage,zero_coverage_creators
creator_group,,,,,,
Emerging,372,23.586022,8774,0.202216,0.111111,141
Established,1137,34.348285,39054,0.104533,0.000000,606



Creators with zero opportunity coverage:
Total: 747


,author_id,creator_group,eligible_user_opportunities,eligible_users_exposed,opportunity_coverage
383,1874,Established,129,0.0,0.0
995,5648,Established,82,0.0,0.0
831,4706,Established,54,0.0,0.0
691,3945,Established,54,0.0,0.0
382,1867,Established,51,0.0,0.0
1187,6512,Established,47,0.0,0.0
270,1176,Established,46,0.0,0.0
1002,5687,Established,45,0.0,0.0
933,5299,Established,42,0.0,0.0
1271,6955,Established,42,0.0,0.0



✅ OPPORTUNITY-ADJUSTED COVERAGE COMPLETED!


In [56]:
# Cell 39: Identify underexposed Emerging creators

# --------------------------------------------------
# 1. Restrict to Emerging creators with opportunities
# --------------------------------------------------

emerging_opportunity_creators = (
    creator_opportunity_coverage[
        (creator_opportunity_coverage["creator_group"] == "Emerging")
        & (creator_opportunity_coverage["eligible_user_opportunities"] > 0)
    ].copy()
)


# --------------------------------------------------
# 2. Distribution of opportunity coverage
# --------------------------------------------------

print("EMERGING CREATOR OPPORTUNITY COVERAGE")
print("=" * 60)

display(
    emerging_opportunity_creators[
        [
            "eligible_user_opportunities",
            "eligible_users_exposed",
            "opportunity_coverage"
        ]
    ].describe(percentiles=[0.10, 0.25, 0.50, 0.75, 0.90])
)


# --------------------------------------------------
# 3. Zero and low coverage counts
# --------------------------------------------------

coverage_thresholds = [0, 0.05, 0.10, 0.20, 0.30]

coverage_threshold_summary = []

for threshold in coverage_thresholds:

    count = (
        emerging_opportunity_creators["opportunity_coverage"]
        <= threshold
    ).sum()

    coverage_threshold_summary.append({
        "coverage_threshold": threshold,
        "creators_at_or_below": count,
        "percentage": (
            count
            / len(emerging_opportunity_creators)
            * 100
        )
    })

coverage_threshold_summary = pd.DataFrame(
    coverage_threshold_summary
)

print("\nCreators at different coverage levels:")
display(coverage_threshold_summary)


# --------------------------------------------------
# 4. Identify creators with meaningful opportunity
#    but very low exposure coverage
# --------------------------------------------------

underexposed_emerging = (
    emerging_opportunity_creators[
        (emerging_opportunity_creators["eligible_user_opportunities"] >= 5)
        & (emerging_opportunity_creators["opportunity_coverage"] <= 0.10)
    ]
    .sort_values(
        ["eligible_user_opportunities", "opportunity_coverage"],
        ascending=[False, True]
    )
)

print("\nEmerging creators with:")
print("- At least 5 eligible user opportunities")
print("- Opportunity coverage <= 10%")

print("Count:", len(underexposed_emerging))

display(
    underexposed_emerging.head(25)
)


# --------------------------------------------------
# 5. Validation
# --------------------------------------------------

assert emerging_opportunity_creators[
    "opportunity_coverage"
].between(0, 1).all()

print("\n✅ UNDEREXPOSED EMERGING CREATOR ANALYSIS COMPLETED!")

EMERGING CREATOR OPPORTUNITY COVERAGE


,eligible_user_opportunities,eligible_users_exposed,opportunity_coverage
count,372.000000,372.000000,372.000000
mean,23.586022,7.483871,0.202216
std,25.226320,13.217943,0.242738
min,1.000000,0.000000,0.000000
10%,2.000000,0.000000,0.000000
25%,6.000000,0.000000,0.000000
50%,14.000000,1.000000,0.111111
75%,34.000000,9.000000,0.327083
90%,56.800000,24.000000,0.576374
max,134.000000,91.000000,1.000000



Creators at different coverage levels:


,coverage_threshold,creators_at_or_below,percentage
0,0.00,141,37.903226
1,0.05,148,39.784946
2,0.10,183,49.193548
3,0.20,234,62.903226
4,0.30,268,72.043011



Emerging creators with:
- At least 5 eligible user opportunities
- Opportunity coverage <= 10%
Count: 119


,author_id,creator_group,eligible_user_opportunities,eligible_users_exposed,opportunity_coverage
1218,6684,Emerging,78,5.0,0.064103
210,849,Emerging,77,6.0,0.077922
261,1126,Emerging,74,1.0,0.013514
641,3542,Emerging,60,5.0,0.083333
760,4331,Emerging,57,2.0,0.035088
507,2592,Emerging,54,3.0,0.055556
467,2397,Emerging,54,5.0,0.092593
1213,6651,Emerging,44,3.0,0.068182
361,1750,Emerging,42,3.0,0.071429
592,3159,Emerging,39,0.0,0.000000



✅ UNDEREXPOSED EMERGING CREATOR ANALYSIS COMPLETED!


In [57]:
# Cell 40: Validate Spotlight target population

# --------------------------------------------------
# 1. Define target creators
# --------------------------------------------------

SPOTLIGHT_MIN_OPPORTUNITIES = 5
SPOTLIGHT_MAX_COVERAGE = 0.10

spotlight_target_creators = (
    creator_opportunity_coverage[
        (creator_opportunity_coverage["creator_group"] == "Emerging")
        & (
            creator_opportunity_coverage[
                "eligible_user_opportunities"
            ] >= SPOTLIGHT_MIN_OPPORTUNITIES
        )
        & (
            creator_opportunity_coverage[
                "opportunity_coverage"
            ] <= SPOTLIGHT_MAX_COVERAGE
        )
    ].copy()
)


target_creator_ids = set(
    spotlight_target_creators["author_id"]
)


# --------------------------------------------------
# 2. Eligible candidate representation
# --------------------------------------------------

target_candidates = spotlight_features[
    spotlight_features["interest_eligible"]
    & spotlight_features["author_id"].isin(target_creator_ids)
].copy()


# --------------------------------------------------
# 3. Creator-level candidate availability
# --------------------------------------------------

target_candidate_availability = (
    target_candidates.groupby(
        ["author_id", "creator_group"]
    )
    .agg(
        eligible_candidate_pairs=("user_id", "count"),
        eligible_users=("user_id", "nunique"),
        eligible_videos=("video_id", "nunique")
    )
    .reset_index()
)


# --------------------------------------------------
# 4. Summary
# --------------------------------------------------

print("SPOTLIGHT TARGET POPULATION VALIDATION")
print("=" * 60)

print("Target creators:", len(spotlight_target_creators))

print("Creators with candidate representation:",
      target_candidate_availability["author_id"].nunique())

print("Eligible candidate pairs:", len(target_candidates))

print("Users with target creator candidates:",
      target_candidates["user_id"].nunique())

print("Unique target videos:",
      target_candidates["video_id"].nunique())


print("\nTarget creator opportunity statistics:")
display(
    spotlight_target_creators[
        [
            "eligible_user_opportunities",
            "eligible_users_exposed",
            "opportunity_coverage"
        ]
    ].describe()
)


print("\nCandidate availability per target creator:")
display(
    target_candidate_availability[
        [
            "eligible_candidate_pairs",
            "eligible_users",
            "eligible_videos"
        ]
    ].describe()
)


# --------------------------------------------------
# 5. Distribution across users
# --------------------------------------------------

target_candidates_per_user = (
    target_candidates.groupby("user_id")
    .size()
    .reindex(
        spotlight_features["user_id"].unique(),
        fill_value=0
    )
)

print("\nTarget creator candidates per user:")
display(target_candidates_per_user.describe())


print("\nUsers with at least one target candidate:",
      (target_candidates_per_user > 0).sum())

print("Users with at least 3 target candidates:",
      (target_candidates_per_user >= 3).sum())

print("Users with at least 5 target candidates:",
      (target_candidates_per_user >= 5).sum())


# --------------------------------------------------
# 6. Validation
# --------------------------------------------------

assert len(spotlight_target_creators) == len(target_creator_ids)

assert (
    target_candidates["interest_eligible"].all()
)

assert (
    target_candidate_availability["author_id"].nunique()
    <= len(target_creator_ids)
)

print("\n✅ SPOTLIGHT TARGET POPULATION VALIDATED!")

SPOTLIGHT TARGET POPULATION VALIDATION
Target creators: 119
Creators with candidate representation: 119
Eligible candidate pairs: 1981
Users with target creator candidates: 1005
Unique target videos: 131

Target creator opportunity statistics:


,eligible_user_opportunities,eligible_users_exposed,opportunity_coverage
count,119.000000,119.000000,119.000000
mean,16.537815,0.621849,0.023681
std,15.237548,1.157083,0.034708
min,5.000000,0.000000,0.000000
25%,7.000000,0.000000,0.000000
50%,11.000000,0.000000,0.000000
75%,18.500000,1.000000,0.055556
max,78.000000,6.000000,0.100000



Candidate availability per target creator:


,eligible_candidate_pairs,eligible_users,eligible_videos
count,119.000000,119.000000,119.000000
mean,16.647059,16.537815,1.100840
std,15.509204,15.237548,0.329225
min,5.000000,5.000000,1.000000
25%,7.000000,7.000000,1.000000
50%,11.000000,11.000000,1.000000
75%,19.000000,18.500000,1.000000
max,78.000000,78.000000,3.000000



Target creator candidates per user:


count    1411.000000
mean        1.403969
std         1.364634
min         0.000000
25%         0.000000
50%         1.000000
75%         2.000000
max         9.000000
dtype: float64


Users with at least one target candidate: 1005
Users with at least 3 target candidates: 255
Users with at least 5 target candidates: 41

✅ SPOTLIGHT TARGET POPULATION VALIDATED!


In [58]:
# Cell 41: Rank target creators by exploration priority

# --------------------------------------------------
# 1. Create deterministic priority ranking
# --------------------------------------------------

spotlight_creator_priority = spotlight_target_creators.copy()

spotlight_creator_priority = (
    spotlight_creator_priority
    .sort_values(
        [
            "opportunity_coverage",
            "eligible_user_opportunities",
            "author_id"
        ],
        ascending=[True, False, True]
    )
    .reset_index(drop=True)
)

spotlight_creator_priority["exploration_priority_rank"] = (
    np.arange(1, len(spotlight_creator_priority) + 1)
)


# --------------------------------------------------
# 2. Priority distribution
# --------------------------------------------------

print("SPOTLIGHT CREATOR PRIORITY RANKING")
print("=" * 60)

print("Target creators:",
      len(spotlight_creator_priority))

print("\nOpportunity coverage distribution:")
display(
    spotlight_creator_priority[
        "opportunity_coverage"
    ].describe()
)


# --------------------------------------------------
# 3. Top priority creators
# --------------------------------------------------

print("\nTop 25 creators by exploration priority:")

display(
    spotlight_creator_priority[
        [
            "author_id",
            "eligible_user_opportunities",
            "eligible_users_exposed",
            "opportunity_coverage",
            "exploration_priority_rank"
        ]
    ].head(25)
)


# --------------------------------------------------
# 4. Priority ranking distribution
# --------------------------------------------------

priority_summary = (
    spotlight_creator_priority
    .assign(
        priority_bucket=pd.cut(
            spotlight_creator_priority[
                "exploration_priority_rank"
            ],
            bins=[0, 25, 50, 75, 100, 119],
            labels=[
                "Top 25",
                "26–50",
                "51–75",
                "76–100",
                "101–119"
            ]
        )
    )
    .groupby("priority_bucket", observed=False)
    .agg(
        creators=("author_id", "count"),
        mean_opportunities=(
            "eligible_user_opportunities", "mean"
        ),
        mean_coverage=("opportunity_coverage", "mean")
    )
)

print("\nPriority buckets:")
display(priority_summary)


# --------------------------------------------------
# 5. Validation
# --------------------------------------------------

assert spotlight_creator_priority[
    "opportunity_coverage"
].is_monotonic_increasing

assert spotlight_creator_priority[
    "author_id"
].nunique() == len(spotlight_creator_priority)

assert spotlight_creator_priority[
    "exploration_priority_rank"
].min() == 1

assert spotlight_creator_priority[
    "exploration_priority_rank"
].max() == len(spotlight_creator_priority)


print("\n✅ CREATOR PRIORITY RANKING COMPLETED!")

SPOTLIGHT CREATOR PRIORITY RANKING
Target creators: 119

Opportunity coverage distribution:


count    119.000000
mean       0.023681
std        0.034708
min        0.000000
25%        0.000000
50%        0.000000
75%        0.055556
max        0.100000
Name: opportunity_coverage, dtype: float64


Top 25 creators by exploration priority:


,author_id,eligible_user_opportunities,eligible_users_exposed,opportunity_coverage,exploration_priority_rank
0,3159,39,0.0,0.0,1
1,536,31,0.0,0.0,2
2,4355,27,0.0,0.0,3
3,6156,25,0.0,0.0,4
4,1677,23,0.0,0.0,5
5,1756,23,0.0,0.0,6
6,2946,22,0.0,0.0,7
7,611,17,0.0,0.0,8
8,3524,17,0.0,0.0,9
9,629,15,0.0,0.0,10



Priority buckets:


,creators,mean_opportunities,mean_coverage
priority_bucket,,,
Top 25,25,17.000000,0.000000
26–50,25,8.160000,0.000000
51–75,25,5.680000,0.000000
76–100,25,28.960000,0.046573
101–119,19,24.894737,0.087039



✅ CREATOR PRIORITY RANKING COMPLETED!


In [59]:
# Cell 42: Baseline overlap with Spotlight target candidates

# --------------------------------------------------
# 1. Extract baseline Top-10 video-user pairs
# --------------------------------------------------

baseline_top10_pairs = baseline_recommendations[
    ["user_id", "video_id"]
].drop_duplicates()


# --------------------------------------------------
# 2. Extract eligible target candidate pairs
# --------------------------------------------------

target_candidate_pairs = target_candidates[
    ["user_id", "video_id", "author_id"]
].drop_duplicates()


# --------------------------------------------------
# 3. Check overlap
# --------------------------------------------------

target_candidate_overlap = target_candidate_pairs.merge(
    baseline_top10_pairs,
    on=["user_id", "video_id"],
    how="inner"
)

print("SPOTLIGHT BASELINE OVERLAP ANALYSIS")
print("=" * 60)

print("Eligible target candidate pairs:",
      len(target_candidate_pairs))

print("Target candidates already in baseline Top-10:",
      len(target_candidate_overlap))

print("Distinct users already receiving target candidates:",
      target_candidate_overlap["user_id"].nunique())

print("Distinct target creators already exposed:",
      target_candidate_overlap["author_id"].nunique())


# --------------------------------------------------
# 4. Calculate genuinely new exploration opportunities
# --------------------------------------------------

new_target_candidate_pairs = target_candidate_pairs.merge(
    baseline_top10_pairs,
    on=["user_id", "video_id"],
    how="left",
    indicator=True
)

new_target_candidate_pairs = new_target_candidate_pairs[
    new_target_candidate_pairs["_merge"] == "left_only"
].drop(columns="_merge")


print("\nNew exploration opportunities:")
print("Candidate pairs not present in baseline Top-10:",
      len(new_target_candidate_pairs))

print("Users with new target opportunities:",
      new_target_candidate_pairs["user_id"].nunique())

print("Unique target videos:",
      new_target_candidate_pairs["video_id"].nunique())

print("Unique target creators:",
      new_target_candidate_pairs["author_id"].nunique())


# --------------------------------------------------
# 5. Validation
# --------------------------------------------------

assert len(target_candidate_pairs) >= len(target_candidate_overlap)

assert len(new_target_candidate_pairs) + \
       len(target_candidate_overlap) == \
       len(target_candidate_pairs)


print("\n✅ BASELINE OVERLAP ANALYSIS COMPLETED!")

SPOTLIGHT BASELINE OVERLAP ANALYSIS
Eligible target candidate pairs: 1981
Target candidates already in baseline Top-10: 74
Distinct users already receiving target candidates: 71
Distinct target creators already exposed: 42

New exploration opportunities:
Candidate pairs not present in baseline Top-10: 1907
Users with new target opportunities: 986
Unique target videos: 131
Unique target creators: 119

✅ BASELINE OVERLAP ANALYSIS COMPLETED!


In [60]:
# Cell 43: Define Spotlight intervention parameters

# --------------------------------------------------
# Core recommendation settings
# --------------------------------------------------

SPOTLIGHT_TOP_K = 10

SPOTLIGHT_CANDIDATE_POOL_SIZE = 50

SPOTLIGHT_MAX_EXPLORATION_SLOTS = 1

SPOTLIGHT_MIN_CF_SCORE = 0.0

SPOTLIGHT_MIN_INTEREST_PERCENTILE = 0.30

SPOTLIGHT_MIN_CREATOR_OPPORTUNITIES = 5

SPOTLIGHT_MAX_CREATOR_COVERAGE = 0.10


# --------------------------------------------------
# Target creator population
# --------------------------------------------------

SPOTLIGHT_TARGET_CREATOR_COUNT = (
    len(spotlight_target_creators)
)


# --------------------------------------------------
# Print configuration
# --------------------------------------------------

print("SPOTLIGHT INTERVENTION CONFIGURATION")
print("=" * 60)

print("Recommendation list size:",
      SPOTLIGHT_TOP_K)

print("Candidate pool size:",
      SPOTLIGHT_CANDIDATE_POOL_SIZE)

print("Maximum exploration slots:",
      SPOTLIGHT_MAX_EXPLORATION_SLOTS)

print("Minimum CF score:",
      SPOTLIGHT_MIN_CF_SCORE)

print("Minimum interest percentile:",
      SPOTLIGHT_MIN_INTEREST_PERCENTILE)

print("Minimum creator opportunities:",
      SPOTLIGHT_MIN_CREATOR_OPPORTUNITIES)

print("Maximum creator opportunity coverage:",
      SPOTLIGHT_MAX_CREATOR_COVERAGE)

print("Target creators:",
      SPOTLIGHT_TARGET_CREATOR_COUNT)


# --------------------------------------------------
# Validation
# --------------------------------------------------

assert SPOTLIGHT_TOP_K == 10

assert SPOTLIGHT_CANDIDATE_POOL_SIZE >= SPOTLIGHT_TOP_K

assert SPOTLIGHT_MAX_EXPLORATION_SLOTS >= 1

assert SPOTLIGHT_MIN_INTEREST_PERCENTILE == 0.30

assert SPOTLIGHT_TARGET_CREATOR_COUNT == 119


print("\n✅ SPOTLIGHT PARAMETERS DEFINED!")

SPOTLIGHT INTERVENTION CONFIGURATION
Recommendation list size: 10
Candidate pool size: 50
Maximum exploration slots: 1
Minimum CF score: 0.0
Minimum interest percentile: 0.3
Minimum creator opportunities: 5
Maximum creator opportunity coverage: 0.1
Target creators: 119

✅ SPOTLIGHT PARAMETERS DEFINED!


In [61]:
# Cell 44: Build the eligible Spotlight exploration pool

# --------------------------------------------------
# 1. Start with target creator candidates
# --------------------------------------------------

spotlight_exploration_candidates = target_candidates.copy()


# --------------------------------------------------
# 2. Apply CF relevance constraint
# --------------------------------------------------

spotlight_exploration_candidates = (
    spotlight_exploration_candidates[
        spotlight_exploration_candidates[
            "predicted_preference"
        ] > SPOTLIGHT_MIN_CF_SCORE
    ]
    .copy()
)


# --------------------------------------------------
# 3. Exclude videos already in baseline Top-10
# --------------------------------------------------

spotlight_exploration_candidates = (
    spotlight_exploration_candidates.merge(
        baseline_top10_pairs,
        on=["user_id", "video_id"],
        how="left",
        indicator=True
    )
)

spotlight_exploration_candidates = (
    spotlight_exploration_candidates[
        spotlight_exploration_candidates["_merge"]
        == "left_only"
    ]
    .drop(columns="_merge")
)


# --------------------------------------------------
# 4. Validate candidate pool
# --------------------------------------------------

spotlight_exploration_candidates = (
    spotlight_exploration_candidates
    .drop_duplicates(
        subset=["user_id", "video_id"]
    )
    .reset_index(drop=True)
)


print("SPOTLIGHT ELIGIBLE EXPLORATION POOL")
print("=" * 60)

print("Eligible exploration candidates:",
      len(spotlight_exploration_candidates))

print("Unique users:",
      spotlight_exploration_candidates["user_id"].nunique())

print("Unique videos:",
      spotlight_exploration_candidates["video_id"].nunique())

print("Unique target creators:",
      spotlight_exploration_candidates["author_id"].nunique())

print("\nCF relevance score:")
display(
    spotlight_exploration_candidates[
        "predicted_preference"
    ].describe()
)


# --------------------------------------------------
# 5. Candidate availability per user
# --------------------------------------------------

exploration_candidates_per_user = (
    spotlight_exploration_candidates
    .groupby("user_id")
    .size()
)

print("\nCandidates per user:")
display(
    exploration_candidates_per_user.describe()
)


# --------------------------------------------------
# 6. Validation
# --------------------------------------------------

assert (
    spotlight_exploration_candidates[
        "interest_eligible"
    ].all()
)

assert (
    spotlight_exploration_candidates[
        "predicted_preference"
    ].gt(SPOTLIGHT_MIN_CF_SCORE).all()
)

assert (
    spotlight_exploration_candidates[
        ["user_id", "video_id"]
    ]
    .duplicated()
    .sum() == 0
)

assert (
    spotlight_exploration_candidates[
        "author_id"
    ].nunique() <= SPOTLIGHT_TARGET_CREATOR_COUNT
)


print("\n✅ SPOTLIGHT EXPLORATION POOL VALIDATED!")

SPOTLIGHT ELIGIBLE EXPLORATION POOL
Eligible exploration candidates: 1907
Unique users: 986
Unique videos: 131
Unique target creators: 119

CF relevance score:


count    1907.000000
mean        0.854434
std         0.088152
min         0.557486
25%         0.798830
50%         0.858807
75%         0.927803
max         1.000000
Name: predicted_preference, dtype: float64


Candidates per user:


count    986.000000
mean       1.934077
std        1.187377
min        1.000000
25%        1.000000
50%        2.000000
75%        2.000000
max        9.000000
dtype: float64


✅ SPOTLIGHT EXPLORATION POOL VALIDATED!


In [62]:
# Cell 45: Build unique user-creator exploration opportunities

# --------------------------------------------------
# 1. Attach creator priority
# --------------------------------------------------

user_creator_exploration = (
    spotlight_exploration_candidates
    .merge(
        spotlight_creator_priority[
            [
                "author_id",
                "exploration_priority_rank",
                "eligible_user_opportunities",
                "opportunity_coverage"
            ]
        ],
        on="author_id",
        how="left",
        validate="many_to_one"
    )
)


# --------------------------------------------------
# 2. Consolidate video candidates into user-creator pairs
# --------------------------------------------------

user_creator_exploration = (
    user_creator_exploration
    .sort_values(
        [
            "user_id",
            "author_id",
            "predicted_preference",
            "interest_percentile"
        ],
        ascending=[True, True, False, False]
    )
    .drop_duplicates(
        subset=["user_id", "author_id"],
        keep="first"
    )
    .reset_index(drop=True)
)


# --------------------------------------------------
# 3. Validate user-creator opportunities
# --------------------------------------------------

print("USER-CREATOR EXPLORATION OPPORTUNITIES")
print("=" * 60)

print("Unique user-creator opportunities:",
      len(user_creator_exploration))

print("Unique users:",
      user_creator_exploration["user_id"].nunique())

print("Unique creators:",
      user_creator_exploration["author_id"].nunique())

print("Unique videos retained:",
      user_creator_exploration["video_id"].nunique())


# --------------------------------------------------
# 4. Creator opportunities per user
# --------------------------------------------------

creators_per_user = (
    user_creator_exploration
    .groupby("user_id")["author_id"]
    .nunique()
)

print("\nDistinct target creators available per user:")
display(creators_per_user.describe())


# --------------------------------------------------
# 5. Creator opportunity distribution
# --------------------------------------------------

creator_user_opportunities = (
    user_creator_exploration
    .groupby("author_id")
    .agg(
        available_users=("user_id", "nunique"),
        candidate_videos=("video_id", "nunique"),
        priority_rank=(
            "exploration_priority_rank",
            "first"
        ),
        baseline_coverage=(
            "opportunity_coverage",
            "first"
        )
    )
    .sort_values("priority_rank")
)

print("\nCreator opportunity distribution:")
display(
    creator_user_opportunities[
        ["available_users", "candidate_videos"]
    ].describe()
)


# --------------------------------------------------
# 6. Validation
# --------------------------------------------------

assert not user_creator_exploration.duplicated(
    subset=["user_id", "author_id"]
).any()

assert user_creator_exploration[
    "exploration_priority_rank"
].notna().all()

assert user_creator_exploration[
    "author_id"
].nunique() <= SPOTLIGHT_TARGET_CREATOR_COUNT


print("\n✅ USER-CREATOR EXPLORATION MATRIX VALIDATED!")

USER-CREATOR EXPLORATION OPPORTUNITIES
Unique user-creator opportunities: 1896
Unique users: 986
Unique creators: 119
Unique videos retained: 131

Distinct target creators available per user:


count    986.000000
mean       1.922921
std        1.175961
min        1.000000
25%        1.000000
50%        2.000000
75%        2.000000
max        9.000000
Name: author_id, dtype: float64


Creator opportunity distribution:


,available_users,candidate_videos
count,119.000000,119.000000
mean,15.932773,1.100840
std,14.321805,0.329225
min,5.000000,1.000000
25%,7.000000,1.000000
50%,10.000000,1.000000
75%,17.500000,1.000000
max,73.000000,3.000000



✅ USER-CREATOR EXPLORATION MATRIX VALIDATED!


In [64]:
# Cell 46: Select one Spotlight exploration creator per user

# --------------------------------------------------
# 1. Rank eligible creators for each user
# --------------------------------------------------

spotlight_user_selections = (
    user_creator_exploration
    .sort_values(
        [
            "user_id",
            "opportunity_coverage",
            "eligible_user_opportunities",
            "predicted_preference",
            "interest_percentile",
            "author_id"
        ],
        ascending=[
            True,   # user
            True,   # lower exposure coverage first
            False,  # more eligible opportunities first
            False,  # higher CF relevance first
            False,  # higher interest percentile first
            True    # deterministic creator ID tie-breaker
        ]
    )
    .groupby("user_id", as_index=False)
    .head(SPOTLIGHT_MAX_EXPLORATION_SLOTS)
    .reset_index(drop=True)
)


# --------------------------------------------------
# 2. Validate selection
# --------------------------------------------------

print("SPOTLIGHT EXPLORATION SELECTION")
print("=" * 60)

print("Selected exploration recommendations:",
      len(spotlight_user_selections))

print("Users receiving exploration:",
      spotlight_user_selections["user_id"].nunique())

print("Creators selected:",
      spotlight_user_selections["author_id"].nunique())

print("Videos selected:",
      spotlight_user_selections["video_id"].nunique())


# --------------------------------------------------
# 3. Selected creator distribution
# --------------------------------------------------

selected_creator_distribution = (
    spotlight_user_selections
    .groupby("author_id")
    .agg(
        selected_users=("user_id", "nunique"),
        baseline_coverage=("opportunity_coverage", "first"),
        creator_priority=(
            "exploration_priority_rank",
            "first"
        )
    )
    .sort_values(
        ["selected_users", "creator_priority"],
        ascending=[False, True]
    )
)

print("\nTop selected creators:")
display(selected_creator_distribution.head(20))


# --------------------------------------------------
# 4. Relevance of selected candidates
# --------------------------------------------------

print("\nSelected candidate relevance:")
display(
    spotlight_user_selections[
        "predicted_preference"
    ].describe()
)


# --------------------------------------------------
# 5. Validation
# --------------------------------------------------

assert not spotlight_user_selections.duplicated(
    subset=["user_id"]
).any()

assert (
    spotlight_user_selections["user_id"].nunique()
    <= len(creators_per_user)
)

assert (
    spotlight_user_selections["opportunity_coverage"]
    .le(SPOTLIGHT_MAX_CREATOR_COVERAGE)
    .all()
)

assert (
    spotlight_user_selections["interest_eligible"]
    .all()
)

print("\n✅ SPOTLIGHT EXPLORATION SELECTION COMPLETED!")

SPOTLIGHT EXPLORATION SELECTION
Selected exploration recommendations: 986
Users receiving exploration: 986
Creators selected: 117
Videos selected: 128

Top selected creators:


,selected_users,baseline_coverage,creator_priority
author_id,,,
3159,39,0.000000,1
4331,37,0.035088,82
6684,36,0.064103,98
1126,34,0.013514,78
536,31,0.000000,2
4355,27,0.000000,3
6156,22,0.000000,4
1677,22,0.000000,5
2946,20,0.000000,7



Selected candidate relevance:


count    986.000000
mean       0.858910
std        0.088650
min        0.557486
25%        0.805633
50%        0.862328
75%        0.933900
max        1.000000
Name: predicted_preference, dtype: float64


✅ SPOTLIGHT EXPLORATION SELECTION COMPLETED!


In [65]:
# Cell 47: Inspect recommendation schemas before reranking

print("BASELINE RECOMMENDATIONS")
print("=" * 60)

print("Shape:", baseline_recommendations.shape)
print("Columns:", baseline_recommendations.columns.tolist())

display(baseline_recommendations.head())


print("\nSPOTLIGHT SELECTIONS")
print("=" * 60)

print("Shape:", spotlight_user_selections.shape)
print("Columns:", spotlight_user_selections.columns.tolist())

display(
    spotlight_user_selections[
        [
            "user_id",
            "video_id",
            "author_id",
            "predicted_preference",
            "interest_percentile",
            "exploration_priority_rank"
        ]
    ].head()
)


print("\nBASELINE RECOMMENDATION COUNT PER USER")
print("=" * 60)

baseline_counts = (
    baseline_recommendations
    .groupby("user_id")
    .size()
)

display(baseline_counts.describe())


print("\nBaseline recommendation score column candidates:")

for column in baseline_recommendations.columns:
    if "score" in column.lower() or "preference" in column.lower():
        print(column)

BASELINE RECOMMENDATIONS
Shape: (14110, 3)
Columns: ['user_id', 'video_id', 'predicted_preference']


,user_id,video_id,predicted_preference
0,14,3992,0.997153
1,14,7263,0.995710
2,14,835,0.990655
3,14,6177,0.990421
4,14,6086,0.966525



SPOTLIGHT SELECTIONS
Shape: (986, 14)
Columns: ['user_id', 'video_id', 'predicted_preference', 'candidate_rank', 'author_id', 'creator_group', 'category_ids', 'interest_match_score', 'cf_percentile', 'interest_percentile', 'interest_eligible', 'exploration_priority_rank', 'eligible_user_opportunities', 'opportunity_coverage']


,user_id,video_id,author_id,predicted_preference,interest_percentile,exploration_priority_rank
0,14,770,536,0.829176,0.54,2
1,23,7654,3964,0.918251,0.82,11
2,36,870,4120,0.933105,0.30,52
3,41,8607,2954,0.895724,0.61,29
4,51,136,3542,0.841327,0.82,105



BASELINE RECOMMENDATION COUNT PER USER


count    1411.0
mean       10.0
std         0.0
min        10.0
25%        10.0
50%        10.0
75%        10.0
max        10.0
dtype: float64


Baseline recommendation score column candidates:
predicted_preference


In [66]:
# Cell 48: Measure potential relevance trade-off

# --------------------------------------------------
# 1. Identify the lowest-ranked baseline recommendation
# --------------------------------------------------

baseline_bottom10 = (
    baseline_recommendations
    .sort_values(
        ["user_id", "predicted_preference"],
        ascending=[True, True]
    )
    .groupby("user_id")
    .head(1)
    .rename(
        columns={
            "video_id": "baseline_removed_video",
            "predicted_preference": "baseline_removed_score"
        }
    )
    [["user_id", "baseline_removed_video",
      "baseline_removed_score"]]
)


# --------------------------------------------------
# 2. Compare with selected exploration candidate
# --------------------------------------------------

exploration_score_comparison = (
    spotlight_user_selections[
        ["user_id", "video_id", "author_id",
         "predicted_preference"]
    ]
    .rename(
        columns={
            "video_id": "exploration_video",
            "author_id": "exploration_creator",
            "predicted_preference": "exploration_score"
        }
    )
    .merge(
        baseline_bottom10,
        on="user_id",
        how="left",
        validate="one_to_one"
    )
)


# --------------------------------------------------
# 3. Calculate score differences
# --------------------------------------------------

exploration_score_comparison[
    "score_difference"
] = (
    exploration_score_comparison["exploration_score"]
    - exploration_score_comparison["baseline_removed_score"]
)


exploration_score_comparison[
    "relevance_cost"
] = (
    exploration_score_comparison["baseline_removed_score"]
    - exploration_score_comparison["exploration_score"]
)


# --------------------------------------------------
# 4. Summary
# --------------------------------------------------

print("SPOTLIGHT RELEVANCE TRADE-OFF")
print("=" * 60)

print("Users compared:",
      len(exploration_score_comparison))

print("\nExploration candidate score:")
display(
    exploration_score_comparison[
        "exploration_score"
    ].describe()
)

print("\nRemoved baseline score:")
display(
    exploration_score_comparison[
        "baseline_removed_score"
    ].describe()
)

print("\nScore difference (exploration - baseline):")
display(
    exploration_score_comparison[
        "score_difference"
    ].describe()
)


# --------------------------------------------------
# 5. Trade-off counts
# --------------------------------------------------

higher = (
    exploration_score_comparison[
        "score_difference"
    ] > 0
).sum()

equal = (
    exploration_score_comparison[
        "score_difference"
    ] == 0
).sum()

lower = (
    exploration_score_comparison[
        "score_difference"
    ] < 0
).sum()


print("\nExploration relevance comparison:")
print("Exploration candidate scores higher:", higher)
print("Equal score:", equal)
print("Exploration candidate scores lower:", lower)

print("Mean relevance cost:",
      exploration_score_comparison[
          "relevance_cost"
      ].mean())


# --------------------------------------------------
# 6. Validation
# --------------------------------------------------

assert len(exploration_score_comparison) == 986

assert exploration_score_comparison[
    "baseline_removed_score"
].notna().all()

assert exploration_score_comparison[
    "exploration_score"
].notna().all()


print("\n✅ RELEVANCE TRADE-OFF ANALYSIS COMPLETED!")

SPOTLIGHT RELEVANCE TRADE-OFF
Users compared: 986

Exploration candidate score:


count    986.000000
mean       0.858910
std        0.088650
min        0.557486
25%        0.805633
50%        0.862328
75%        0.933900
max        1.000000
Name: exploration_score, dtype: float64


Removed baseline score:


count    986.000000
mean       0.946794
std        0.050653
min        0.662120
25%        0.924835
50%        0.957747
75%        0.985976
max        1.000000
Name: baseline_removed_score, dtype: float64


Score difference (exploration - baseline):


count    986.000000
mean      -0.087884
std        0.074841
min       -0.345317
25%       -0.134673
50%       -0.081912
75%       -0.037007
max        0.258830
Name: score_difference, dtype: float64


Exploration relevance comparison:
Exploration candidate scores higher: 62
Equal score: 2
Exploration candidate scores lower: 922
Mean relevance cost: 0.08788405530966192

✅ RELEVANCE TRADE-OFF ANALYSIS COMPLETED!


In [67]:
# CELL 49: RELEVANCE-COST SENSITIVITY ANALYSIS

cost_thresholds = [0.05, 0.10, 0.15, 0.20]

relevance_sensitivity = []

for threshold in cost_thresholds:

    retained = exploration_score_comparison[
        exploration_score_comparison["relevance_cost"] <= threshold
    ]

    relevance_sensitivity.append({
        "max_relevance_cost": threshold,
        "users_retained": len(retained),
        "retention_percentage": len(retained) / len(exploration_score_comparison) * 100,
        "users_excluded": len(exploration_score_comparison) - len(retained)
    })

relevance_sensitivity_df = pd.DataFrame(relevance_sensitivity)

print("RELEVANCE-COST SENSITIVITY")
print("=" * 60)
print(relevance_sensitivity_df.round(4).to_string(index=False))

RELEVANCE-COST SENSITIVITY
 max_relevance_cost  users_retained  retention_percentage  users_excluded
               0.05             332               33.6714             654
               0.10             572               58.0122             414
               0.15             804               81.5416             182
               0.20             915               92.7992              71


In [69]:
# CELL 50A: INSPECT COLUMN NAMES

print("Columns in exploration_score_comparison:")
print(exploration_score_comparison.columns.tolist())

print("\nColumns in spotlight_user_selections:")
print(spotlight_user_selections.columns.tolist())

print("\nSample rows:")
print(exploration_score_comparison.head(3))

Columns in exploration_score_comparison:
['user_id', 'exploration_video', 'exploration_creator', 'exploration_score', 'baseline_removed_video', 'baseline_removed_score', 'score_difference', 'relevance_cost']

Columns in spotlight_user_selections:
['user_id', 'video_id', 'predicted_preference', 'candidate_rank', 'author_id', 'creator_group', 'category_ids', 'interest_match_score', 'cf_percentile', 'interest_percentile', 'interest_eligible', 'exploration_priority_rank', 'eligible_user_opportunities', 'opportunity_coverage']

Sample rows:
   user_id  exploration_video  exploration_creator  exploration_score  baseline_removed_video  baseline_removed_score  \
0       14                770                  536           0.829176                    4268                0.945336   
1       23               7654                 3964           0.918251                    7248                0.994966   
2       36                870                 4120           0.933105                    5205  

In [70]:
# CELL 50: APPLY RELEVANCE GUARDRAIL

MAX_RELEVANCE_COST = 0.15

spotlight_constrained_selections = exploration_score_comparison[
    exploration_score_comparison["relevance_cost"] <= MAX_RELEVANCE_COST
].copy()

print("RELEVANCE-CONSTRAINED SPOTLIGHT")
print("=" * 60)

print("Maximum relevance cost:", MAX_RELEVANCE_COST)
print("Original exploration selections:", len(exploration_score_comparison))
print("Retained selections:", len(spotlight_constrained_selections))
print("Excluded selections:",
      len(exploration_score_comparison) - len(spotlight_constrained_selections))

print("Retention rate:",
      round(len(spotlight_constrained_selections) /
            len(exploration_score_comparison) * 100, 2), "%")

print("\nUnique users retained:",
      spotlight_constrained_selections["user_id"].nunique())

print("Unique Emerging creators retained:",
      spotlight_constrained_selections["exploration_creator"].nunique())

print("Unique videos retained:",
      spotlight_constrained_selections["exploration_video"].nunique())

print("\nRelevance cost statistics:")
print(spotlight_constrained_selections["relevance_cost"].describe())

# Validation
assert spotlight_constrained_selections["user_id"].nunique() == len(spotlight_constrained_selections)
assert spotlight_constrained_selections["relevance_cost"].max() <= MAX_RELEVANCE_COST

print("\n✅ RELEVANCE CONSTRAINT APPLIED SUCCESSFULLY!")

RELEVANCE-CONSTRAINED SPOTLIGHT
Maximum relevance cost: 0.15
Original exploration selections: 986
Retained selections: 804
Excluded selections: 182
Retention rate: 81.54 %

Unique users retained: 804
Unique Emerging creators retained: 107
Unique videos retained: 115

Relevance cost statistics:
count    804.000000
mean       0.062723
std        0.055176
min       -0.258830
25%        0.028142
50%        0.063246
75%        0.105505
max        0.149869
Name: relevance_cost, dtype: float64

✅ RELEVANCE CONSTRAINT APPLIED SUCCESSFULLY!


In [71]:
# CELL 51: CREATOR RETENTION UNDER RELEVANCE CONSTRAINT

# Creator-level counts before the constraint
unconstrained_creator_counts = (
    exploration_score_comparison
    .groupby("exploration_creator")
    .agg(
        unconstrained_users=("user_id", "nunique"),
        unconstrained_videos=("exploration_video", "nunique")
    )
)

# Creator-level counts after the constraint
constrained_creator_counts = (
    spotlight_constrained_selections
    .groupby("exploration_creator")
    .agg(
        constrained_users=("user_id", "nunique"),
        constrained_videos=("exploration_video", "nunique")
    )
)

# Combine both tables
creator_retention_analysis = (
    unconstrained_creator_counts
    .join(constrained_creator_counts, how="left")
    .fillna(0)
)

creator_retention_analysis["constrained_users"] = (
    creator_retention_analysis["constrained_users"].astype(int)
)

creator_retention_analysis["user_retention_rate"] = (
    creator_retention_analysis["constrained_users"] /
    creator_retention_analysis["unconstrained_users"]
)

creator_retention_analysis["users_removed"] = (
    creator_retention_analysis["unconstrained_users"] -
    creator_retention_analysis["constrained_users"]
)

print("CREATOR RETENTION ANALYSIS")
print("=" * 60)

print("Creators before constraint:",
      len(unconstrained_creator_counts))

print("Creators retained after constraint:",
      len(constrained_creator_counts))

print("Mean creator user-retention rate:",
      creator_retention_analysis["user_retention_rate"].mean())

print("\nCreator retention distribution:")
print(creator_retention_analysis["user_retention_rate"].describe())

print("\nCreators with zero retained selections:",
      (creator_retention_analysis["constrained_users"] == 0).sum())

print("\nCreators with 100% retention:",
      (creator_retention_analysis["user_retention_rate"] == 1).sum())

print("\nLowest-retention creators:")
print(
    creator_retention_analysis
    .sort_values(["user_retention_rate", "unconstrained_users"])
    .head(15)
    .round(4)
)

print("\nHighest-retention creators:")
print(
    creator_retention_analysis
    .sort_values(["user_retention_rate", "unconstrained_users"], ascending=False)
    .head(15)
    .round(4)
)

CREATOR RETENTION ANALYSIS
Creators before constraint: 117
Creators retained after constraint: 107
Mean creator user-retention rate: 0.7539712677875374

Creator retention distribution:
count    117.000000
mean       0.753971
std        0.334801
min        0.000000
25%        0.500000
50%        1.000000
75%        1.000000
max        1.000000
Name: user_retention_rate, dtype: float64

Creators with zero retained selections: 10

Creators with 100% retention: 62

Lowest-retention creators:
                     unconstrained_users  unconstrained_videos  constrained_users  constrained_videos  \
exploration_creator                                                                                     
2272                                   1                     1                  0                 0.0   
2469                                   1                     1                  0                 0.0   
7445                                   1                     1                  0      

In [73]:
# CELL 52: RETENTION BY CREATOR OPPORTUNITY SIZE

creator_retention_tiers = creator_retention_analysis.copy()

creator_retention_tiers["opportunity_tier"] = pd.cut(
    creator_retention_tiers["unconstrained_users"],
    bins=[0, 2, 5, 10, float("inf")],
    labels=["1-2 users", "3-5 users", "6-10 users", "11+ users"],
    include_lowest=True
)

retention_by_tier = (
    creator_retention_tiers
    .groupby("opportunity_tier", observed=False)
    .agg(
        creators=("unconstrained_users", "count"),
        original_selections=("unconstrained_users", "sum"),
        retained_selections=("constrained_users", "sum"),
        mean_creator_retention=("user_retention_rate", "mean"),
        median_creator_retention=("user_retention_rate", "median"),
        zero_retention_creators=(
            "constrained_users",
            lambda x: (x == 0).sum()
        )
    )
)

retention_by_tier["selection_retention_rate"] = (
    retention_by_tier["retained_selections"] /
    retention_by_tier["original_selections"]
)

print("RETENTION BY CREATOR OPPORTUNITY SIZE")
print("=" * 70)

print(retention_by_tier.round(4).to_string())

RETENTION BY CREATOR OPPORTUNITY SIZE
                  creators  original_selections  retained_selections  mean_creator_retention  median_creator_retention  zero_retention_creators  selection_retention_rate
opportunity_tier                                                                                                                                                         
1-2 users               19                   33                   21                  0.6053                      1.00                        6                    0.6364
3-5 users               39                  156                  107                  0.7013                      1.00                        4                    0.6859
6-10 users              32                  257                  213                  0.8245                      0.95                        0                    0.8288
11+ users               27                  540                  463                  0.8512                    

In [74]:
# CELL 53: OPPORTUNITY SIZE VS CREATOR RETENTION

from scipy.stats import spearmanr

creator_retention_valid = creator_retention_analysis.copy()

correlation, p_value = spearmanr(
    creator_retention_valid["unconstrained_users"],
    creator_retention_valid["user_retention_rate"]
)

print("OPPORTUNITY SIZE VS RETENTION")
print("=" * 60)

print("Number of creators:", len(creator_retention_valid))
print("Spearman correlation:", round(correlation, 4))
print("P-value:", round(p_value, 6))

print("\nInterpretation:")

if correlation > 0:
    print("Creator retention tends to increase with opportunity size.")
elif correlation < 0:
    print("Creator retention tends to decrease with opportunity size.")
else:
    print("No monotonic relationship detected.")

print("\nOpportunity size:")
print(creator_retention_valid["unconstrained_users"].describe())

print("\nCreator retention:")
print(creator_retention_valid["user_retention_rate"].describe())

OPPORTUNITY SIZE VS RETENTION
Number of creators: 117
Spearman correlation: 0.1127
P-value: 0.226445

Interpretation:
Creator retention tends to increase with opportunity size.

Opportunity size:
count    117.000000
mean       8.427350
std        7.863399
min        1.000000
25%        3.000000
50%        6.000000
75%       10.000000
max       39.000000
Name: unconstrained_users, dtype: float64

Creator retention:
count    117.000000
mean       0.753971
std        0.334801
min        0.000000
25%        0.500000
50%        1.000000
75%        1.000000
max        1.000000
Name: user_retention_rate, dtype: float64


In [76]:
# CELL 54: CONSTRUCT RELEVANCE-CONSTRAINED SPOTLIGHT RECOMMENDATIONS

# --------------------------------------------------
# 1. Copy original baseline recommendations
# --------------------------------------------------

spotlight_constrained_recommendations = baseline_recommendations.copy()


# --------------------------------------------------
# 2. Identify the lowest-scoring recommendation per user
# --------------------------------------------------

baseline_bottom10 = (
    baseline_recommendations
    .sort_values(
        ["user_id", "predicted_preference", "video_id"],
        ascending=[True, True, True]
    )
    .groupby("user_id")
    .head(1)
    [["user_id", "video_id", "predicted_preference"]]
    .rename(
        columns={
            "video_id": "baseline_removed_video",
            "predicted_preference": "baseline_removed_score"
        }
    )
)


# --------------------------------------------------
# 3. Identify users whose exploration passed the constraint
# --------------------------------------------------

users_with_exploration = spotlight_constrained_selections["user_id"].unique()


# --------------------------------------------------
# 4. Create exact user-video pairs to remove
# --------------------------------------------------

videos_to_remove = baseline_bottom10[
    baseline_bottom10["user_id"].isin(users_with_exploration)
][["user_id", "baseline_removed_video"]].rename(
    columns={"baseline_removed_video": "video_id"}
)


# --------------------------------------------------
# 5. Remove only the corresponding video for each user
# --------------------------------------------------

spotlight_constrained_recommendations = (
    spotlight_constrained_recommendations
    .merge(
        videos_to_remove.assign(remove_recommendation=True),
        on=["user_id", "video_id"],
        how="left"
    )
)

spotlight_constrained_recommendations = (
    spotlight_constrained_recommendations[
        spotlight_constrained_recommendations["remove_recommendation"].isna()
    ]
    .drop(columns="remove_recommendation")
    .copy()
)


# --------------------------------------------------
# 6. Insert the exploration recommendations
# --------------------------------------------------

exploration_to_insert = spotlight_constrained_selections[
    ["user_id", "exploration_video", "exploration_score"]
].rename(
    columns={
        "exploration_video": "video_id",
        "exploration_score": "predicted_preference"
    }
)

spotlight_constrained_recommendations = pd.concat(
    [
        spotlight_constrained_recommendations,
        exploration_to_insert
    ],
    ignore_index=True
)


# --------------------------------------------------
# 7. Validate recommendation counts and uniqueness
# --------------------------------------------------

recommendation_counts = (
    spotlight_constrained_recommendations
    .groupby("user_id")
    .size()
)

duplicate_recommendations = (
    spotlight_constrained_recommendations
    .duplicated(["user_id", "video_id"])
    .sum()
)


print("CONSTRAINED SPOTLIGHT RECOMMENDATIONS")
print("=" * 60)

print("Total recommendation rows:",
      len(spotlight_constrained_recommendations))

print("Unique users:",
      spotlight_constrained_recommendations["user_id"].nunique())

print("Users receiving exploration:",
      len(users_with_exploration))

print("Mean recommendations per user:",
      recommendation_counts.mean())

print("Minimum recommendations per user:",
      recommendation_counts.min())

print("Maximum recommendations per user:",
      recommendation_counts.max())

print("Duplicate user-video recommendations:",
      duplicate_recommendations)

print("Users with exactly 10 recommendations:",
      (recommendation_counts == 10).sum())


# --------------------------------------------------
# 8. Assertions
# --------------------------------------------------

assert len(spotlight_constrained_recommendations) == len(baseline_recommendations)

assert recommendation_counts.eq(10).all()

assert duplicate_recommendations == 0

assert spotlight_constrained_recommendations["user_id"].nunique() == \
       baseline_recommendations["user_id"].nunique()


print("\n✅ CONSTRAINED SPOTLIGHT LIST CONSTRUCTED SUCCESSFULLY!")

CONSTRAINED SPOTLIGHT RECOMMENDATIONS
Total recommendation rows: 14110
Unique users: 1411
Users receiving exploration: 804
Mean recommendations per user: 10.0
Minimum recommendations per user: 10
Maximum recommendations per user: 10
Duplicate user-video recommendations: 0
Users with exactly 10 recommendations: 1411

✅ CONSTRAINED SPOTLIGHT LIST CONSTRUCTED SUCCESSFULLY!


In [77]:
# CELL 55: BASELINE VS CONSTRAINED SPOTLIGHT GROUP EXPOSURE

# --------------------------------------------------
# 1. Prepare creator-group mapping
# --------------------------------------------------

video_group_map = video_creator_groups[
    ["video_id", "creator_group"]
].drop_duplicates("video_id")


# --------------------------------------------------
# 2. Calculate group exposure for each recommender
# --------------------------------------------------

def calculate_group_exposure(recommendations, model_name):

    exposure = recommendations.merge(
        video_group_map,
        on="video_id",
        how="left"
    )

    group_counts = (
        exposure["creator_group"]
        .value_counts()
        .rename_axis("creator_group")
        .reset_index(name="exposure_slots")
    )

    group_counts["model"] = model_name

    group_counts["exposure_share"] = (
        group_counts["exposure_slots"] /
        len(recommendations)
    )

    return group_counts


baseline_group_exposure_eval = calculate_group_exposure(
    baseline_recommendations,
    "Baseline"
)

spotlight_group_exposure_eval = calculate_group_exposure(
    spotlight_constrained_recommendations,
    "Constrained Spotlight"
)


# --------------------------------------------------
# 3. Combine results
# --------------------------------------------------

group_exposure_comparison = pd.concat(
    [
        baseline_group_exposure_eval,
        spotlight_group_exposure_eval
    ],
    ignore_index=True
)


print("BASELINE VS CONSTRAINED SPOTLIGHT GROUP EXPOSURE")
print("=" * 70)

print(
    group_exposure_comparison
    .pivot(
        index="creator_group",
        columns="model",
        values="exposure_slots"
    )
    .to_string()
)

print("\nExposure share:")
print(
    group_exposure_comparison
    .pivot(
        index="creator_group",
        columns="model",
        values="exposure_share"
    )
    .round(4)
    .to_string()
)


# --------------------------------------------------
# 4. Calculate exposure-share change
# --------------------------------------------------

exposure_share_comparison = (
    group_exposure_comparison
    .pivot(
        index="creator_group",
        columns="model",
        values="exposure_share"
    )
)

exposure_share_comparison["change"] = (
    exposure_share_comparison["Constrained Spotlight"] -
    exposure_share_comparison["Baseline"]
)

exposure_share_comparison["change_percentage_points"] = (
    exposure_share_comparison["change"] * 100
)

print("\nEXPOSURE SHARE CHANGE")
print("=" * 70)

print(
    exposure_share_comparison[
        ["Baseline", "Constrained Spotlight", "change_percentage_points"]
    ]
    .round(4)
    .to_string()
)


# --------------------------------------------------
# 5. Validate
# --------------------------------------------------

assert group_exposure_comparison["exposure_slots"].sum() == (
    len(baseline_recommendations) +
    len(spotlight_constrained_recommendations)
)

print("\n✅ GROUP EXPOSURE COMPARISON COMPLETED!")

BASELINE VS CONSTRAINED SPOTLIGHT GROUP EXPOSURE
model          Baseline  Constrained Spotlight
creator_group                                 
Emerging           3678                   4297
Established       10432                   9813

Exposure share:
model          Baseline  Constrained Spotlight
creator_group                                 
Emerging         0.2607                 0.3045
Established      0.7393                 0.6955

EXPOSURE SHARE CHANGE
model          Baseline  Constrained Spotlight  change_percentage_points
creator_group                                                           
Emerging         0.2607                 0.3045                     4.387
Established      0.7393                 0.6955                    -4.387

✅ GROUP EXPOSURE COMPARISON COMPLETED!


In [78]:
# CELL 56: BASELINE VS SPOTLIGHT CREATOR-LEVEL EXPOSURE

# --------------------------------------------------
# 1. Create creator mapping
# --------------------------------------------------

creator_map = video_creator_groups[
    ["video_id", "author_id", "creator_group"]
].drop_duplicates("video_id")


# --------------------------------------------------
# 2. Calculate creator-level exposure
# --------------------------------------------------

def calculate_creator_exposure(recommendations, model_name):

    exposure = recommendations.merge(
        creator_map,
        on="video_id",
        how="left"
    )

    creator_metrics = (
        exposure
        .groupby(["author_id", "creator_group"])
        .agg(
            exposure_slots=("video_id", "count"),
            unique_videos=("video_id", "nunique"),
            exposed_users=("user_id", "nunique")
        )
        .reset_index()
    )

    creator_metrics["model"] = model_name

    return creator_metrics


baseline_creator_exposure_eval = calculate_creator_exposure(
    baseline_recommendations,
    "Baseline"
)

spotlight_creator_exposure_eval = calculate_creator_exposure(
    spotlight_constrained_recommendations,
    "Constrained Spotlight"
)


# --------------------------------------------------
# 3. Combine creator exposure tables
# --------------------------------------------------

creator_exposure_comparison = pd.merge(
    baseline_creator_exposure_eval,
    spotlight_creator_exposure_eval,
    on=["author_id", "creator_group"],
    how="outer",
    suffixes=("_baseline", "_spotlight")
).fillna(0)


# --------------------------------------------------
# 4. Calculate changes
# --------------------------------------------------

creator_exposure_comparison["slot_change"] = (
    creator_exposure_comparison["exposure_slots_spotlight"] -
    creator_exposure_comparison["exposure_slots_baseline"]
)

creator_exposure_comparison["user_change"] = (
    creator_exposure_comparison["exposed_users_spotlight"] -
    creator_exposure_comparison["exposed_users_baseline"]
)


# --------------------------------------------------
# 5. Emerging creator analysis
# --------------------------------------------------

emerging_creator_changes = creator_exposure_comparison[
    creator_exposure_comparison["creator_group"] == "Emerging"
].copy()

print("INDIVIDUAL EMERGING CREATOR EXPOSURE")
print("=" * 70)

print("Emerging creators analysed:",
      len(emerging_creator_changes))

print("Creators gaining exposure:",
      (emerging_creator_changes["slot_change"] > 0).sum())

print("Creators losing exposure:",
      (emerging_creator_changes["slot_change"] < 0).sum())

print("Creators with unchanged exposure:",
      (emerging_creator_changes["slot_change"] == 0).sum())

print("\nTotal Emerging slot change:",
      emerging_creator_changes["slot_change"].sum())

print("\nTop 15 Emerging creators gaining exposure:")
print(
    emerging_creator_changes
    .sort_values("slot_change", ascending=False)
    [[
        "author_id",
        "exposure_slots_baseline",
        "exposure_slots_spotlight",
        "slot_change",
        "exposed_users_baseline",
        "exposed_users_spotlight",
        "user_change"
    ]]
    .head(15)
    .to_string(index=False)
)

print("\nTop 15 Emerging creators losing exposure:")
print(
    emerging_creator_changes
    .sort_values("slot_change")
    [[
        "author_id",
        "exposure_slots_baseline",
        "exposure_slots_spotlight",
        "slot_change",
        "exposed_users_baseline",
        "exposed_users_spotlight",
        "user_change"
    ]]
    .head(15)
    .to_string(index=False)
)


# --------------------------------------------------
# 6. Validation
# --------------------------------------------------

assert creator_exposure_comparison["slot_change"].sum() == 0

print("\n✅ INDIVIDUAL CREATOR EXPOSURE COMPARISON COMPLETED!")

INDIVIDUAL EMERGING CREATOR EXPOSURE
Emerging creators analysed: 320
Creators gaining exposure: 107
Creators losing exposure: 99
Creators with unchanged exposure: 114

Total Emerging slot change: 619.0

Top 15 Emerging creators gaining exposure:
 author_id  exposure_slots_baseline  exposure_slots_spotlight  slot_change  exposed_users_baseline  exposed_users_spotlight  user_change
      4331                      3.0                      40.0         37.0                     3.0                     40.0         37.0
       536                      0.0                      30.0         30.0                     0.0                     30.0         30.0
      3159                      0.0                      29.0         29.0                     0.0                     29.0         29.0
      6684                      5.0                      33.0         28.0                     5.0                     33.0         28.0
      1126                      1.0                      29.0        

In [79]:
# CELL 57: FULL CREATOR COVERAGE & ZERO-EXPOSURE COMPARISON

# --------------------------------------------------
# 1. Build the complete creator universe
# --------------------------------------------------

creator_universe = (
    video_creator_groups[
        ["author_id", "creator_group"]
    ]
    .drop_duplicates()
    .dropna(subset=["author_id"])
)

print("Total creators in classified universe:",
      len(creator_universe))


# --------------------------------------------------
# 2. Extract baseline creator exposure
# --------------------------------------------------

baseline_full = baseline_creator_exposure_eval[
    ["author_id", "exposure_slots", "exposed_users"]
].rename(columns={
    "exposure_slots": "baseline_slots",
    "exposed_users": "baseline_users"
})


# --------------------------------------------------
# 3. Extract Spotlight creator exposure
# --------------------------------------------------

spotlight_full = spotlight_creator_exposure_eval[
    ["author_id", "exposure_slots", "exposed_users"]
].rename(columns={
    "exposure_slots": "spotlight_slots",
    "exposed_users": "spotlight_users"
})


# --------------------------------------------------
# 4. Merge with complete creator universe
# --------------------------------------------------

full_creator_coverage = (
    creator_universe
    .merge(baseline_full, on="author_id", how="left")
    .merge(spotlight_full, on="author_id", how="left")
)

exposure_columns = [
    "baseline_slots",
    "baseline_users",
    "spotlight_slots",
    "spotlight_users"
]

full_creator_coverage[exposure_columns] = (
    full_creator_coverage[exposure_columns]
    .fillna(0)
)


# --------------------------------------------------
# 5. Zero-exposure flags
# --------------------------------------------------

full_creator_coverage["baseline_exposed"] = (
    full_creator_coverage["baseline_slots"] > 0
)

full_creator_coverage["spotlight_exposed"] = (
    full_creator_coverage["spotlight_slots"] > 0
)


# --------------------------------------------------
# 6. Group-level creator coverage
# --------------------------------------------------

creator_coverage_summary = (
    full_creator_coverage
    .groupby("creator_group")
    .agg(
        total_creators=("author_id", "nunique"),
        baseline_exposed_creators=("baseline_exposed", "sum"),
        spotlight_exposed_creators=("spotlight_exposed", "sum"),
        baseline_slots=("baseline_slots", "sum"),
        spotlight_slots=("spotlight_slots", "sum")
    )
)

creator_coverage_summary["baseline_coverage_rate"] = (
    creator_coverage_summary["baseline_exposed_creators"] /
    creator_coverage_summary["total_creators"]
)

creator_coverage_summary["spotlight_coverage_rate"] = (
    creator_coverage_summary["spotlight_exposed_creators"] /
    creator_coverage_summary["total_creators"]
)

creator_coverage_summary["baseline_zero_exposure"] = (
    creator_coverage_summary["total_creators"] -
    creator_coverage_summary["baseline_exposed_creators"]
)

creator_coverage_summary["spotlight_zero_exposure"] = (
    creator_coverage_summary["total_creators"] -
    creator_coverage_summary["spotlight_exposed_creators"]
)

creator_coverage_summary["coverage_change_pp"] = (
    creator_coverage_summary["spotlight_coverage_rate"] -
    creator_coverage_summary["baseline_coverage_rate"]
) * 100


# --------------------------------------------------
# 7. Display results
# --------------------------------------------------

print("\nFULL CREATOR COVERAGE COMPARISON")
print("=" * 80)

print(
    creator_coverage_summary
    .round(4)
    .to_string()
)


# --------------------------------------------------
# 8. Emerging-specific zero exposure
# --------------------------------------------------

emerging_full = full_creator_coverage[
    full_creator_coverage["creator_group"] == "Emerging"
]

print("\nEMERGING CREATOR ZERO-EXPOSURE ANALYSIS")
print("=" * 80)

print("Total Emerging creators:",
      len(emerging_full))

print("Baseline zero-exposure creators:",
      (~emerging_full["baseline_exposed"]).sum())

print("Spotlight zero-exposure creators:",
      (~emerging_full["spotlight_exposed"]).sum())

print("Creators newly exposed by Spotlight:",
      (
          (~emerging_full["baseline_exposed"]) &
          emerging_full["spotlight_exposed"]
      ).sum())

print("Creators losing all exposure:",
      (
          emerging_full["baseline_exposed"] &
          (~emerging_full["spotlight_exposed"])
      ).sum())


# --------------------------------------------------
# 9. Validation
# --------------------------------------------------

assert full_creator_coverage["author_id"].is_unique

assert full_creator_coverage["baseline_slots"].sum() == \
       baseline_recommendations.shape[0]

assert full_creator_coverage["spotlight_slots"].sum() == \
       spotlight_constrained_recommendations.shape[0]

print("\n✅ FULL CREATOR COVERAGE ANALYSIS COMPLETED!")

Total creators in classified universe: 2031

FULL CREATOR COVERAGE COMPARISON
               total_creators  baseline_exposed_creators  spotlight_exposed_creators  baseline_slots  spotlight_slots  baseline_coverage_rate  spotlight_coverage_rate  baseline_zero_exposure  spotlight_zero_exposure  coverage_change_pp
creator_group                                                                                                                                                                                                                              
Emerging                  505                        262                         317          3678.0           4297.0                  0.5188                   0.6277                     243                      188             10.8911
Established              1515                        624                         605         10432.0           9813.0                  0.4119                   0.3993                     891                      91

In [81]:
# CELL 58: SELF-CONTAINED EMERGING CREATOR DENOMINATOR AUDIT

print("EMERGING CREATOR DENOMINATOR AUDIT")
print("=" * 70)

print("\nvideo_creator_groups columns:")
print(video_creator_groups.columns.tolist())

print("\ncreator_popularity columns:")
print(creator_popularity.columns.tolist())


# Unique creator counts from the video mapping
video_map_counts = (
    video_creator_groups
    .drop_duplicates("author_id")
    .groupby("creator_group")["author_id"]
    .nunique()
)

print("\nCreator counts from video_creator_groups:")
print(video_map_counts)


# Unique creator counts from creator popularity classification
popularity_counts = (
    creator_popularity
    .groupby("creator_group")["author_id"]
    .nunique()
)

print("\nCreator counts from creator_popularity:")
print(popularity_counts)


# Compare Emerging creator IDs directly
emerging_video_ids = set(
    video_creator_groups.loc[
        video_creator_groups["creator_group"] == "Emerging",
        "author_id"
    ].dropna().unique()
)

emerging_popularity_ids = set(
    creator_popularity.loc[
        creator_popularity["creator_group"] == "Emerging",
        "author_id"
    ].dropna().unique()
)

print("\nEmerging creators in video_creator_groups:",
      len(emerging_video_ids))

print("Emerging creators in creator_popularity:",
      len(emerging_popularity_ids))

print("\nIDs present only in video_creator_groups:")
print(sorted(emerging_video_ids - emerging_popularity_ids))

print("\nIDs present only in creator_popularity:")
print(sorted(emerging_popularity_ids - emerging_video_ids))

print("\n✅ DENOMINATOR AUDIT COMPLETED!")

EMERGING CREATOR DENOMINATOR AUDIT

video_creator_groups columns:
['video_id', 'author_id', 'average_video_daily_shows', 'creator_group']

creator_popularity columns:
['author_id', 'eligible_video_count', 'average_video_daily_shows', 'total_observed_video_days', 'creator_group']

Creator counts from video_creator_groups:
creator_group
Emerging        505
Established    1515
Name: author_id, dtype: int64

Creator counts from creator_popularity:
creator_group
Emerging        505
Established    1515
Name: author_id, dtype: int64

Emerging creators in video_creator_groups: 505
Emerging creators in creator_popularity: 505

IDs present only in video_creator_groups:
[]

IDs present only in creator_popularity:
[]

✅ DENOMINATOR AUDIT COMPLETED!


In [82]:
# CELL 58: EMERGING CREATOR EXPOSURE CONCENTRATION

import numpy as np
import pandas as pd


# --------------------------------------------------
# 1. GINI COEFFICIENT
# --------------------------------------------------

def calculate_gini(values):

    values = np.asarray(values, dtype=float)

    if len(values) == 0:
        return np.nan

    values = np.sort(values)

    if values.sum() == 0:
        return 0.0

    n = len(values)

    cumulative = np.cumsum(values)

    gini = (
        2 * np.sum((np.arange(1, n + 1) * values))
        / (n * values.sum())
    ) - ((n + 1) / n)

    return gini


# --------------------------------------------------
# 2. ANALYSIS FUNCTION
# --------------------------------------------------

def calculate_concentration(values):

    values = pd.Series(values).astype(float)

    total_exposure = values.sum()

    sorted_values = values.sort_values(ascending=False)

    results = {
        "total_creators": len(values),
        "total_exposure_slots": total_exposure,
        "mean_exposure": values.mean(),
        "median_exposure": values.median(),
        "std_exposure": values.std(),
        "zero_exposure_creators": (values == 0).sum(),
        "gini": calculate_gini(values)
    }

    for percentage in [1, 5, 10, 20]:

        n_creators = max(
            1,
            int(np.ceil(len(values) * percentage / 100))
        )

        top_exposure = sorted_values.head(n_creators).sum()

        results[f"top_{percentage}_creator_count"] = n_creators

        results[f"top_{percentage}_exposure_share"] = (
            top_exposure / total_exposure
            if total_exposure > 0
            else 0
        )

    return results


# --------------------------------------------------
# 3. EMERGING CREATOR POPULATION
# --------------------------------------------------

emerging_concentration_data = full_creator_coverage[
    full_creator_coverage["creator_group"] == "Emerging"
].copy()


# --------------------------------------------------
# 4. BASELINE CONCENTRATION
# --------------------------------------------------

baseline_concentration = calculate_concentration(
    emerging_concentration_data["baseline_slots"]
)


# --------------------------------------------------
# 5. SPOTLIGHT CONCENTRATION
# --------------------------------------------------

spotlight_concentration = calculate_concentration(
    emerging_concentration_data["spotlight_slots"]
)


# --------------------------------------------------
# 6. COMPARISON TABLE
# --------------------------------------------------

emerging_concentration_comparison = pd.DataFrame({
    "Baseline": baseline_concentration,
    "Constrained Spotlight": spotlight_concentration
})


print("EMERGING CREATOR EXPOSURE CONCENTRATION")
print("=" * 80)

print(
    emerging_concentration_comparison
    .round(4)
    .to_string()
)


# --------------------------------------------------
# 7. GINI CHANGE
# --------------------------------------------------

gini_change = (
    spotlight_concentration["gini"] -
    baseline_concentration["gini"]
)

print("\nGINI CHANGE")
print("=" * 80)

print("Baseline Gini:",
      round(baseline_concentration["gini"], 4))

print("Spotlight Gini:",
      round(spotlight_concentration["gini"], 4))

print("Change:",
      round(gini_change, 4))

if gini_change < 0:
    print("Interpretation: Exposure inequality decreased.")
elif gini_change > 0:
    print("Interpretation: Exposure inequality increased.")
else:
    print("Interpretation: Exposure inequality remained unchanged.")


print("\n✅ EMERGING EXPOSURE CONCENTRATION ANALYSIS COMPLETED!")

EMERGING CREATOR EXPOSURE CONCENTRATION
                         Baseline  Constrained Spotlight
total_creators           505.0000               505.0000
total_exposure_slots    3678.0000              4297.0000
mean_exposure              7.2832                 8.5089
median_exposure            1.0000                 3.0000
std_exposure              14.7176                14.4159
zero_exposure_creators   243.0000               188.0000
gini                       0.7845                 0.7117
top_1_creator_count        6.0000                 6.0000
top_1_exposure_share       0.1465                 0.1233
top_5_creator_count       26.0000                26.0000
top_5_exposure_share       0.4048                 0.3391
top_10_creator_count      51.0000                51.0000
top_10_exposure_share      0.5916                 0.5043
top_20_creator_count     101.0000               101.0000
top_20_exposure_share      0.8219                 0.7203

GINI CHANGE
Baseline Gini: 0.7845
Spotlight Gin

In [83]:
# CELL 59: EXPOSURE CONCENTRATION COMPARISON ACROSS BOTH GROUPS

groups = ["Emerging", "Established"]

concentration_results = []

for group in groups:

    group_data = full_creator_coverage[
        full_creator_coverage["creator_group"] == group
    ]

    baseline_metrics = calculate_concentration(
        group_data["baseline_slots"]
    )

    spotlight_metrics = calculate_concentration(
        group_data["spotlight_slots"]
    )

    concentration_results.append({
        "creator_group": group,

        "baseline_gini": baseline_metrics["gini"],
        "spotlight_gini": spotlight_metrics["gini"],
        "gini_change": (
            spotlight_metrics["gini"] -
            baseline_metrics["gini"]
        ),

        "baseline_zero_exposure": (
            baseline_metrics["zero_exposure_creators"]
        ),
        "spotlight_zero_exposure": (
            spotlight_metrics["zero_exposure_creators"]
        ),

        "baseline_top10_share": (
            baseline_metrics["top_10_exposure_share"]
        ),
        "spotlight_top10_share": (
            spotlight_metrics["top_10_exposure_share"]
        ),

        "baseline_top20_share": (
            baseline_metrics["top_20_exposure_share"]
        ),
        "spotlight_top20_share": (
            spotlight_metrics["top_20_exposure_share"]
        )
    })


concentration_group_comparison = pd.DataFrame(
    concentration_results
)

print("EXPOSURE CONCENTRATION: EMERGING VS ESTABLISHED")
print("=" * 100)

print(
    concentration_group_comparison
    .round(4)
    .to_string(index=False)
)


print("\nGINI CHANGE INTERPRETATION")
print("=" * 100)

for _, row in concentration_group_comparison.iterrows():

    group = row["creator_group"]
    change = row["gini_change"]

    print(f"\n{group}:")
    print(f"Baseline Gini: {row['baseline_gini']:.4f}")
    print(f"Spotlight Gini: {row['spotlight_gini']:.4f}")
    print(f"Gini change: {change:.4f}")

    if change < 0:
        print("Exposure concentration decreased.")
    elif change > 0:
        print("Exposure concentration increased.")
    else:
        print("Exposure concentration remained unchanged.")


print("\n✅ BOTH-GROUP CONCENTRATION COMPARISON COMPLETED!")

EXPOSURE CONCENTRATION: EMERGING VS ESTABLISHED
creator_group  baseline_gini  spotlight_gini  gini_change  baseline_zero_exposure  spotlight_zero_exposure  baseline_top10_share  spotlight_top10_share  baseline_top20_share  spotlight_top20_share
     Emerging         0.7845          0.7117      -0.0728                     243                      188                0.5916                 0.5043                0.8219                 0.7203
  Established         0.8659          0.8706       0.0047                     891                      910                0.7604                 0.7712                0.9250                 0.9309

GINI CHANGE INTERPRETATION

Emerging:
Baseline Gini: 0.7845
Spotlight Gini: 0.7117
Gini change: -0.0728
Exposure concentration decreased.

Established:
Baseline Gini: 0.8659
Spotlight Gini: 0.8706
Gini change: 0.0047
Exposure concentration increased.

✅ BOTH-GROUP CONCENTRATION COMPARISON COMPLETED!


In [84]:
# CELL 60: PREPARE RANKING-QUALITY EVALUATION

import pandas as pd
import numpy as np

# --------------------------------------------------
# 1. Load the fixed test split
# --------------------------------------------------

testing_interactions = pd.read_csv(
    "../processed_data/testing_interactions.csv"
)

print("TESTING INTERACTIONS")
print("=" * 70)

print("Shape:", testing_interactions.shape)
print("Columns:", testing_interactions.columns.tolist())

print("\nSample:")
display(testing_interactions.head())


# --------------------------------------------------
# 2. Verify recommendation list structure
# --------------------------------------------------

print("\nBASELINE RECOMMENDATIONS")
print("=" * 70)

print("Shape:", baseline_recommendations.shape)
print("Columns:", baseline_recommendations.columns.tolist())

display(baseline_recommendations.head())


print("\nCONSTRAINED SPOTLIGHT RECOMMENDATIONS")
print("=" * 70)

print("Shape:", spotlight_constrained_recommendations.shape)
print("Columns:", spotlight_constrained_recommendations.columns.tolist())

display(spotlight_constrained_recommendations.head())


# --------------------------------------------------
# 3. Validate recommendation list lengths
# --------------------------------------------------

baseline_counts = baseline_recommendations.groupby("user_id").size()

spotlight_counts = (
    spotlight_constrained_recommendations
    .groupby("user_id")
    .size()
)

print("\nRECOMMENDATION COUNT VALIDATION")
print("=" * 70)

print("Baseline users:", baseline_counts.nunique())
print("Baseline list length min/max:",
      baseline_counts.min(), baseline_counts.max())

print("Spotlight users:", spotlight_counts.nunique())
print("Spotlight list length min/max:",
      spotlight_counts.min(), spotlight_counts.max())


# --------------------------------------------------
# 4. Validate user overlap
# --------------------------------------------------

baseline_users = set(baseline_recommendations["user_id"])

spotlight_users = set(
    spotlight_constrained_recommendations["user_id"]
)

test_users = set(testing_interactions["user_id"])

print("\nUSER OVERLAP")
print("=" * 70)

print("Baseline recommendation users:", len(baseline_users))
print("Spotlight recommendation users:", len(spotlight_users))
print("Test users:", len(test_users))

print("Baseline/Spotlight identical users:",
      baseline_users == spotlight_users)

print("Recommendation/test user overlap:",
      len(baseline_users & test_users))


print("\n✅ RANKING EVALUATION DATA PREPARATION COMPLETED!")

TESTING INTERACTIONS
Shape: (107196, 9)
Columns: ['user_id', 'video_id', 'play_duration', 'video_duration', 'time', 'date', 'timestamp', 'watch_ratio', 'interaction_date']

Sample:


,user_id,video_id,play_duration,video_duration,time,date,timestamp,watch_ratio,interaction_date
0,14,1564,8509,6321,2020-08-30 01:56:00.972,20200830.0,1.598724e+09,1.346148,2020-08-30
1,14,3266,4505,19321,2020-08-30 02:04:17.678,20200830.0,1.598724e+09,0.233166,2020-08-30
2,14,6307,8260,32417,2020-08-30 02:18:14.314,20200830.0,1.598725e+09,0.254805,2020-08-30
3,14,3141,4487,8000,2020-08-30 02:20:39.806,20200830.0,1.598725e+09,0.560875,2020-08-30
4,14,7313,4539,8534,2020-08-30 02:37:37.662,20200830.0,1.598726e+09,0.531873,2020-08-30



BASELINE RECOMMENDATIONS
Shape: (14110, 3)
Columns: ['user_id', 'video_id', 'predicted_preference']


,user_id,video_id,predicted_preference
0,14,3992,0.997153
1,14,7263,0.995710
2,14,835,0.990655
3,14,6177,0.990421
4,14,6086,0.966525



CONSTRAINED SPOTLIGHT RECOMMENDATIONS
Shape: (14110, 3)
Columns: ['user_id', 'video_id', 'predicted_preference']


,user_id,video_id,predicted_preference
0,14,3992,0.997153
1,14,7263,0.995710
2,14,835,0.990655
3,14,6177,0.990421
4,14,6086,0.966525



RECOMMENDATION COUNT VALIDATION
Baseline users: 1
Baseline list length min/max: 10 10
Spotlight users: 1
Spotlight list length min/max: 10 10

USER OVERLAP
Baseline recommendation users: 1411
Spotlight recommendation users: 1411
Test users: 1411
Baseline/Spotlight identical users: True
Recommendation/test user overlap: 1411

✅ RANKING EVALUATION DATA PREPARATION COMPLETED!


In [85]:
# CELL 61: CONSTRUCT WARM-START GROUND TRUTH

# --------------------------------------------------
# 1. Identify videos available to the models
# --------------------------------------------------

warm_start_videos = set(
    interaction_matrix.columns
)

print("Videos in training catalogue:",
      len(warm_start_videos))


# --------------------------------------------------
# 2. Filter test interactions to warm-start videos
# --------------------------------------------------

warm_start_test = testing_interactions[
    testing_interactions["video_id"].isin(warm_start_videos)
].copy()

cold_start_test = testing_interactions[
    ~testing_interactions["video_id"].isin(warm_start_videos)
].copy()


# --------------------------------------------------
# 3. Construct ground truth
# --------------------------------------------------

ground_truth = (
    warm_start_test
    .groupby("user_id")["video_id"]
    .apply(set)
    .to_dict()
)


# --------------------------------------------------
# 4. Evaluation validation
# --------------------------------------------------

recommendation_users = set(
    baseline_recommendations["user_id"].unique()
)

ground_truth_users = set(ground_truth.keys())

users_without_ground_truth = (
    recommendation_users - ground_truth_users
)

users_without_recommendations = (
    ground_truth_users - recommendation_users
)


# --------------------------------------------------
# 5. Print results
# --------------------------------------------------

print("\nWARM-START EVALUATION DATA")
print("=" * 70)

print("Total test interactions:",
      len(testing_interactions))

print("Warm-start test interactions:",
      len(warm_start_test))

print("Cold-start test interactions:",
      len(cold_start_test))

print("Warm-start test users:",
      len(ground_truth_users))

print("Users with no warm-start test interactions:",
      len(users_without_ground_truth))

print("Users missing recommendations:",
      len(users_without_recommendations))

print("Total unique warm-start relevant videos:",
      warm_start_test["video_id"].nunique())

print("Mean relevant videos per user:",
      round(
          warm_start_test.groupby("user_id")["video_id"].nunique().mean(),
          2
      ))


# --------------------------------------------------
# 6. Assertions
# --------------------------------------------------

assert len(warm_start_test) + len(cold_start_test) == \
       len(testing_interactions)

assert len(ground_truth) > 0

assert warm_start_test["video_id"].isin(
    warm_start_videos
).all()

assert not cold_start_test["video_id"].isin(
    warm_start_videos
).any()

print("\n✅ WARM-START GROUND TRUTH CREATED!")

Videos in training catalogue: 3288

WARM-START EVALUATION DATA
Total test interactions: 107196
Warm-start test interactions: 55641
Cold-start test interactions: 51555
Warm-start test users: 1411
Users with no warm-start test interactions: 0
Users missing recommendations: 0
Total unique warm-start relevant videos: 1123
Mean relevant videos per user: 39.43

✅ WARM-START GROUND TRUTH CREATED!


In [86]:
# CELL 62: BASELINE VS SPOTLIGHT RANKING QUALITY

import numpy as np
import pandas as pd


# --------------------------------------------------
# 1. METRIC FUNCTIONS
# --------------------------------------------------

def precision_at_k(recommended, relevant, k=10):

    recommended_k = recommended[:k]

    hits = sum(
        video in relevant
        for video in recommended_k
    )

    return hits / k


def recall_at_k(recommended, relevant, k=10):

    if len(relevant) == 0:
        return 0.0

    recommended_k = recommended[:k]

    hits = sum(
        video in relevant
        for video in recommended_k
    )

    return hits / len(relevant)


def ndcg_at_k(recommended, relevant, k=10):

    recommended_k = recommended[:k]

    relevance = np.array([
        1 if video in relevant else 0
        for video in recommended_k
    ])

    discounts = np.log2(
        np.arange(2, len(relevance) + 2)
    )

    dcg = np.sum(relevance / discounts)

    ideal_hits = min(len(relevant), k)

    ideal_relevance = np.ones(ideal_hits)

    ideal_discounts = np.log2(
        np.arange(2, ideal_hits + 2)
    )

    idcg = np.sum(
        ideal_relevance / ideal_discounts
    )

    return dcg / idcg if idcg > 0 else 0.0


def hit_rate_at_k(recommended, relevant, k=10):

    recommended_k = recommended[:k]

    return int(
        any(video in relevant for video in recommended_k)
    )


# --------------------------------------------------
# 2. EVALUATION FUNCTION
# --------------------------------------------------

def evaluate_recommendations(
    recommendations,
    ground_truth,
    k=10,
    model_name="Model"
):

    user_metrics = []

    # Preserve the recommendation ranking order
    recommendations = recommendations.sort_values(
        ["user_id", "predicted_preference"],
        ascending=[True, False],
        kind="mergesort"
    )

    for user_id, group in recommendations.groupby("user_id"):

        recommended_videos = group["video_id"].tolist()

        relevant_videos = ground_truth.get(
            user_id, set()
        )

        user_metrics.append({
            "user_id": user_id,
            "precision": precision_at_k(
                recommended_videos, relevant_videos, k
            ),
            "recall": recall_at_k(
                recommended_videos, relevant_videos, k
            ),
            "ndcg": ndcg_at_k(
                recommended_videos, relevant_videos, k
            ),
            "hit_rate": hit_rate_at_k(
                recommended_videos, relevant_videos, k
            )
        })

    user_metrics_df = pd.DataFrame(user_metrics)

    summary = pd.DataFrame({
        "Model": [model_name],
        "Precision@10": [user_metrics_df["precision"].mean()],
        "Recall@10": [user_metrics_df["recall"].mean()],
        "NDCG@10": [user_metrics_df["ndcg"].mean()],
        "HitRate@10": [user_metrics_df["hit_rate"].mean()]
    })

    return summary, user_metrics_df


# --------------------------------------------------
# 3. BASELINE EVALUATION
# --------------------------------------------------

baseline_ranking_summary, baseline_user_metrics = (
    evaluate_recommendations(
        baseline_recommendations,
        ground_truth,
        k=10,
        model_name="Baseline"
    )
)


# --------------------------------------------------
# 4. SPOTLIGHT EVALUATION
# --------------------------------------------------

spotlight_ranking_summary, spotlight_user_metrics = (
    evaluate_recommendations(
        spotlight_constrained_recommendations,
        ground_truth,
        k=10,
        model_name="Constrained Spotlight"
    )
)


# --------------------------------------------------
# 5. COMBINE RESULTS
# --------------------------------------------------

ranking_quality_comparison = pd.concat(
    [
        baseline_ranking_summary,
        spotlight_ranking_summary
    ],
    ignore_index=True
)


print("RANKING QUALITY COMPARISON")
print("=" * 80)

print(
    ranking_quality_comparison
    .set_index("Model")
    .round(6)
    .to_string()
)


# --------------------------------------------------
# 6. ABSOLUTE AND RELATIVE CHANGE
# --------------------------------------------------

baseline_values = (
    baseline_ranking_summary
    .iloc[0]
    .drop("Model")
    .astype(float)
)

spotlight_values = (
    spotlight_ranking_summary
    .iloc[0]
    .drop("Model")
    .astype(float)
)

metric_changes = pd.DataFrame({
    "Baseline": baseline_values,
    "Spotlight": spotlight_values,
    "Absolute_Change": spotlight_values - baseline_values
})

metric_changes["Relative_Change_%"] = (
    (spotlight_values - baseline_values) /
    baseline_values.replace(0, np.nan)
) * 100


print("\nMETRIC CHANGES")
print("=" * 80)

print(
    metric_changes
    .round(6)
    .to_string()
)


# --------------------------------------------------
# 7. VALIDATION
# --------------------------------------------------

assert len(baseline_user_metrics) == 1411
assert len(spotlight_user_metrics) == 1411

assert baseline_user_metrics["user_id"].nunique() == 1411
assert spotlight_user_metrics["user_id"].nunique() == 1411

print("\n✅ RANKING QUALITY EVALUATION COMPLETED!")

RANKING QUALITY COMPARISON
                       Precision@10  Recall@10   NDCG@10  HitRate@10
Model                                                               
Baseline                   0.139405   0.037370  0.139410    0.696669
Constrained Spotlight      0.136003   0.036405  0.137262    0.691708

METRIC CHANGES
              Baseline  Spotlight  Absolute_Change  Relative_Change_%
Precision@10  0.139405   0.136003        -0.003402          -2.440264
Recall@10     0.037370   0.036405        -0.000965          -2.581220
NDCG@10       0.139410   0.137262        -0.002148          -1.540603
HitRate@10    0.696669   0.691708        -0.004961          -0.712106

✅ RANKING QUALITY EVALUATION COMPLETED!


In [87]:
# CELL 63: PAIRED BOOTSTRAP FOR RANKING METRICS

import numpy as np
import pandas as pd


# --------------------------------------------------
# 1. ALIGN USER-LEVEL RESULTS
# --------------------------------------------------

metrics = [
    "precision",
    "recall",
    "ndcg",
    "hit_rate"
]

baseline_paired = (
    baseline_user_metrics
    .set_index("user_id")
    .sort_index()
)

spotlight_paired = (
    spotlight_user_metrics
    .set_index("user_id")
    .sort_index()
)

assert baseline_paired.index.equals(
    spotlight_paired.index
)

n_users = len(baseline_paired)

print("Users in paired analysis:", n_users)


# --------------------------------------------------
# 2. CALCULATE OBSERVED DIFFERENCES
# --------------------------------------------------

observed_differences = {}

for metric in metrics:

    observed_differences[metric] = (
        spotlight_paired[metric].mean()
        -
        baseline_paired[metric].mean()
    )


# --------------------------------------------------
# 3. PAIRED BOOTSTRAP
# --------------------------------------------------

N_BOOTSTRAP = 10000
RANDOM_STATE = 42

rng = np.random.default_rng(RANDOM_STATE)

bootstrap_differences = {
    metric: np.zeros(N_BOOTSTRAP)
    for metric in metrics
}

for b in range(N_BOOTSTRAP):

    sampled_indices = rng.integers(
        0,
        n_users,
        size=n_users
    )

    for metric in metrics:

        baseline_sample = (
            baseline_paired[metric]
            .to_numpy()[sampled_indices]
        )

        spotlight_sample = (
            spotlight_paired[metric]
            .to_numpy()[sampled_indices]
        )

        bootstrap_differences[metric][b] = (
            spotlight_sample.mean()
            -
            baseline_sample.mean()
        )


# --------------------------------------------------
# 4. CONFIDENCE INTERVALS
# --------------------------------------------------

bootstrap_results = []

for metric in metrics:

    differences = bootstrap_differences[metric]

    lower_ci = np.percentile(
        differences, 2.5
    )

    upper_ci = np.percentile(
        differences, 97.5
    )

    bootstrap_results.append({
        "Metric": metric,
        "Observed_Difference": observed_differences[metric],
        "CI_Lower_95%": lower_ci,
        "CI_Upper_95%": upper_ci,
        "Bootstrap_Probability_Positive": (
            differences > 0
        ).mean(),
        "Bootstrap_Probability_Negative": (
            differences < 0
        ).mean()
    })


bootstrap_ranking_results = pd.DataFrame(
    bootstrap_results
)


# --------------------------------------------------
# 5. DISPLAY RESULTS
# --------------------------------------------------

print("\nPAIRED BOOTSTRAP RESULTS")
print("=" * 100)

print(
    bootstrap_ranking_results
    .round(6)
    .to_string(index=False)
)


print("\nInterpretation guide:")
print("• CI entirely below 0 → evidence of a decrease")
print("• CI entirely above 0 → evidence of an increase")
print("• CI includes 0 → inconclusive at this confidence level")

Users in paired analysis: 1411

PAIRED BOOTSTRAP RESULTS
   Metric  Observed_Difference  CI_Lower_95%  CI_Upper_95%  Bootstrap_Probability_Positive  Bootstrap_Probability_Negative
precision            -0.003402     -0.005032     -0.001772                          0.0000                          0.9999
   recall            -0.000965     -0.001515     -0.000433                          0.0003                          0.9997
     ndcg            -0.002148     -0.003257     -0.001051                          0.0001                          0.9999
 hit_rate            -0.004961     -0.012757      0.002126                          0.0786                          0.8879

Interpretation guide:
• CI entirely below 0 → evidence of a decrease
• CI entirely above 0 → evidence of an increase
• CI includes 0 → inconclusive at this confidence level


In [90]:
# CELL 64: BOOTSTRAP P-VALUES + HOLM MULTIPLE-TESTING CORRECTION

import numpy as np
import pandas as pd


# --------------------------------------------------
# 1. CALCULATE TWO-SIDED BOOTSTRAP P-VALUES
# --------------------------------------------------

alpha = 0.05

p_values = {}

for metric in metrics:

    observed = observed_differences[metric]

    bootstrap_values = bootstrap_differences[metric]

    # Center bootstrap distribution under the null hypothesis
    null_distribution = bootstrap_values - observed

    # Two-sided p-value
    extreme_count = np.sum(
        np.abs(null_distribution) >= abs(observed)
    )

    # Finite-sample correction
    p_value = (
        extreme_count + 1
    ) / (
        N_BOOTSTRAP + 1
    )

    p_values[metric] = p_value


# --------------------------------------------------
# 2. HOLM-BONFERRONI CORRECTION
# --------------------------------------------------

p_value_series = pd.Series(p_values)

sorted_metrics = p_value_series.sort_values()

n_tests = len(sorted_metrics)

holm_adjusted = {}

previous_adjusted_p = 0

for rank, (metric, p_value) in enumerate(
    sorted_metrics.items()
):

    multiplier = n_tests - rank

    adjusted_p = min(
        1.0,
        multiplier * p_value
    )

    # Enforce monotonicity of adjusted p-values
    adjusted_p = max(
        adjusted_p,
        previous_adjusted_p
    )

    holm_adjusted[metric] = adjusted_p

    previous_adjusted_p = adjusted_p


# --------------------------------------------------
# 3. CREATE FINAL RESULTS TABLE
# --------------------------------------------------

statistical_results = []

for metric in metrics:

    raw_p = p_values[metric]

    adjusted_p = holm_adjusted[metric]

    statistical_results.append({

        "Metric": metric,

        "Observed_Difference": (
            observed_differences[metric]
        ),

        "Bootstrap_p_value": raw_p,

        "Holm_Adjusted_p": adjusted_p,

        "Significant_Before_Correction": (
            raw_p < alpha
        ),

        "Significant_After_Holm": (
            adjusted_p < alpha
        )
    })


statistical_results_df = pd.DataFrame(
    statistical_results
)


# --------------------------------------------------
# 4. DISPLAY RESULTS
# --------------------------------------------------

print("MULTIPLE TESTING CORRECTION: HOLM")
print("=" * 100)

print(
    statistical_results_df
    .round(6)
    .to_string(index=False)
)

# --------------------------------------------------
# 5. SUMMARY
# --------------------------------------------------

significant_metrics = statistical_results_df.loc[
    statistical_results_df["Significant_After_Holm"],
    "Metric"
].tolist()

print("\nSIGNIFICANT METRICS AFTER HOLM CORRECTION")
print("=" * 100)

print(
    significant_metrics
    if len(significant_metrics) > 0
    else "No metrics remain statistically significant."
)

print("\nSignificance level:", alpha)

print("\n✅ HOLM MULTIPLE-TESTING ANALYSIS COMPLETED!")


MULTIPLE TESTING CORRECTION: HOLM
   Metric  Observed_Difference  Bootstrap_p_value  Holm_Adjusted_p  Significant_Before_Correction  Significant_After_Holm
precision            -0.003402           0.000300         0.001200                           True                    True
   recall            -0.000965           0.000700         0.001400                           True                    True
     ndcg            -0.002148           0.000300         0.001200                           True                    True
 hit_rate            -0.004961           0.212779         0.212779                          False                   False

SIGNIFICANT METRICS AFTER HOLM CORRECTION
['precision', 'recall', 'ndcg']

Significance level: 0.05

✅ HOLM MULTIPLE-TESTING ANALYSIS COMPLETED!


In [89]:
# --------------------------------------------------
# 5. SUMMARY
# --------------------------------------------------

significant_metrics = statistical_results_df.loc[
    statistical_results_df["Significant_After_Holm"],
    "Metric"
].tolist()

print("\nSIGNIFICANT METRICS AFTER HOLM CORRECTION")
print("=" * 100)

print(
    significant_metrics
    if len(significant_metrics) > 0
    else "No metrics remain statistically significant."
)

print("\nSignificance level:", alpha)

print("\n✅ HOLM MULTIPLE-TESTING ANALYSIS COMPLETED!")


SIGNIFICANT METRICS AFTER HOLM CORRECTION
['precision', 'recall', 'ndcg']

Significance level: 0.05

✅ HOLM MULTIPLE-TESTING ANALYSIS COMPLETED!


In [91]:
# CELL 65: SAVE RANKING EVALUATION RESULTS

from pathlib import Path

RESULTS_DIR = Path("../results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)


# --------------------------------------------------
# 1. SAVE SUMMARY RESULTS
# --------------------------------------------------

ranking_quality_comparison.to_csv(
    RESULTS_DIR / "ranking_quality_comparison.csv",
    index=False
)

metric_changes.to_csv(
    RESULTS_DIR / "ranking_metric_changes.csv"
)

statistical_results_df.to_csv(
    RESULTS_DIR / "ranking_statistical_tests_holm.csv",
    index=False
)

bootstrap_ranking_results.to_csv(
    RESULTS_DIR / "ranking_bootstrap_confidence_intervals.csv",
    index=False
)


# --------------------------------------------------
# 2. SAVE USER-LEVEL RESULTS
# --------------------------------------------------

baseline_user_metrics.to_csv(
    RESULTS_DIR / "baseline_user_ranking_metrics.csv",
    index=False
)

spotlight_user_metrics.to_csv(
    RESULTS_DIR / "spotlight_user_ranking_metrics.csv",
    index=False
)


# --------------------------------------------------
# 3. SAVE BOOTSTRAP DISTRIBUTIONS
# --------------------------------------------------

bootstrap_distributions_df = pd.DataFrame(
    bootstrap_differences
)

bootstrap_distributions_df.to_csv(
    RESULTS_DIR / "ranking_bootstrap_distributions.csv",
    index=False
)


# --------------------------------------------------
# 4. READ-BACK VERIFICATION
# --------------------------------------------------

saved_files = [
    "ranking_quality_comparison.csv",
    "ranking_metric_changes.csv",
    "ranking_statistical_tests_holm.csv",
    "ranking_bootstrap_confidence_intervals.csv",
    "baseline_user_ranking_metrics.csv",
    "spotlight_user_ranking_metrics.csv",
    "ranking_bootstrap_distributions.csv"
]

print("SAVED EVALUATION FILES")
print("=" * 80)

for filename in saved_files:

    filepath = RESULTS_DIR / filename

    assert filepath.exists(), f"Missing file: {filename}"

    check = pd.read_csv(filepath)

    print(
        f"{filename:<50} "
        f"Rows: {len(check):>6} | "
        f"Columns: {len(check.columns):>3}"
    )


print("\n✅ RANKING EVALUATION RESULTS SAVED AND VERIFIED!")

SAVED EVALUATION FILES
ranking_quality_comparison.csv                     Rows:      2 | Columns:   5
ranking_metric_changes.csv                         Rows:      4 | Columns:   5
ranking_statistical_tests_holm.csv                 Rows:      4 | Columns:   6
ranking_bootstrap_confidence_intervals.csv         Rows:      4 | Columns:   6
baseline_user_ranking_metrics.csv                  Rows:   1411 | Columns:   5
spotlight_user_ranking_metrics.csv                 Rows:   1411 | Columns:   5
ranking_bootstrap_distributions.csv                Rows:  10000 | Columns:   4

✅ RANKING EVALUATION RESULTS SAVED AND VERIFIED!


In [92]:
# CELL 66: RECOMMENDATION DIVERSITY EVALUATION

import pandas as pd
import numpy as np
import ast
from pathlib import Path
from itertools import combinations


# --------------------------------------------------
# 1. LOAD REQUIRED DATA
# --------------------------------------------------

PROCESSED_DIR = Path("../processed_data")

video_creator_map = pd.read_csv(
    PROCESSED_DIR / "video_creator_map.csv"
)

video_metadata = pd.read_csv(
    PROCESSED_DIR / "video_metadata.csv"
)

# Parse category IDs safely
def parse_category_ids(value):

    if pd.isna(value):
        return []

    if isinstance(value, list):
        return value

    try:
        parsed = ast.literal_eval(str(value))

        if isinstance(parsed, list):
            return parsed

        return [parsed]

    except (ValueError, SyntaxError):
        return []


video_metadata["category_ids"] = (
    video_metadata["category_ids"]
    .apply(parse_category_ids)
)


# --------------------------------------------------
# 2. BUILD VIDEO METADATA LOOKUP
# --------------------------------------------------

video_information = video_creator_map.merge(
    video_metadata[["video_id", "category_ids"]],
    on="video_id",
    how="left",
    validate="one_to_one"
)

assert video_information["video_id"].is_unique

print("Video information records:", len(video_information))
print(
    "Missing creator information:",
    video_information["author_id"].isna().sum()
)
print(
    "Missing category information:",
    video_information["category_ids"].isna().sum()
)


# --------------------------------------------------
# 3. DIVERSITY METRICS FUNCTION
# --------------------------------------------------

def evaluate_diversity(recommendations, model_name):

    recommendations = recommendations.copy()

    recommendations = recommendations.merge(
        video_information[
            ["video_id", "author_id", "category_ids"]
        ],
        on="video_id",
        how="left",
        validate="many_to_one"
    )

    results = []

    for user_id, group in recommendations.groupby("user_id"):

        creators = group["author_id"].tolist()

        categories_per_video = (
            group["category_ids"]
            .apply(lambda x: x if isinstance(x, list) else [])
            .tolist()
        )

        # Unique creators
        unique_creators = len(set(creators))

        # Flatten category lists
        all_categories = set()

        for category_list in categories_per_video:
            all_categories.update(category_list)

        unique_categories = len(all_categories)

        # Creator pairwise diversity:
        # proportion of video pairs with different creators
        creator_pairs = list(
            combinations(creators, 2)
        )

        creator_pair_diversity = (
            sum(
                creators[i] != creators[j]
                for i, j in combinations(
                    range(len(creators)), 2
                )
            ) / len(creator_pairs)
            if len(creator_pairs) > 0
            else 0
        )

        # Category pairwise diversity:
        # Jaccard distance between category sets
        category_pair_distances = []

        for i, j in combinations(
            range(len(categories_per_video)), 2
        ):

            cat_i = set(categories_per_video[i])
            cat_j = set(categories_per_video[j])

            union = cat_i | cat_j
            intersection = cat_i & cat_j

            if len(union) == 0:

                distance = 0

            else:

                distance = 1 - (
                    len(intersection) / len(union)
                )

            category_pair_distances.append(distance)

        category_pair_diversity = (
            np.mean(category_pair_distances)
            if category_pair_distances
            else 0
        )

        results.append({
            "user_id": user_id,
            "Model": model_name,
            "Unique_Creators@10": unique_creators,
            "Unique_Categories@10": unique_categories,
            "Creator_Diversity@10": unique_creators / len(group),
            "Creator_Pairwise_Diversity": creator_pair_diversity,
            "Category_Pairwise_Diversity": category_pair_diversity
        })

    return pd.DataFrame(results)


# --------------------------------------------------
# 4. EVALUATE BOTH MODELS
# --------------------------------------------------

baseline_diversity = evaluate_diversity(
    baseline_recommendations,
    "Baseline"
)

spotlight_diversity = evaluate_diversity(
    spotlight_constrained_recommendations,
    "Spotlight"
)


# --------------------------------------------------
# 5. COMBINE USER-LEVEL RESULTS
# --------------------------------------------------

diversity_user_results = pd.concat(
    [
        baseline_diversity,
        spotlight_diversity
    ],
    ignore_index=True
)


# --------------------------------------------------
# 6. OVERALL DIVERSITY SUMMARY
# --------------------------------------------------

diversity_metrics = [
    "Unique_Creators@10",
    "Unique_Categories@10",
    "Creator_Diversity@10",
    "Creator_Pairwise_Diversity",
    "Category_Pairwise_Diversity"
]

diversity_summary = (
    diversity_user_results
    .groupby("Model")[diversity_metrics]
    .mean()
)


print("\nRECOMMENDATION DIVERSITY COMPARISON")
print("=" * 100)

print(
    diversity_summary
    .round(6)
    .to_string()
)


# --------------------------------------------------
# 7. SPOTLIGHT - BASELINE CHANGE
# --------------------------------------------------

diversity_change = (
    diversity_summary.loc["Spotlight"]
    -
    diversity_summary.loc["Baseline"]
)

diversity_comparison = pd.DataFrame({
    "Baseline": diversity_summary.loc["Baseline"],
    "Spotlight": diversity_summary.loc["Spotlight"],
    "Absolute_Change": diversity_change
})


print("\nDIVERSITY METRIC CHANGES")
print("=" * 100)

print(
    diversity_comparison
    .round(6)
    .to_string()
)


# --------------------------------------------------
# 8. VALIDATION
# --------------------------------------------------

assert len(baseline_diversity) == 1411
assert len(spotlight_diversity) == 1411

assert baseline_diversity["user_id"].nunique() == 1411
assert spotlight_diversity["user_id"].nunique() == 1411

print("\n✅ DIVERSITY EVALUATION COMPLETED!")

Video information records: 3327
Missing creator information: 0
Missing category information: 0

RECOMMENDATION DIVERSITY COMPARISON
           Unique_Creators@10  Unique_Categories@10  Creator_Diversity@10  Creator_Pairwise_Diversity  Category_Pairwise_Diversity
Model                                                                                                                             
Baseline             9.756910              7.197732              0.975691                    0.993181                     0.807795
Spotlight            9.772502              7.289865              0.977250                    0.993669                     0.814849

DIVERSITY METRIC CHANGES
                             Baseline  Spotlight  Absolute_Change
Unique_Creators@10           9.756910   9.772502         0.015592
Unique_Categories@10         7.197732   7.289865         0.092133
Creator_Diversity@10         0.975691   0.977250         0.001559
Creator_Pairwise_Diversity   0.993181   0.993669      

In [93]:
# CELL 67: PAIRED BOOTSTRAP - DIVERSITY METRICS

import numpy as np
import pandas as pd


# --------------------------------------------------
# 1. ALIGN USER-LEVEL DIVERSITY RESULTS
# --------------------------------------------------

diversity_metrics = [
    "Unique_Creators@10",
    "Unique_Categories@10",
    "Creator_Diversity@10",
    "Creator_Pairwise_Diversity",
    "Category_Pairwise_Diversity"
]

baseline_diversity_paired = (
    baseline_diversity
    .set_index("user_id")
    .sort_index()
)

spotlight_diversity_paired = (
    spotlight_diversity
    .set_index("user_id")
    .sort_index()
)

assert baseline_diversity_paired.index.equals(
    spotlight_diversity_paired.index
)

n_users_diversity = len(
    baseline_diversity_paired
)

print(
    "Users in paired diversity analysis:",
    n_users_diversity
)


# --------------------------------------------------
# 2. OBSERVED DIFFERENCES
# --------------------------------------------------

observed_diversity_differences = {}

for metric in diversity_metrics:

    observed_diversity_differences[metric] = (
        spotlight_diversity_paired[metric].mean()
        -
        baseline_diversity_paired[metric].mean()
    )


# --------------------------------------------------
# 3. BOOTSTRAP
# --------------------------------------------------

N_BOOTSTRAP_DIVERSITY = 10000

RANDOM_STATE_DIVERSITY = 42

rng_diversity = np.random.default_rng(
    RANDOM_STATE_DIVERSITY
)

bootstrap_diversity_differences = {
    metric: np.zeros(N_BOOTSTRAP_DIVERSITY)
    for metric in diversity_metrics
}


for b in range(N_BOOTSTRAP_DIVERSITY):

    sampled_indices = rng_diversity.integers(
        0,
        n_users_diversity,
        size=n_users_diversity
    )

    for metric in diversity_metrics:

        baseline_sample = (
            baseline_diversity_paired[metric]
            .to_numpy()[sampled_indices]
        )

        spotlight_sample = (
            spotlight_diversity_paired[metric]
            .to_numpy()[sampled_indices]
        )

        bootstrap_diversity_differences[metric][b] = (
            spotlight_sample.mean()
            -
            baseline_sample.mean()
        )


# --------------------------------------------------
# 4. CONFIDENCE INTERVALS AND P-VALUES
# --------------------------------------------------

diversity_bootstrap_results = []

diversity_p_values = {}

for metric in diversity_metrics:

    differences = (
        bootstrap_diversity_differences[metric]
    )

    observed = (
        observed_diversity_differences[metric]
    )

    lower_ci = np.percentile(
        differences, 2.5
    )

    upper_ci = np.percentile(
        differences, 97.5
    )

    # Two-sided bootstrap p-value
    null_distribution = differences - observed

    extreme_count = np.sum(
        np.abs(null_distribution) >= abs(observed)
    )

    p_value = (
        extreme_count + 1
    ) / (
        N_BOOTSTRAP_DIVERSITY + 1
    )

    diversity_p_values[metric] = p_value

    diversity_bootstrap_results.append({

        "Metric": metric,

        "Observed_Difference": observed,

        "CI_Lower_95%": lower_ci,

        "CI_Upper_95%": upper_ci,

        "Bootstrap_p_value": p_value
    })


diversity_bootstrap_results_df = pd.DataFrame(
    diversity_bootstrap_results
)


# --------------------------------------------------
# 5. HOLM CORRECTION
# --------------------------------------------------

sorted_diversity_p = pd.Series(
    diversity_p_values
).sort_values()

n_diversity_tests = len(
    sorted_diversity_p
)

holm_diversity = {}

previous_p = 0

for rank, (metric, p_value) in enumerate(
    sorted_diversity_p.items()
):

    adjusted_p = min(
        1.0,
        (n_diversity_tests - rank) * p_value
    )

    adjusted_p = max(
        adjusted_p,
        previous_p
    )

    holm_diversity[metric] = adjusted_p

    previous_p = adjusted_p


# --------------------------------------------------
# 6. FINAL STATISTICAL TABLE
# --------------------------------------------------

diversity_bootstrap_results_df[
    "Holm_Adjusted_p"
] = diversity_bootstrap_results_df[
    "Metric"
].map(holm_diversity)


diversity_bootstrap_results_df[
    "Significant_After_Holm"
] = (
    diversity_bootstrap_results_df[
        "Holm_Adjusted_p"
    ] < 0.05
)


print("\nDIVERSITY PAIRED BOOTSTRAP RESULTS")
print("=" * 120)

print(
    diversity_bootstrap_results_df
    .round(6)
    .to_string(index=False)
)


print("\nSIGNIFICANT DIVERSITY METRICS AFTER HOLM")
print("=" * 120)

significant_diversity = (
    diversity_bootstrap_results_df.loc[
        diversity_bootstrap_results_df[
            "Significant_After_Holm"
        ],
        "Metric"
    ].tolist()
)

print(
    significant_diversity
    if len(significant_diversity) > 0
    else "No diversity metrics remain significant."
)


print("\n✅ DIVERSITY STATISTICAL TESTING COMPLETED!")

Users in paired diversity analysis: 1411

DIVERSITY PAIRED BOOTSTRAP RESULTS
                     Metric  Observed_Difference  CI_Lower_95%  CI_Upper_95%  Bootstrap_p_value  Holm_Adjusted_p  Significant_After_Holm
         Unique_Creators@10             0.015592      0.009213      0.022679             0.0001           0.0005                    True
       Unique_Categories@10             0.092133      0.055989      0.128278             0.0001           0.0005                    True
       Creator_Diversity@10             0.001559      0.000921      0.002268             0.0001           0.0005                    True
 Creator_Pairwise_Diversity             0.000488      0.000268      0.000756             0.0006           0.0006                    True
Category_Pairwise_Diversity             0.007055      0.005394      0.008797             0.0001           0.0005                    True

SIGNIFICANT DIVERSITY METRICS AFTER HOLM
['Unique_Creators@10', 'Unique_Categories@10', 'Creator_Div

In [94]:
# CELL 68: SAVE DIVERSITY EVALUATION RESULTS

from pathlib import Path
import pandas as pd


RESULTS_DIR = Path("../results")
RESULTS_DIR.mkdir(exist_ok=True)


# --------------------------------------------------
# 1. SAVE OVERALL DIVERSITY SUMMARY
# --------------------------------------------------

diversity_summary.to_csv(
    RESULTS_DIR / "diversity_summary.csv"
)


diversity_comparison.to_csv(
    RESULTS_DIR / "diversity_metric_changes.csv"
)


# --------------------------------------------------
# 2. SAVE STATISTICAL RESULTS
# --------------------------------------------------

diversity_bootstrap_results_df.to_csv(
    RESULTS_DIR / "diversity_bootstrap_holm_results.csv",
    index=False
)


# --------------------------------------------------
# 3. SAVE USER-LEVEL RESULTS
# --------------------------------------------------

baseline_diversity.to_csv(
    RESULTS_DIR / "baseline_user_diversity.csv",
    index=False
)

spotlight_diversity.to_csv(
    RESULTS_DIR / "spotlight_user_diversity.csv",
    index=False
)


# --------------------------------------------------
# 4. SAVE BOOTSTRAP DISTRIBUTIONS
# --------------------------------------------------

diversity_bootstrap_distributions_df = pd.DataFrame(
    bootstrap_diversity_differences
)

diversity_bootstrap_distributions_df.to_csv(
    RESULTS_DIR / "diversity_bootstrap_distributions.csv",
    index=False
)


# --------------------------------------------------
# 5. VERIFY SAVED FILES
# --------------------------------------------------

diversity_saved_files = [
    "diversity_summary.csv",
    "diversity_metric_changes.csv",
    "diversity_bootstrap_holm_results.csv",
    "baseline_user_diversity.csv",
    "spotlight_user_diversity.csv",
    "diversity_bootstrap_distributions.csv"
]

print("SAVED DIVERSITY EVALUATION FILES")
print("=" * 90)

for filename in diversity_saved_files:

    filepath = RESULTS_DIR / filename

    assert filepath.exists(), f"File missing: {filename}"

    check = pd.read_csv(filepath)

    print(
        f"{filename:<48} "
        f"Rows: {len(check):>6} | "
        f"Columns: {len(check.columns):>3}"
    )


print("\n✅ DIVERSITY RESULTS SAVED AND VERIFIED!")

SAVED DIVERSITY EVALUATION FILES
diversity_summary.csv                            Rows:      2 | Columns:   6
diversity_metric_changes.csv                     Rows:      5 | Columns:   4
diversity_bootstrap_holm_results.csv             Rows:      5 | Columns:   7
baseline_user_diversity.csv                      Rows:   1411 | Columns:   7
spotlight_user_diversity.csv                     Rows:   1411 | Columns:   7
diversity_bootstrap_distributions.csv            Rows:  10000 | Columns:   5

✅ DIVERSITY RESULTS SAVED AND VERIFIED!


In [95]:
# CELL 69: INSPECT VIDEO AND CREATOR POPULARITY DATA

from pathlib import Path
import pandas as pd

PROCESSED_DIR = Path("../processed_data")

creator_popularity = pd.read_csv(
    PROCESSED_DIR / "creator_popularity.csv"
)

video_creator_groups = pd.read_csv(
    PROCESSED_DIR / "video_creator_groups.csv"
)

item_daily_features = pd.read_csv(
    "../data/item_daily_features.csv"
)

print("CREATOR POPULARITY DATA")
print("=" * 80)

print("Shape:", creator_popularity.shape)
print("\nColumns:")
print(creator_popularity.columns.tolist())

print("\nPreview:")
print(creator_popularity.head())


print("\nVIDEO CREATOR GROUPS")
print("=" * 80)

print("Shape:", video_creator_groups.shape)
print("\nColumns:")
print(video_creator_groups.columns.tolist())

print("\nPreview:")
print(video_creator_groups.head())


print("\nITEM DAILY FEATURES")
print("=" * 80)

print("Shape:", item_daily_features.shape)
print("\nColumns:")
print(item_daily_features.columns.tolist())

print("\nPreview:")
print(item_daily_features.head())

CREATOR POPULARITY DATA
Shape: (2020, 5)

Columns:
['author_id', 'eligible_video_count', 'average_video_daily_shows', 'total_observed_video_days', 'creator_group']

Preview:
   author_id  eligible_video_count  average_video_daily_shows  total_observed_video_days creator_group
0          0                     1              769195.380952                         42   Established
1          2                     2              185340.641039                         87   Established
2          5                     1              374240.682927                         41   Established
3          8                     1               58768.127273                         55   Established
4         11                     8               34178.782466                        252   Established

VIDEO CREATOR GROUPS
Shape: (3327, 4)

Columns:
['video_id', 'author_id', 'average_video_daily_shows', 'creator_group']

Preview:
   video_id  author_id  average_video_daily_shows creator_group
0       103  

In [96]:
# CELL 70: VIDEO POPULARITY DISTRIBUTION CHECK

import numpy as np
import pandas as pd


# --------------------------------------------------
# 1. POPULARITY DATA
# --------------------------------------------------

popularity_data = video_creator_groups[
    ["video_id", "average_video_daily_shows", "creator_group"]
].copy()

popularity_data = popularity_data.drop_duplicates(
    subset="video_id"
)


# --------------------------------------------------
# 2. VALIDATION
# --------------------------------------------------

print("VIDEO POPULARITY VALIDATION")
print("=" * 80)

print("Total videos:", len(popularity_data))

print(
    "Missing popularity values:",
    popularity_data["average_video_daily_shows"].isna().sum()
)

print(
    "Zero popularity values:",
    (popularity_data["average_video_daily_shows"] == 0).sum()
)

print(
    "Negative popularity values:",
    (popularity_data["average_video_daily_shows"] < 0).sum()
)


# --------------------------------------------------
# 3. DISTRIBUTION STATISTICS
# --------------------------------------------------

print("\nPOPULARITY DISTRIBUTION")
print("=" * 80)

print(
    popularity_data[
        "average_video_daily_shows"
    ].describe(
        percentiles=[
            0.01,
            0.05,
            0.10,
            0.25,
            0.50,
            0.75,
            0.90,
            0.95,
            0.99
        ]
    ).round(3)
)


# --------------------------------------------------
# 4. GROUP-WISE POPULARITY
# --------------------------------------------------

print("\nPOPULARITY BY CREATOR GROUP")
print("=" * 80)

print(
    popularity_data
    .groupby("creator_group")[
        "average_video_daily_shows"
    ]
    .agg([
        "count",
        "mean",
        "median",
        "min",
        "max"
    ])
    .round(3)
)


# --------------------------------------------------
# 5. SKEWNESS
# --------------------------------------------------

print("\nPopularity skewness:")

print(
    popularity_data[
        "average_video_daily_shows"
    ].skew()
)


print("\n✅ POPULARITY DISTRIBUTION INSPECTION COMPLETED!")

VIDEO POPULARITY VALIDATION
Total videos: 3327
Missing popularity values: 12
Zero popularity values: 0
Negative popularity values: 0

POPULARITY DISTRIBUTION
count    3.315000e+03
mean     3.876020e+05
std      8.852908e+05
min      1.700000e+01
1%       6.108616e+03
5%       1.019715e+04
10%      1.558411e+04
25%      4.509359e+04
50%      1.785724e+05
75%      4.136655e+05
90%      8.418090e+05
95%      1.407294e+06
99%      3.400218e+06
max      1.915338e+07
Name: average_video_daily_shows, dtype: float64

POPULARITY BY CREATOR GROUP
               count        mean      median      min           max
creator_group                                                      
Emerging         555   14398.452   13451.154     17.0  2.751162e+04
Established     2760  462648.350  241595.643  27697.0  1.915338e+07

Popularity skewness:
9.822499104863665

✅ POPULARITY DISTRIBUTION INSPECTION COMPLETED!


In [97]:
# CELL 71: NOVELTY AND POPULARITY BIAS EVALUATION

import numpy as np
import pandas as pd


# --------------------------------------------------
# 1. PREPARE POPULARITY LOOKUP
# --------------------------------------------------

video_popularity = video_creator_groups[
    ["video_id", "average_video_daily_shows"]
].drop_duplicates("video_id").copy()

video_popularity = video_popularity.dropna(
    subset=["average_video_daily_shows"]
)

assert (video_popularity["average_video_daily_shows"] > 0).all()

# Popularity probability across the experimental video catalogue
total_catalogue_popularity = (
    video_popularity["average_video_daily_shows"].sum()
)

video_popularity["popularity_probability"] = (
    video_popularity["average_video_daily_shows"]
    / total_catalogue_popularity
)

# Information content / novelty
video_popularity["video_novelty"] = (
    -np.log2(
        video_popularity["popularity_probability"]
    )
)


print("Popularity catalogue prepared")
print("=" * 80)
print("Videos with valid popularity:", len(video_popularity))
print("Total catalogue popularity:", total_catalogue_popularity)


# --------------------------------------------------
# 2. EVALUATION FUNCTION
# --------------------------------------------------

def evaluate_novelty(recommendations, model_name):

    recs = recommendations.copy()

    # Preserve ranking order
    recs = recs.sort_values(
        ["user_id", "predicted_preference"],
        ascending=[True, False],
        kind="mergesort"
    )

    recs = recs.merge(
        video_popularity[
            [
                "video_id",
                "average_video_daily_shows",
                "popularity_probability",
                "video_novelty"
            ]
        ],
        on="video_id",
        how="left",
        validate="many_to_one"
    )

    user_results = []

    for user_id, group in recs.groupby("user_id"):

        valid = group.dropna(
            subset=["average_video_daily_shows"]
        )

        user_results.append({

            "user_id": user_id,
            "Model": model_name,

            # Average information content of recommendations
            "Novelty@10": valid["video_novelty"].mean(),

            # Mean and median video popularity
            "Mean_Popularity@10": (
                valid["average_video_daily_shows"].mean()
            ),

            "Median_Popularity@10": (
                valid["average_video_daily_shows"].median()
            ),

            # Fraction of recommendation slots with valid popularity
            "Popularity_Metadata_Coverage@10": (
                len(valid) / len(group)
            ),

            "Valid_Recommendations": len(valid)

        })

    return pd.DataFrame(user_results)


# --------------------------------------------------
# 3. EVALUATE BASELINE AND SPOTLIGHT
# --------------------------------------------------

baseline_novelty = evaluate_novelty(
    baseline_recommendations,
    "Baseline"
)

spotlight_novelty = evaluate_novelty(
    spotlight_constrained_recommendations,
    "Spotlight"
)


# --------------------------------------------------
# 4. OVERALL SUMMARY
# --------------------------------------------------

novelty_metrics = [
    "Novelty@10",
    "Mean_Popularity@10",
    "Median_Popularity@10",
    "Popularity_Metadata_Coverage@10"
]

novelty_summary = pd.concat(
    [
        baseline_novelty,
        spotlight_novelty
    ],
    ignore_index=True
).groupby("Model")[novelty_metrics].mean()


print("\nNOVELTY AND POPULARITY COMPARISON")
print("=" * 100)

print(
    novelty_summary.round(6).to_string()
)


# --------------------------------------------------
# 5. ABSOLUTE CHANGES
# --------------------------------------------------

novelty_changes = (
    novelty_summary.loc["Spotlight"]
    -
    novelty_summary.loc["Baseline"]
)

novelty_comparison = pd.DataFrame({

    "Baseline": novelty_summary.loc["Baseline"],

    "Spotlight": novelty_summary.loc["Spotlight"],

    "Absolute_Change": novelty_changes

})


print("\nNOVELTY METRIC CHANGES")
print("=" * 100)

print(
    novelty_comparison.round(6).to_string()
)


# --------------------------------------------------
# 6. METADATA COVERAGE DIAGNOSTICS
# --------------------------------------------------

print("\nPOPULARITY METADATA COVERAGE")
print("=" * 100)

for model_name, results in [
    ("Baseline", baseline_novelty),
    ("Spotlight", spotlight_novelty)
]:

    total_slots = len(results) * 10

    valid_slots = results[
        "Valid_Recommendations"
    ].sum()

    missing_slots = total_slots - valid_slots

    print(f"\n{model_name}")
    print("Total recommendation slots:", total_slots)
    print("Valid popularity slots:", valid_slots)
    print("Missing popularity slots:", missing_slots)
    print(
        "Coverage:",
        round(valid_slots / total_slots, 6)
    )


# --------------------------------------------------
# 7. VALIDATION
# --------------------------------------------------

assert len(baseline_novelty) == 1411
assert len(spotlight_novelty) == 1411

assert baseline_novelty["user_id"].nunique() == 1411
assert spotlight_novelty["user_id"].nunique() == 1411

print("\n✅ NOVELTY AND POPULARITY EVALUATION COMPLETED!")

Popularity catalogue prepared
Videos with valid popularity: 3315
Total catalogue popularity: 1284900586.5542288

NOVELTY AND POPULARITY COMPARISON
           Novelty@10  Mean_Popularity@10  Median_Popularity@10  Popularity_Metadata_Coverage@10
Model                                                                                           
Baseline    13.823319       242817.288567         125207.468416                              1.0
Spotlight   13.982993       228772.707754         113315.693255                              1.0

NOVELTY METRIC CHANGES
                                      Baseline      Spotlight  Absolute_Change
Novelty@10                           13.823319      13.982993         0.159674
Mean_Popularity@10               242817.288567  228772.707754    -14044.580813
Median_Popularity@10             125207.468416  113315.693255    -11891.775161
Popularity_Metadata_Coverage@10       1.000000       1.000000         0.000000

POPULARITY METADATA COVERAGE

Baseline
Total 

In [99]:
# CELL 72: PAIRED BOOTSTRAP - NOVELTY AND POPULARITY

import numpy as np
import pandas as pd


# --------------------------------------------------
# 1. ALIGN USER-LEVEL RESULTS
# --------------------------------------------------

novelty_test_metrics = [
    "Novelty@10",
    "Mean_Popularity@10",
    "Median_Popularity@10"
]

baseline_novelty_paired = (
    baseline_novelty
    .set_index("user_id")
    .sort_index()
)

spotlight_novelty_paired = (
    spotlight_novelty
    .set_index("user_id")
    .sort_index()
)

assert baseline_novelty_paired.index.equals(
    spotlight_novelty_paired.index
)

n_novelty_users = len(
    baseline_novelty_paired
)

print("Users in paired novelty analysis:", n_novelty_users)


# --------------------------------------------------
# 2. OBSERVED DIFFERENCES
# --------------------------------------------------

observed_novelty_differences = {}

for metric in novelty_test_metrics:

    observed_novelty_differences[metric] = (
        spotlight_novelty_paired[metric].mean()
        -
        baseline_novelty_paired[metric].mean()
    )


# --------------------------------------------------
# 3. PAIRED BOOTSTRAP
# --------------------------------------------------

N_BOOTSTRAP_NOVELTY = 10000

RANDOM_STATE_NOVELTY = 42

rng_novelty = np.random.default_rng(
    RANDOM_STATE_NOVELTY
)

bootstrap_novelty_differences = {
    metric: np.zeros(N_BOOTSTRAP_NOVELTY)
    for metric in novelty_test_metrics
}


for b in range(N_BOOTSTRAP_NOVELTY):

    sampled_indices = rng_novelty.integers(
        0,
        n_novelty_users,
        size=n_novelty_users
    )

    for metric in novelty_test_metrics:

        baseline_sample = (
            baseline_novelty_paired[metric]
            .to_numpy()[sampled_indices]
        )

        spotlight_sample = (
            spotlight_novelty_paired[metric]
            .to_numpy()[sampled_indices]
        )

        bootstrap_novelty_differences[metric][b] = (
            spotlight_sample.mean()
            -
            baseline_sample.mean()
        )


# --------------------------------------------------
# 4. CONFIDENCE INTERVALS AND P-VALUES
# --------------------------------------------------

novelty_bootstrap_results = []

novelty_p_values = {}

for metric in novelty_test_metrics:

    differences = (
        bootstrap_novelty_differences[metric]
    )

    observed = (
        observed_novelty_differences[metric]
    )

    lower_ci = np.percentile(
        differences, 2.5
    )

    upper_ci = np.percentile(
        differences, 97.5
    )

    # Center bootstrap distribution under the null
    null_distribution = differences - observed

    extreme_count = np.sum(
        np.abs(null_distribution) >= abs(observed)
    )

    p_value = (
        extreme_count + 1
    ) / (
        N_BOOTSTRAP_NOVELTY + 1
    )

    novelty_p_values[metric] = p_value

    novelty_bootstrap_results.append({

        "Metric": metric,

        "Observed_Difference": observed,

        "CI_Lower_95%": lower_ci,

        "CI_Upper_95%": upper_ci,

        "Bootstrap_p_value": p_value
    })


novelty_bootstrap_results_df = pd.DataFrame(
    novelty_bootstrap_results
)


# --------------------------------------------------
# 5. HOLM CORRECTION
# --------------------------------------------------

sorted_novelty_p = pd.Series(
    novelty_p_values
).sort_values()

n_novelty_tests = len(
    sorted_novelty_p
)

holm_novelty = {}

previous_p = 0

for rank, (metric, p_value) in enumerate(
    sorted_novelty_p.items()
):

    adjusted_p = min(
        1.0,
        (n_novelty_tests - rank) * p_value
    )

    adjusted_p = max(
        adjusted_p,
        previous_p
    )

    holm_novelty[metric] = adjusted_p

    previous_p = adjusted_p


# --------------------------------------------------
# 6. FINAL RESULTS TABLE
# --------------------------------------------------

novelty_bootstrap_results_df[
    "Holm_Adjusted_p"
] = novelty_bootstrap_results_df[
    "Metric"
].map(holm_novelty)


novelty_bootstrap_results_df[
    "Significant_After_Holm"
] = (
    novelty_bootstrap_results_df[
        "Holm_Adjusted_p"
    ] < 0.05
)


print("\nNOVELTY AND POPULARITY PAIRED BOOTSTRAP RESULTS")
print("=" * 120)

print(
    novelty_bootstrap_results_df
    .round(6)
    .to_string(index=False)
)


print("\nSIGNIFICANT METRICS AFTER HOLM")
print("=" * 120)

significant_novelty_metrics = (
    novelty_bootstrap_results_df.loc[
        novelty_bootstrap_results_df[
            "Significant_After_Holm"
        ],
        "Metric"
    ].tolist()
)

print(
    significant_novelty_metrics
    if len(significant_novelty_metrics) > 0
    else "No metrics remain statistically significant."
)


print("\nSignificance level: 0.05")

print("\n✅ NOVELTY STATISTICAL TESTING COMPLETED!")



Users in paired novelty analysis: 1411

NOVELTY AND POPULARITY PAIRED BOOTSTRAP RESULTS
              Metric  Observed_Difference  CI_Lower_95%  CI_Upper_95%  Bootstrap_p_value  Holm_Adjusted_p  Significant_After_Holm
          Novelty@10             0.159674      0.148697      0.170647             0.0001           0.0003                    True
  Mean_Popularity@10        -14044.580813 -16755.311984 -11732.687930             0.0001           0.0003                    True
Median_Popularity@10        -11891.775161 -13226.359585 -10605.253182             0.0001           0.0003                    True

SIGNIFICANT METRICS AFTER HOLM
['Novelty@10', 'Mean_Popularity@10', 'Median_Popularity@10']

Significance level: 0.05

✅ NOVELTY STATISTICAL TESTING COMPLETED!


In [100]:
# CELL 73: SAVE NOVELTY AND POPULARITY EVALUATION RESULTS

import os
import pandas as pd
import numpy as np

os.makedirs(RESULTS_DIR, exist_ok=True)


# --------------------------------------------------
# 1. SAVE SUMMARY
# --------------------------------------------------

novelty_summary.to_csv(
    os.path.join(
        RESULTS_DIR,
        "novelty_summary.csv"
    )
)


# --------------------------------------------------
# 2. SAVE METRIC CHANGES
# --------------------------------------------------

novelty_comparison.to_csv(
    os.path.join(
        RESULTS_DIR,
        "novelty_metric_changes.csv"
    )
)


# --------------------------------------------------
# 3. SAVE STATISTICAL RESULTS
# --------------------------------------------------

novelty_bootstrap_results_df.to_csv(
    os.path.join(
        RESULTS_DIR,
        "novelty_bootstrap_holm_results.csv"
    ),
    index=False
)


# --------------------------------------------------
# 4. SAVE BOOTSTRAP DISTRIBUTIONS
# --------------------------------------------------

novelty_bootstrap_distributions = pd.DataFrame(
    bootstrap_novelty_differences
)

novelty_bootstrap_distributions.to_csv(
    os.path.join(
        RESULTS_DIR,
        "novelty_bootstrap_distributions.csv"
    ),
    index=False
)


# --------------------------------------------------
# 5. SAVE USER-LEVEL RESULTS
# --------------------------------------------------

baseline_novelty.to_csv(
    os.path.join(
        RESULTS_DIR,
        "baseline_user_novelty.csv"
    ),
    index=False
)

spotlight_novelty.to_csv(
    os.path.join(
        RESULTS_DIR,
        "spotlight_user_novelty.csv"
    ),
    index=False
)


# --------------------------------------------------
# 6. VERIFY FILES
# --------------------------------------------------

novelty_files = [
    "novelty_summary.csv",
    "novelty_metric_changes.csv",
    "novelty_bootstrap_holm_results.csv",
    "novelty_bootstrap_distributions.csv",
    "baseline_user_novelty.csv",
    "spotlight_user_novelty.csv"
]

print("NOVELTY RESULTS FILE VERIFICATION")
print("=" * 80)

for filename in novelty_files:

    filepath = os.path.join(
        RESULTS_DIR,
        filename
    )

    assert os.path.exists(filepath), (
        f"Missing file: {filename}"
    )

    saved_df = pd.read_csv(filepath)

    print(
        f"{filename:<45} "
        f"{saved_df.shape}"
    )


print("\n✅ ALL NOVELTY AND POPULARITY RESULTS SAVED AND VERIFIED!")

NOVELTY RESULTS FILE VERIFICATION
novelty_summary.csv                           (2, 5)
novelty_metric_changes.csv                    (4, 4)
novelty_bootstrap_holm_results.csv            (3, 7)
novelty_bootstrap_distributions.csv           (10000, 3)
baseline_user_novelty.csv                     (1411, 7)
spotlight_user_novelty.csv                    (1411, 7)

✅ ALL NOVELTY AND POPULARITY RESULTS SAVED AND VERIFIED!


In [101]:
# CELL 74: CONSOLIDATED SPOTLIGHT PERFORMANCE SUMMARY

import pandas as pd
import numpy as np

# --------------------------------------------------
# 1. CONSOLIDATE RESULTS
# --------------------------------------------------

final_model_comparison = pd.DataFrame({

    "Metric": [
        "Precision@10",
        "Recall@10",
        "NDCG@10",
        "HitRate@10",
        "Unique Creators@10",
        "Unique Categories@10",
        "Creator Diversity@10",
        "Creator Pairwise Diversity",
        "Category Pairwise Diversity",
        "Novelty@10",
        "Mean Popularity@10",
        "Median Popularity@10",
        "Emerging Creator Exposure Slots",
        "Emerging Creator Exposure Share",
        "Emerging Creator Coverage",
        "Emerging Creator Zero Exposure",
        "Emerging Creator Gini",
        "Established Creator Gini"
    ],

    "Baseline": [
        0.139405,
        0.037370,
        0.139410,
        0.696669,
        9.756910,
        7.197732,
        0.975691,
        0.993181,
        0.807795,
        13.823319,
        242817.288567,
        125207.468416,
        3678,
        0.260666,
        0.518812,
        243,
        0.7845,
        0.8659
    ],

    "Spotlight": [
        0.136003,
        0.036405,
        0.137262,
        0.691708,
        9.772502,
        7.289865,
        0.977250,
        0.993669,
        0.814849,
        13.982993,
        228772.707754,
        113315.693255,
        4297,
        0.3045,
        0.627723,
        188,
        0.7117,
        0.8706
    ]
})


# --------------------------------------------------
# 2. CALCULATE CHANGE
# --------------------------------------------------

final_model_comparison["Absolute_Change"] = (
    final_model_comparison["Spotlight"]
    -
    final_model_comparison["Baseline"]
)


final_model_comparison["Relative_Change_Percent"] = (
    (
        final_model_comparison["Spotlight"]
        -
        final_model_comparison["Baseline"]
    )
    /
    final_model_comparison["Baseline"].replace(0, np.nan)
) * 100


# --------------------------------------------------
# 3. DISPLAY RESULTS
# --------------------------------------------------

print("FINAL SPOTLIGHT VS BASELINE COMPARISON")
print("=" * 120)

display(
    final_model_comparison.round(6)
)


# --------------------------------------------------
# 4. SAVE RESULTS
# --------------------------------------------------

final_model_comparison.to_csv(
    os.path.join(
        RESULTS_DIR,
        "final_spotlight_comparison.csv"
    ),
    index=False
)


print("\nSaved: final_spotlight_comparison.csv")

print("\n✅ FINAL MODEL COMPARISON COMPLETED!")

FINAL SPOTLIGHT VS BASELINE COMPARISON


,Metric,Baseline,Spotlight,Absolute_Change,Relative_Change_Percent
0,Precision@10,0.139405,0.136003,-0.003402,-2.440372
1,Recall@10,0.037370,0.036405,-0.000965,-2.582285
2,NDCG@10,0.139410,0.137262,-0.002148,-1.540779
3,HitRate@10,0.696669,0.691708,-0.004961,-0.712103
4,Unique Creators@10,9.756910,9.772502,0.015592,0.159805
5,Unique Categories@10,7.197732,7.289865,0.092133,1.280028
6,Creator Diversity@10,0.975691,0.977250,0.001559,0.159784
7,Creator Pairwise Diversity,0.993181,0.993669,0.000488,0.049135
8,Category Pairwise Diversity,0.807795,0.814849,0.007054,0.873241
9,Novelty@10,13.823319,13.982993,0.159674,1.155106



Saved: final_spotlight_comparison.csv

✅ FINAL MODEL COMPARISON COMPLETED!


In [102]:
# CELL 75: AUDIT SAVED STATISTICAL RESULTS

import os
import pandas as pd

statistical_files = {
    "Ranking": "ranking_statistical_tests_holm.csv",
    "Diversity": "diversity_bootstrap_holm_results.csv",
    "Novelty": "novelty_bootstrap_holm_results.csv"
}

statistical_results = {}

for family, filename in statistical_files.items():

    filepath = os.path.join(
        RESULTS_DIR,
        filename
    )

    assert os.path.exists(filepath), (
        f"Missing statistical result file: {filepath}"
    )

    df = pd.read_csv(filepath)

    statistical_results[family] = df

    print("\n" + "=" * 100)
    print(f"{family.upper()} STATISTICAL RESULTS")
    print("=" * 100)

    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist())

    display(df.head())


print("\n✅ STATISTICAL RESULTS AUDIT COMPLETED!")


RANKING STATISTICAL RESULTS
Shape: (4, 6)
Columns: ['Metric', 'Observed_Difference', 'Bootstrap_p_value', 'Holm_Adjusted_p', 'Significant_Before_Correction', 'Significant_After_Holm']


,Metric,Observed_Difference,Bootstrap_p_value,Holm_Adjusted_p,Significant_Before_Correction,Significant_After_Holm
0,precision,-0.003402,0.000300,0.001200,True,True
1,recall,-0.000965,0.000700,0.001400,True,True
2,ndcg,-0.002148,0.000300,0.001200,True,True
3,hit_rate,-0.004961,0.212779,0.212779,False,False



DIVERSITY STATISTICAL RESULTS
Shape: (5, 7)
Columns: ['Metric', 'Observed_Difference', 'CI_Lower_95%', 'CI_Upper_95%', 'Bootstrap_p_value', 'Holm_Adjusted_p', 'Significant_After_Holm']


,Metric,Observed_Difference,CI_Lower_95%,CI_Upper_95%,Bootstrap_p_value,Holm_Adjusted_p,Significant_After_Holm
0,Unique_Creators@10,0.015592,0.009213,0.022679,0.0001,0.0005,True
1,Unique_Categories@10,0.092133,0.055989,0.128278,0.0001,0.0005,True
2,Creator_Diversity@10,0.001559,0.000921,0.002268,0.0001,0.0005,True
3,Creator_Pairwise_Diversity,0.000488,0.000268,0.000756,0.0006,0.0006,True
4,Category_Pairwise_Diversity,0.007055,0.005394,0.008797,0.0001,0.0005,True



NOVELTY STATISTICAL RESULTS
Shape: (3, 7)
Columns: ['Metric', 'Observed_Difference', 'CI_Lower_95%', 'CI_Upper_95%', 'Bootstrap_p_value', 'Holm_Adjusted_p', 'Significant_After_Holm']


,Metric,Observed_Difference,CI_Lower_95%,CI_Upper_95%,Bootstrap_p_value,Holm_Adjusted_p,Significant_After_Holm
0,Novelty@10,0.159674,0.148697,0.170647,0.0001,0.0003,True
1,Mean_Popularity@10,-14044.580813,-16755.311984,-11732.687930,0.0001,0.0003,True
2,Median_Popularity@10,-11891.775161,-13226.359585,-10605.253182,0.0001,0.0003,True



✅ STATISTICAL RESULTS AUDIT COMPLETED!


In [103]:
# CELL 76: CONSOLIDATE STATISTICAL EVIDENCE

import pandas as pd
import os

# --------------------------------------------------
# 1. RANKING RESULTS
# --------------------------------------------------

ranking_stats = statistical_results["Ranking"].copy()

ranking_stats["Evaluation_Family"] = "Ranking Quality"

ranking_stats = ranking_stats.rename(columns={
    "Holm_Adjusted_p": "Adjusted_p_value",
    "Significant_After_Holm": "Significant"
})


# --------------------------------------------------
# 2. DIVERSITY RESULTS
# --------------------------------------------------

diversity_stats = statistical_results["Diversity"].copy()

diversity_stats["Evaluation_Family"] = "Diversity"

diversity_stats = diversity_stats.rename(columns={
    "Holm_Adjusted_p": "Adjusted_p_value",
    "Significant_After_Holm": "Significant"
})


# --------------------------------------------------
# 3. NOVELTY RESULTS
# --------------------------------------------------

novelty_stats = statistical_results["Novelty"].copy()

novelty_stats["Evaluation_Family"] = "Novelty & Popularity"

novelty_stats = novelty_stats.rename(columns={
    "Holm_Adjusted_p": "Adjusted_p_value",
    "Significant_After_Holm": "Significant"
})


# --------------------------------------------------
# 4. STANDARDIZE COMMON COLUMNS
# --------------------------------------------------

ranking_stats["Confidence_Interval_Lower"] = np.nan
ranking_stats["Confidence_Interval_Upper"] = np.nan

combined_statistical_results = pd.concat(
    [
        ranking_stats,
        diversity_stats,
        novelty_stats
    ],
    ignore_index=True,
    sort=False
)


# --------------------------------------------------
# 5. SELECT FINAL COLUMNS
# --------------------------------------------------

preferred_columns = [
    "Evaluation_Family",
    "Metric",
    "Observed_Difference",
    "Confidence_Interval_Lower",
    "Confidence_Interval_Upper",
    "Bootstrap_p_value",
    "Adjusted_p_value",
    "Significant"
]

available_columns = [
    col for col in preferred_columns
    if col in combined_statistical_results.columns
]

combined_statistical_results = (
    combined_statistical_results[available_columns]
)


# --------------------------------------------------
# 6. DISPLAY
# --------------------------------------------------

print("CONSOLIDATED STATISTICAL EVIDENCE")
print("=" * 120)

display(
    combined_statistical_results.round(6)
)


# --------------------------------------------------
# 7. SUMMARY COUNTS
# --------------------------------------------------

print("\nSIGNIFICANCE SUMMARY")
print("=" * 80)

print(
    "Total tested metrics:",
    len(combined_statistical_results)
)

print(
    "Significant metrics:",
    combined_statistical_results["Significant"].sum()
)

print(
    "Non-significant metrics:",
    (~combined_statistical_results["Significant"]).sum()
)


# --------------------------------------------------
# 8. SAVE
# --------------------------------------------------

combined_statistical_results.to_csv(
    os.path.join(
        RESULTS_DIR,
        "combined_statistical_evidence.csv"
    ),
    index=False
)

print("\nSaved: combined_statistical_evidence.csv")

print("\n✅ CONSOLIDATED STATISTICAL EVIDENCE COMPLETED!")

CONSOLIDATED STATISTICAL EVIDENCE


,Evaluation_Family,Metric,Observed_Difference,Confidence_Interval_Lower,Confidence_Interval_Upper,Bootstrap_p_value,Adjusted_p_value,Significant
0,Ranking Quality,precision,-0.003402,NaN,NaN,0.000300,0.001200,True
1,Ranking Quality,recall,-0.000965,NaN,NaN,0.000700,0.001400,True
2,Ranking Quality,ndcg,-0.002148,NaN,NaN,0.000300,0.001200,True
3,Ranking Quality,hit_rate,-0.004961,NaN,NaN,0.212779,0.212779,False
4,Diversity,Unique_Creators@10,0.015592,NaN,NaN,0.000100,0.000500,True
5,Diversity,Unique_Categories@10,0.092133,NaN,NaN,0.000100,0.000500,True
6,Diversity,Creator_Diversity@10,0.001559,NaN,NaN,0.000100,0.000500,True
7,Diversity,Creator_Pairwise_Diversity,0.000488,NaN,NaN,0.000600,0.000600,True
8,Diversity,Category_Pairwise_Diversity,0.007055,NaN,NaN,0.000100,0.000500,True
9,Novelty & Popularity,Novelty@10,0.159674,NaN,NaN,0.000100,0.000300,True



SIGNIFICANCE SUMMARY
Total tested metrics: 12
Significant metrics: 11
Non-significant metrics: 1

Saved: combined_statistical_evidence.csv

✅ CONSOLIDATED STATISTICAL EVIDENCE COMPLETED!


In [104]:
# CELL 77: EMERGING CREATOR FAIRNESS SUMMARY

import pandas as pd
import os

# --------------------------------------------------
# 1. FAIRNESS RESULTS
# --------------------------------------------------

fairness_summary = pd.DataFrame({

    "Metric": [
        "Emerging Creator Exposure Slots",
        "Emerging Creator Exposure Share",
        "Emerging Creator Coverage",
        "Emerging Creators with Zero Exposure",
        "Emerging Creator Exposure Gini",
        "Established Creator Exposure Gini",
        "Established Creator Coverage",
        "Established Creators with Zero Exposure"
    ],

    "Baseline": [
        3678,
        0.260666,
        0.518812,
        243,
        0.7845,
        0.8659,
        0.411881,
        891
    ],

    "Spotlight": [
        4297,
        0.304500,
        0.627723,
        188,
        0.7117,
        0.8706,
        0.399340,
        910
    ],

    "Desired_Direction": [
        "Increase",
        "Context-dependent",
        "Increase",
        "Decrease",
        "Decrease",
        "Decrease",
        "Increase",
        "Decrease"
    ]
})


# --------------------------------------------------
# 2. ABSOLUTE CHANGES
# --------------------------------------------------

fairness_summary["Absolute_Change"] = (
    fairness_summary["Spotlight"]
    -
    fairness_summary["Baseline"]
)


# --------------------------------------------------
# 3. RELATIVE CHANGES
# --------------------------------------------------

fairness_summary["Relative_Change_Percent"] = (
    (
        fairness_summary["Spotlight"]
        -
        fairness_summary["Baseline"]
    )
    /
    fairness_summary["Baseline"].replace(0, pd.NA)
) * 100


# --------------------------------------------------
# 4. DISPLAY
# --------------------------------------------------

print("EMERGING CREATOR FAIRNESS SUMMARY")
print("=" * 110)

display(
    fairness_summary.round(6)
)


# --------------------------------------------------
# 5. KEY FAIRNESS CHANGES
# --------------------------------------------------

print("\nKEY FAIRNESS OUTCOMES")
print("=" * 80)

print(
    "Additional Emerging creator slots:",
    4297 - 3678
)

print(
    "Change in Emerging creator coverage (percentage points):",
    round((0.627723 - 0.518812) * 100, 4)
)

print(
    "Reduction in Emerging creators with zero exposure:",
    243 - 188
)

print(
    "Emerging creator Gini reduction:",
    round(0.7845 - 0.7117, 4)
)


# --------------------------------------------------
# 6. SAVE
# --------------------------------------------------

fairness_summary.to_csv(
    os.path.join(
        RESULTS_DIR,
        "spotlight_fairness_summary.csv"
    ),
    index=False
)

print("\nSaved: spotlight_fairness_summary.csv")

print("\n✅ FAIRNESS SUMMARY COMPLETED!")

EMERGING CREATOR FAIRNESS SUMMARY


,Metric,Baseline,Spotlight,Desired_Direction,Absolute_Change,Relative_Change_Percent
0,Emerging Creator Exposure Slots,3678.000000,4297.000000,Increase,619.000000,16.829799
1,Emerging Creator Exposure Share,0.260666,0.304500,Context-dependent,0.043834,16.816156
2,Emerging Creator Coverage,0.518812,0.627723,Increase,0.108911,20.992383
3,Emerging Creators with Zero Exposure,243.000000,188.000000,Decrease,-55.000000,-22.633745
4,Emerging Creator Exposure Gini,0.784500,0.711700,Decrease,-0.072800,-9.279796
5,Established Creator Exposure Gini,0.865900,0.870600,Decrease,0.004700,0.542788
6,Established Creator Coverage,0.411881,0.399340,Increase,-0.012541,-3.044811
7,Established Creators with Zero Exposure,891.000000,910.000000,Decrease,19.000000,2.132435



KEY FAIRNESS OUTCOMES
Additional Emerging creator slots: 619
Change in Emerging creator coverage (percentage points): 10.8911
Reduction in Emerging creators with zero exposure: 55
Emerging creator Gini reduction: 0.0728

Saved: spotlight_fairness_summary.csv

✅ FAIRNESS SUMMARY COMPLETED!
